# PVT v2 + MoE — v12 standalone (sv1 architecture)

**Lineage:** v9 (B200 originals) → **v10** (v9 + 31 inline fixes, now
`archive/PVT_Tutelmoe_v10_patched.ipynb`) → **v12** (this file). v11 is the thin
launcher over the package (`notebooks/v11_train.ipynb`) and stays the day-to-day
runner.

**What this file is.** Every code cell below is the corresponding `pvt_moe/`
source file inlined **verbatim** (generated by `tools/make_v12_notebook.py`
from commit `7a720d7`; only the intra-package import lines are replaced by a
marker, since everything shares one notebook namespace). It trains the same
model, byte for byte, as `python train.py` — `tests/verify_v12_notebook.py`
builds the model from these cells and from the package and asserts the outputs
are identical. **Do not hand-edit the code cells**: change `pvt_moe/` and
regenerate, or the two will drift like v9/v10 did.

**How to read the Δ cells.** Before each section a *"Δ since v10"* cell shows
how the code moved: <del>struck-through lines are v10</del>, plain lines are
today, and the indented line under each pair says why.

**Before the first run on a new box** (the 4×5090 lesson — a machine can be
wrong while every smoke test passes):

1. `python3 tests/run_all.py` — two of the tests TRAIN through the whole
   chain, under the GPU's real precision when one is present.
2. `python tools/check_kernels.py --variant b2 --img 224 --batch 128` — every
   op at the real shapes, forward AND backward, bf16 vs fp32 references.
3. Decode a few train images next to their label *names* and look at them.
4. `python train.py --overfit-check 200` — the coarse bisect (see its PASS
   message for exactly what it does and does not clear).

Edit **only the CONFIG cell** below, then Run All.


In [ ]:
# ── dependencies ────────────────────────────────────────────────────────────
# Torch itself is assumed installed (pick the build matching your CUDA).
# Tutel is OPTIONAL: it builds a CUDA extension (takes minutes). Without it,
# set cfg["model"]["moe"]["backend"] = "native" — same routing semantics,
# pure-PyTorch experts (that fallback did not exist in v10).
import importlib, subprocess, sys

def _ensure(mod, pip=None):
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip or mod])

for mod, pip in [("pytorch_lightning", "pytorch-lightning"), ("torchmetrics", None),
                 ("timm", None), ("datasets", None), ("torchvision", None),
                 ("PIL", "pillow"), ("numpy", None), ("yaml", "pyyaml")]:
    _ensure(mod, pip)
# _ensure("tutel", "git+https://github.com/microsoft/tutel@main")   # only for backend "tutel"
# _ensure("wandb")                                                  # only for USE_WANDB=True
print("dependencies OK")


## Δ since v10 — the CONFIG cell

v10's config was a hand-grown dict; v12 embeds the dict **the package
resolves** and mutates it with a few knobs. The consequential changes:

**Identity & mode — v10 was a resume, v12 is from-scratch:**
<pre>
<del>"run_name": "pvt_tutel_LR_aux_fixed_p2_v10",          # hand-set</del>
<del>"version": 10,</del>
<del>"ckpt_path": ".../epoch=53-MulticlassAccuracy/val=0.7227.ckpt",</del>
<del>"resuming": True,</del>
cfg["run_name"] = build_run_tag(cfg)   # derived: sv1_b2_in1k_r224_dense_norope_ln_scratch90
cfg["version"]  = "sv1"                # bumped on every architecture change
cfg["mode"], cfg["ckpt_path"] = "scratch", None
</pre>
&nbsp;&nbsp;*why:* two configs that differ in anything that changes the model must
never share a name/checkpoint dir (tests enforce it); and nothing in this repo
has yet demonstrated the from-scratch recipe converging — v10's "working" run
was a warm resume at lr 1e-4, which is why it proved less than it seemed.

**Attention — GQA is gone:**
<pre>
<del>"num_kv_heads": [1, 1, 1, 2],     # grouped-query attention per stage</del>
# plain MHA everywhere; the key removed from the schema (a GQA value is refused)
</pre>
&nbsp;&nbsp;*why:* the thesis doesn't ablate GQA, and RoPE-Mixed needs one frequency
set per *query* head, i.e. MHA anyway.

**RoPE — fixed axial → learnable mixed:**
<pre>
<del>"use_rope": True, "rope_last_n_stages": 1, "rope_theta": 50,   # axial</del>
"ablation": {"use_rope": ..., "rope_placement": [[], [], [], [-1]],
             "rope_mode": "mixed",   # learnable per-head 2D freqs (rope-vit)
             "rope_theta": None}     # resolves per mode: 10 (init spread) / 50 axial
</pre>

**Schedule — resume numbers → the scratch recipe:**
<pre>
<del>"epochs": 100, "lr": 1e-4, "warmup_epochs": 0, "start_factor": 1e-6,</del>
<del>"drop_path_rate": 0.2,  "stage4_lr_multiplier": 10.0,</del>
epochs 90 | lr 1e-3 @ effective 1024 | warmup 5 ep from 1e-6 | cosine → 1e-6
wd 0.05 | grad clip 5.0 | drop_path 0.1 (variant rule) | stage4 multiplier 1.0
</pre>
&nbsp;&nbsp;*why:* v10's numbers continued a mature checkpoint; these are the
PVT v2 / DeiT from-scratch numbers the ablation grid is calibrated for.

**MoE knobs:**
<pre>
<del>"moe_capacity_factor": 2.0,</del>
"capacity_factor": 1.0,   # standard top-1 setting; the drops it causes are now MEASURED
"gate_noise": 0.5, "num_experts": 4, "top_k": 1,        # unchanged
"shared_expert": True, "moe_block_dwconv": True,        # v10's always-on pair, kept
"backend": "tutel",                                     # or "native": no CUDA extension
</pre>
&nbsp;&nbsp;*why:* capacity 2.0 hid imbalance by over-provisioning; at 1.0 the
RoutingMonitor's `train_drop_rate` becomes the honest health signal
(`train_aux` cannot be one — see the diagnostics cell's docstring).

**Kept from v10:** micro-batch 128 × accumulate 8 = effective 1024, bf16-mixed,
8 workers, milestone checkpoints, `stop_at_epoch` for running a long schedule
in pieces.


In [ ]:
# ═══ lifted VERBATIM from pvt_moe/config.py @ 7a720d7 (registry + naming) ═══
DATASETS = {
    "imagenet-1k": {"num_classes": 1000, "labelled": True, "tag": "in1k",
                    "hf_id": "ILSVRC/imagenet-1k", "gated": True, "finetune_epochs": None,
                    "licence": "ImageNet terms of access; gated on HF"},
    "imagenet-22k": {"num_classes": 21841, "labelled": True, "tag": "in22k",
                     "hf_id": "timm/imagenet-22k-wds", "gated": True, "finetune_epochs": None,
                     "licence": "ImageNet terms of access; gated on HF"},
    "pass": {"num_classes": 0, "labelled": False, "tag": "pass",
             "hf_id": None, "gated": False, "finetune_epochs": None,
             "hf_id_hint": "yukimasano/pass (loading script — not loadable; use Zenodo)",
             "licence": "CC-BY 4.0 (images and dataset)"},
    # --- small transfer / downstream sets (supervised, scratch or fine-tune) ---
    # Native resolutions are far below 224; dataset.img_size (224) upsamples
    # them in the train/val transforms, so results on these partly measure
    # interpolation (docs/GUIDE.md). ``hf_id`` is None where the Hub id has
    # not been verified from this machine — download_data.py takes --hf-id
    # (``hf_id_hint`` is the id to try first) or, for MedMNIST, --npz.
    # ``finetune_epochs`` is the FIXED per-dataset fine-tune budget.
    "fashionmnist": {"num_classes": 10, "labelled": True, "tag": "fmnist", "hf_id": None,
                     "hf_id_hint": "zalando-datasets/fashion_mnist",
                     "gated": False, "finetune_epochs": 30, "native_size": 28,
                     "channels": 1,          # grayscale; the loader converts to RGB
                     "splits": "train 60,000 / test 10,000 (no validation split: "
                               "download_data.py carves a seeded 10% of train)",
                     "licence": "MIT (Zalando SE, 2017; github.com/zalandoresearch/"
                                "fashion-mnist, LICENSE)"},
    "eurosat": {"num_classes": 10, "labelled": True, "tag": "eurosat", "hf_id": None,
                "hf_id_hint": "blanchon/EuroSAT_RGB",
                "gated": False, "finetune_epochs": 50, "native_size": 64,
                "channels": 3,               # the RGB release (not the 13-band MS one)
                "splits": "27,000 images, no official split: download_data.py carves "
                          "seeded validation (10%) and test (10%) from the whole set",
                "licence": "MIT (Patrick Helber; github.com/phelber/EuroSAT, LICENSE); "
                           "imagery: Sentinel-2, ESA Copernicus open data"},
    "pathmnist": {"num_classes": 9, "labelled": True, "tag": "path", "hf_id": None,
                  "hf_id_hint": None,        # MedMNIST ships npz files, not a Hub repo
                  "gated": False, "finetune_epochs": 30,
                  # MedMNIST+ (v2.2+, Yang et al. 2023) ships PathMNIST at 28, 64,
                  # 128 and 224 px; use the 224 file (pathmnist_224.npz) so the
                  # run needs no upsampling. The 28-px v2 file also loads.
                  "native_size": 224, "channels": 3,
                  "splits": "train 89,996 / validation 10,004 / test 7,180 "
                            "(MedMNIST's own split, kept as is)",
                  "licence": "CC BY 4.0 (MedMNIST v2 / MedMNIST+; source NCT-CRC-HE-100K, "
                             "Kather et al. 2018, CC BY 4.0)"},
}


VARIANTS = {
    "b0": {"depths": [2, 2, 2, 2],   "embed_dims": [32, 64, 160, 256],
           "num_heads": [1, 2, 5, 8], "mlp_ratios": [8, 8, 4, 4], "sr_ratios": [8, 4, 2, 1],
           "drop_path": 0.1, "clip_grad": None, "params_m": 3.7,  "top1": 70.5,
           "hf_id": "OpenGVLab/pvt_v2_b0"},
    "b1": {"depths": [2, 2, 2, 2],   "embed_dims": [64, 128, 320, 512],
           "num_heads": [1, 2, 5, 8], "mlp_ratios": [8, 8, 4, 4], "sr_ratios": [8, 4, 2, 1],
           "drop_path": 0.1, "clip_grad": None, "params_m": 14.0, "top1": 78.7,
           "hf_id": "OpenGVLab/pvt_v2_b1"},
    "b2": {"depths": [3, 4, 6, 3],   "embed_dims": [64, 128, 320, 512],
           "num_heads": [1, 2, 5, 8], "mlp_ratios": [8, 8, 4, 4], "sr_ratios": [8, 4, 2, 1],
           "drop_path": 0.1, "clip_grad": None, "params_m": 25.4, "top1": 82.0,
           "hf_id": "OpenGVLab/pvt_v2_b2"},
    "b3": {"depths": [3, 4, 18, 3],  "embed_dims": [64, 128, 320, 512],
           "num_heads": [1, 2, 5, 8], "mlp_ratios": [8, 8, 4, 4], "sr_ratios": [8, 4, 2, 1],
           "drop_path": 0.3, "clip_grad": 1.0,  "params_m": 45.2, "top1": 83.1,
           "hf_id": "OpenGVLab/pvt_v2_b3"},
    "b4": {"depths": [3, 8, 27, 3],  "embed_dims": [64, 128, 320, 512],
           "num_heads": [1, 2, 5, 8], "mlp_ratios": [8, 8, 4, 4], "sr_ratios": [8, 4, 2, 1],
           "drop_path": 0.3, "clip_grad": 1.0,  "params_m": 62.6, "top1": 83.6,
           "hf_id": "OpenGVLab/pvt_v2_b4"},
    "b5": {"depths": [3, 6, 40, 3],  "embed_dims": [64, 128, 320, 512],
           "num_heads": [1, 2, 5, 8], "mlp_ratios": [4, 4, 4, 4], "sr_ratios": [8, 4, 2, 1],
           "drop_path": 0.3, "clip_grad": 1.0,  "params_m": 82.0, "top1": 83.8,
           "hf_id": "OpenGVLab/pvt_v2_b5"},
}


VARIANT_ARCH_KEYS = ("depths", "embed_dims", "num_heads", "mlp_ratios", "sr_ratios")


def resolve_placement(placement, last_n, depths):
    """Resolve a per-stage/per-block placement specification.

    ``placement`` is the authoritative form: a list (length = num stages) of
    lists of block indices. A negative index counts from the end of the stage
    (``-1`` = last block), which is how "last block of stage 4" stays correct
    across variants of different depth; the resolved form is always
    non-negative. ``last_n`` is a convenience that, when not None, generates
    "all blocks of the last N stages".
    """
    num_stages = len(depths)
    if last_n is not None:
        if not 0 <= last_n <= num_stages:
            raise ValueError(f"last_n_stages must be in [0, {num_stages}], got {last_n}")
        return [
            list(range(depths[i])) if i >= num_stages - last_n else []
            for i in range(num_stages)
        ]
    if len(placement) != num_stages:
        raise ValueError(
            f"placement must have {num_stages} entries (one per stage), got {len(placement)}"
        )
    resolved = []
    for i, blocks in enumerate(placement):
        normalized = set()
        for b in blocks:
            b = int(b)
            if not -depths[i] <= b < depths[i]:
                raise ValueError(
                    f"placement stage {i + 1}: block index {b} out of range "
                    f"(depth {depths[i]}: valid 0..{depths[i] - 1}, or "
                    f"-1..-{depths[i]} counting from the last block)"
                )
            normalized.add(b % depths[i])
        resolved.append(sorted(normalized))
    return resolved


def _placement_tag(placement, depths) -> str:
    """Compact human-readable tag, e.g. [[],[],[],[0,1]] -> 's4'; [[],[],[1],[0]] -> 's3b1+s4b0'."""
    parts = []
    for i, blocks in enumerate(placement):
        if not blocks:
            continue
        if blocks == list(range(depths[i])):
            parts.append(f"s{i + 1}")           # full stage
        else:
            parts.append(f"s{i + 1}b{''.join(map(str, blocks))}")  # partial stage
    return "+".join(parts) if parts else "none"


def stage_tag(cfg: dict) -> str:
    """One pipeline stage in words: ``"simmim_pretrain@pass_r224"``,
    ``"ssl_finetune+moe@imagenet-1k_r224"``.

    ``cfg["chain"]`` is the list of these, oldest first, so a result can name
    the whole path that produced it (SSL pretrain -> intermediate supervised
    ImageNet fine-tune -> downstream task). ``+moe`` marks a stage whose
    backbone carried routed experts, which is what tells the three
    pretraining paths apart in a chain (docs/SIMMIM_GUIDE.md §6).
    """
    ds = cfg["dataset"]["name"]
    res = f"r{cfg['dataset']['img_size']}"
    abl = cfg["model"]["ablation"]
    moe = "+moe" if abl.get("use_moe") and any(abl.get("moe_placement") or []) else ""
    if cfg.get("task") == "ssl":
        return f"{cfg['ssl']['method']}_pretrain{moe}@{ds}_{res}"
    kind = {"scratch": "scratch", "pretrained": "hf_finetune",
            "ssl_finetune": "ssl_finetune", "downstream": "downstream"}.get(
        cfg.get("recipe"), cfg.get("mode") or "run")
    return f"{kind}{moe}@{ds}_{res}"


def parent_tag(ckpt_path: str | None) -> str | None:
    """A short tag naming the run a warm-start checkpoint came from, read from
    its PATH (``<root>/<parent run name>/<file>``): ``from-dense-simmim200``,
    ``from-moe-sslft100-from-dense-simmim200``.

    Two fine-tunes that differ ONLY in their parent — path 2 (MoE pretrain)
    vs path 3 (dense pretrain, upcycled now) — would otherwise share a run
    name and a checkpoint directory. Needs no torch and no file access, so
    ``--dry-run`` shows it. None when the parent directory does not follow
    this repo's naming (pass --run-name then).
    """
    if not ckpt_path:
        return None
    import os

    parent = os.path.basename(os.path.dirname(os.path.abspath(ckpt_path)))
    fields = parent.split("_")
    if len(fields) < 7 or not fields[0].startswith(("sv", "v")):
        return None
    if fields[4] == "dense":
        moe = "dense"
    elif fields[4].startswith("moe-"):
        moe = "moe"
    else:
        return None
    norm_at = next((i for i in range(5, len(fields)) if fields[i] in ("ln", "rms")), None)
    if norm_at is None or norm_at + 1 >= len(fields):
        return None
    budget = "-".join(fields[norm_at + 1:])
    return f"from-{moe}-{budget}"


def build_run_tag(cfg: dict) -> str:
    """Derive a self-documenting run name from the ablation flags.

    Example: ``sv1_b1_in1k_r224_moe-s4b1-e4k1+sh_rope-s4b1_ln_scratch90``

    The variant sits right after the version: two sizes in one W&B project
    are otherwise indistinguishable, and a B2 run would overwrite a B1 run's
    checkpoint directory.
    """
    ds = DATASETS[cfg["dataset"]["name"]]["tag"]
    # Resolution is part of the identity: a 224 arm and a 256 arm are not
    # comparable and must not share a checkpoint directory.
    res = f"r{cfg['dataset']['img_size']}"
    variant = cfg["model"]["variant"]
    abl = cfg["model"]["ablation"]
    depths = cfg["model"]["depths"]

    if abl["use_moe"]:
        moe_pl = resolve_placement(abl["moe_placement"], abl["moe_last_n_stages"], depths)
        moe_cfg = cfg["model"]["moe"]
        # Per-backend tag. A binary "tutel or -mb" test silently labelled the
        # native backend as megablocks; every backend needs its own marker or
        # two different implementations share a checkpoint directory.
        backend = {"tutel": "", "native": "-nat", "megablocks": "-mb"}[
            moe_cfg["backend"]]
        shared = "+sh" if moe_cfg.get("shared_expert") else ""
        # The random-expert-init control (pretrained ladder row 6) is
        # architecturally identical to the upcycled run it is compared against,
        # so the init has to appear in the name or the two overwrite each other.
        # The init only applies to an upcycled run with a shared expert; tag
        # the non-default arms there so an init ablation cannot put two runs in
        # one checkpoint directory. Tagging it everywhere would put a marker on
        # every from-scratch run, which never upcycles anything.
        init_applies = (
            cfg.get("mode") in ("hf_pretrained", "ssl_init")
            and moe_cfg.get("shared_expert")
            and cfg["model"].get("seed_moe_from_dense", True)
        )
        init = {"shared_zero": "-szi", "none": "-nozi"}.get(
            moe_cfg.get("upcycle_init"), "") if init_applies else ""
        # A MoE'd block with and without its DWConv are different models;
        # without this they would share a checkpoint directory.
        plain = ("-plain" if moe_cfg.get("shared_expert")
                 and not moe_cfg.get("moe_block_dwconv", True) else "")
        randexp = (
            "-randexp"
            if cfg.get("mode") == "hf_pretrained"
            and not cfg["model"].get("seed_moe_from_dense", True)
            else ""
        )
        moe = (
            f"moe-{_placement_tag(moe_pl, depths)}-"
            f"e{moe_cfg['num_experts']}k{moe_cfg['top_k']}{shared}{plain}{init}{randexp}{backend}"
        )
    else:
        moe = "dense"

    if abl["use_rope"]:
        rope_pl = resolve_placement(abl["rope_placement"], abl["rope_last_n_stages"], depths)
        # Mixed (the default) is untagged; the fixed-frequency arm is "-ax".
        # Two RoPE flavours in one placement are different models, so the
        # flavour has to be in the name or they share a checkpoint directory.
        flavour = "-ax" if abl["rope_mode"] == "axial" else ""
        rope = f"rope-{_placement_tag(rope_pl, depths)}{flavour}"
    else:
        rope = "norope"

    # Without this, "conv-FFN intact" and "no DWConv" dense arms produce the
    # same run name and overwrite each other's checkpoints.
    dwconv = "" if cfg["model"].get("dense_dwconv", True) else "_nodw"
    norm = {"layernorm": "ln", "rmsnorm": "rms"}[cfg["model"]["norm_type"]]
    # Budget tag: the epoch count is an ablation axis of its own (90/150/300
    # from scratch vs 100 fine-tuned), so it belongs in the run name.
    budget = {"scratch": "scratch", "pretrained": "ft", "ssl_finetune": "sslft",
              "downstream": "dstr"}.get(cfg.get("recipe"), "run")
    # Repeat marker: last, so the arm is still readable left to right.
    suffix = f"_{cfg['run_suffix']}" if cfg.get("run_suffix") else ""
    if cfg.get("task") == "ssl":
        # An SSL run is identified by its method and pretraining length; the
        # mask space changes what the encoder sees, so it is tagged too.
        ssl = cfg["ssl"]
        px = "-px" if ssl.get("mask_space") == "pixel" else ""
        return (f"{cfg['version']}_{variant}_{ds}_{res}_{moe}_{rope}{dwconv}_{norm}_"
                f"{ssl['method']}{ssl['epochs']}{px}{suffix}")
    # epochs == 0 is the eval-only row of the pretrained ladder.
    budget = "eval" if cfg["epochs"] == 0 else f"{budget}{cfg['epochs']}"
    # A warm start from an SSL / fine-tuned checkpoint is named after its
    # parent too (parent_tag), so paths 2 and 3 never share a directory.
    parent = parent_tag(cfg.get("ckpt_path")) if cfg.get("mode") == "ssl_init" else None
    parent = f"_{parent}" if parent else ""
    return f"{cfg['version']}_{variant}_{ds}_{res}_{moe}_{rope}{dwconv}_{norm}_{budget}{parent}{suffix}"


# stub: low-shot subsets need the full package (pvt_moe.eval.lowshot)
def load_subset(*a, **k):
    raise RuntimeError("dataset.subset_file needs the pvt_moe package "
                       "(pvt_moe.eval.lowshot); unset it in this notebook")


In [ ]:
# ═══════════════════════ CONFIG — the only cell to edit ═════════════════════
# The template below is the config the PACKAGE resolves for the default arm
# (embedded verbatim at generation time); the knobs mutate only what they name.
VARIANT      = "b2"          # b0..b5 — depths/dims/heads set as one unit below
USE_MOE      = False         # routed FFN in the placed blocks
USE_ROPE     = False         # RoPE-Mixed in the placed blocks
NORM         = "layernorm"   # "layernorm" | "rmsnorm" (stage 4 keeps LN)
MOE_PLACEMENT  = [[], [], [], [-1]]   # per-stage block lists; -1 = stage's last block
ROPE_PLACEMENT = [[], [], [], [-1]]
MOE_BACKEND  = "tutel"       # "tutel" | "native" (no CUDA extension needed)

EPOCHS        = 90
STOP_AT_EPOCH = None         # train a long schedule in pieces (cosine still spans EPOCHS)
MILESTONES    = [25, 50, 75]
BATCH, ACCUM  = 128, 8       # micro-batch x accumulation = effective 1024 (LR calibration)
NUM_WORKERS   = 8
PRECISION     = "bf16-mixed"
SEED          = 42

DATA_DIR        = "/data/imagenet_arrow"      # Arrow snapshot (download_data.py)
CHECKPOINT_ROOT = "/data/runs/checkpoints"
LOG_ROOT        = "/data/runs/logs"
USE_WANDB       = False
RESUME_FROM     = None       # ".../<run_name>/last.ckpt" to continue that run
RUN_SUFFIX      = None       # e.g. "v2": rerun an arm without sharing its directory

import copy

CFG_TEMPLATE = {'accumulate_grad_batches': 8,
 'batch_size': 128,
 'chain': ['scratch@imagenet-1k_r224'],
 'checkpoint_root': '/workspace/ModelTraining/checkpoints',
 'ckpt_path': None,
 'dataset': {'arrow_dirs': {'eurosat': '<set arrow dir for eurosat>',
                            'fashionmnist': '<set arrow dir for fashionmnist>',
                            'imagenet-1k': '<set arrow dir for imagenet-1k>',
                            'imagenet-22k': '<set arrow dir for imagenet-22k>',
                            'pass': '<set arrow dir for pass>',
                            'pathmnist': '<set arrow dir for pathmnist>'},
             'crop_pct': 0.875,
             'img_size': 224,
             'name': 'imagenet-1k',
             'num_classes': 1000,
             'randaugment': 'rand-m9-mstd0.5-inc1',
             'randaugment_magnitude': 9,
             'randaugment_ops': 2,
             'random_erasing': 0.25,
             'repeated_aug': 3,
             'subset_file': None},
 'deterministic': False,
 'effective_batch_size': 1024,
 'epochs': 90,
 'experiment_group': 'ablations',
 'limit_train_batches': None,
 'limit_val_batches': None,
 'log_root': '/workspace/ModelTraining/logs',
 'loss': {'aux_clamp': 10.0,
          'aux_weight': 0.01,
          'cutmix_alpha': 1.0,
          'label_smoothing': 0.1,
          'mixup_alpha': 0.8,
          'mixup_prob': 0.8,
          'mixup_switch_prob': 0.5},
 'milestones': [],
 'mode': 'scratch',
 'model': {'ablation': {'moe_last_n_stages': None,
                        'moe_placement': [[], [], [], [2]],
                        'rope_last_n_stages': None,
                        'rope_mode': 'mixed',
                        'rope_placement': [[], [], [], [2]],
                        'rope_theta': None,
                        'use_moe': False,
                        'use_rope': False},
           'attn_drop_rate': 0.0,
           'dense_dwconv': True,
           'depths': [3, 4, 6, 3],
           'drop_path_rate': 0.1,
           'drop_rate': 0.0,
           'embed_dims': [64, 128, 320, 512],
           'grad_checkpointing': [],
           'in_chans': 3,
           'linear_attention': False,
           'mlp_ratios': [8, 8, 4, 4],
           'moe': {},
           'norm_eps': 1e-06,
           'norm_type': 'layernorm',
           'num_frozen_stages': 0,
           'num_heads': [1, 2, 5, 8],
           'pretrained_hf_id': 'OpenGVLab/pvt_v2_b2',
           'qkv_bias': True,
           'resume_check_identity': True,
           'seed_moe_from_dense': True,
           'sr_ratios': [8, 4, 2, 1],
           'ssl_init_check_arch': True,
           'stage4_keeps_layernorm': True,
           'variant': 'b2'},
 'num_workers': 8,
 'optim': {'base_lr': None,
           'betas': [0.9, 0.999],
           'eta_min': 1e-06,
           'grad_clip': 5.0,
           'layer_decay': 1.0,
           'lr': 0.001,
           'lr_reference_batch': None,
           'stage4_lr_multiplier': 1.0,
           'warmup_epochs': 5,
           'warmup_start_factor': 0.001,
           'weight_decay': 0.05},
 'precision': 'bf16-mixed',
 'recipe': 'scratch',
 'run_name': None,
 'run_suffix': None,
 'seed': 42,
 'ssl': {'base_lr': 0.0002,
         'betas': [0.9, 0.999],
         'ema_momentum': 0.996,
         'ema_momentum_end': 1.0,
         'epochs': 200,
         'final_lr': None,
         'final_lr_base': 1e-05,
         'grad_clip': 5.0,
         'lr': None,
         'lr_reference_batch': 512,
         'mask_aspect_ratio': [0.75, 1.5],
         'mask_block_area': [0.1, 0.2],
         'mask_n_blocks': 4,
         'mask_patch_size': 32,
         'mask_ratio': 0.6,
         'mask_space': 'token',
         'method': 'simmim',
         'predictor_depth': 6,
         'predictor_dim': 384,
         'predictor_heads': 6,
         'warmup_epochs': 10,
         'warmup_lr': None,
         'warmup_lr_base': 1e-06,
         'weight_decay': 0.05,
         'weight_decay_end': 0.4},
 'stop_at_epoch': None,
 'task': 'supervised',
 'use_tensorboard': False,
 'use_wandb': True,
 'val_batch_multiplier': 2,
 'version': 'sv1',
 'wandb_project': 'pvt-moe-imagenet-FINAL'}

MOE_DEFAULTS = {'backend': 'tutel',
 'capacity_factor': 1.0,
 'gate_noise': 0.5,
 'moe_block_dwconv': True,
 'num_experts': 4,
 'routing_monitor': True,
 'shared_expert': True,
 'top_k': 1,
 'upcycle_init': 'none'}

cfg = copy.deepcopy(CFG_TEMPLATE)
arch = VARIANTS[VARIANT]
for key in VARIANT_ARCH_KEYS:
    cfg["model"][key] = list(arch[key])
cfg["model"]["variant"] = VARIANT
cfg["model"]["drop_path_rate"] = 0.1 if VARIANT in ("b0", "b1", "b2") else arch["drop_path"]
cfg["model"]["norm_type"] = NORM
cfg["model"]["moe"] = copy.deepcopy(MOE_DEFAULTS)
cfg["model"]["moe"]["backend"] = MOE_BACKEND

abl = cfg["model"]["ablation"]
depths = cfg["model"]["depths"]
abl["use_moe"], abl["use_rope"] = bool(USE_MOE), bool(USE_ROPE)
abl["moe_placement"] = resolve_placement(MOE_PLACEMENT, None, depths) if USE_MOE else [[] for _ in depths]
abl["rope_placement"] = resolve_placement(ROPE_PLACEMENT, None, depths) if USE_ROPE else [[] for _ in depths]
abl["moe_last_n_stages"] = abl["rope_last_n_stages"] = None

cfg["epochs"], cfg["stop_at_epoch"], cfg["milestones"] = EPOCHS, STOP_AT_EPOCH, list(MILESTONES)
cfg["batch_size"], cfg["accumulate_grad_batches"] = BATCH, ACCUM
cfg["effective_batch_size"] = BATCH * ACCUM
cfg["num_workers"], cfg["precision"], cfg["seed"] = NUM_WORKERS, PRECISION, SEED
cfg["dataset"]["arrow_dirs"]["imagenet-1k"] = DATA_DIR
cfg["checkpoint_root"], cfg["log_root"] = CHECKPOINT_ROOT, LOG_ROOT
cfg["use_wandb"] = USE_WANDB
cfg["run_suffix"] = RUN_SUFFIX
if RESUME_FROM:
    cfg["mode"], cfg["ckpt_path"] = "resume", RESUME_FROM

cfg["run_name"] = build_run_tag(cfg)
cfg["chain"] = [stage_tag(cfg)]
print(f"run:   {cfg['run_name']}")
print(f"chain: {cfg['chain'][0]}")
print(f"{VARIANT} depths {depths} | moe {abl['moe_placement']} | rope {abl['rope_placement']}"
      f" | {cfg['epochs']} ep | batch {BATCH} x {ACCUM} = {cfg['effective_batch_size']}"
      f" | lr {cfg['optim']['lr']} (warmup {cfg['optim']['warmup_epochs']} ep)")


## Δ since v10 — environment

v10 scattered this across `%env` lines and ad-hoc cells; it is now one audited
function (`setup_environment`) with a memory-budget check that names the
micro-batch that fits.

<pre>
<del>os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # ad-hoc cell</del>
<del># TF32 left at torch defaults</del>
torch.set_float32_matmul_precision("high")        # TF32 on for fp32 matmuls
torch.backends.cudnn.benchmark = True             # unless cfg["deterministic"]
# deterministic=True → cudnn.deterministic, no benchmark (bit-reproducible, slower)
</pre>

W&B and the HF token come from the environment (`WANDB_API_KEY`, `HF_TOKEN`) —
<del>the v10 cell that mounted a private Google Drive to read tokens</del> is gone;
never put credentials in a notebook cell.


In [ ]:
# ═══ pvt_moe/engine/env.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""Environment setup for plain-Jupyter training boxes (B200 / RTX 5090).

No Colab assumptions anywhere: credentials come from environment variables
(``HF_TOKEN``, ``WANDB_API_KEY``), optionally interactively via getpass when
running in a terminal/notebook and the variable is unset.
"""

from __future__ import annotations

import os
import sys

import torch


def setup_environment(cfg: dict, interactive_secrets: bool = False) -> torch.device:
    """Seed, precision, allocator, and (optional) credential setup.

    - ``deterministic=True``: bit-reproducible (cudnn deterministic, no
      benchmark autotune) at a real speed cost. Default False keeps
      ``cudnn.benchmark`` autotuning on — seeds still fix data order and init,
      but kernels may be nondeterministic.
    """
    import pytorch_lightning as pl

    # expandable_segments uses CUDA's virtual-memory APIs and is not supported
    # on Windows — setting it there produces an allocator warning on every run
    # and does nothing. Keep it to Linux.
    if sys.platform.startswith("linux"):
        os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    pl.seed_everything(cfg["seed"], workers=True)
    torch.set_float32_matmul_precision("high")
    if cfg["deterministic"]:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)
    else:
        torch.backends.cudnn.benchmark = True

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        name = torch.cuda.get_device_name(0)
        cap = torch.cuda.get_device_capability(0)
        total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[env] GPU: {name} (sm_{cap[0]}{cap[1]}, {total_gb:.1f} GiB) "
              f"| torch {torch.__version__} | CUDA {torch.version.cuda}")
        _check_memory_budget(cfg, total_gb)
    else:
        print(f"[env] No GPU visible — CPU only (torch {torch.__version__})")

    _setup_secrets(cfg, interactive=interactive_secrets)
    return device


#: Peak VRAM per image measured for PVT v2 B1 at 224^2 under bf16 autocast,
#: as a rough planning figure only — real usage depends on MoE placement,
#: capacity factor and the allocator's fragmentation.
_APPROX_GIB_PER_IMAGE = 0.035

#: Activation cost of each variant relative to B1, from the MACs of this
#: repo's dense model at 224^2 (torch FlopCounterMode, MHA): b0 0.53 G,
#: b1 2.03 G, b2 3.88 G, b3 6.68 G, b4 9.79 G, b5 11.35 G. Activation memory
#: tracks the same sum over stages of depth x tokens x width, so the ratio is
#: a fair planning multiplier — it is NOT a measurement of the larger sizes.
_VRAM_SCALE_VS_B1 = {"b0": 0.27, "b1": 1.0, "b2": 1.9, "b3": 3.3,
                     "b4": 4.8, "b5": 5.6, "custom": 1.0}


def gib_per_image(variant: str = "b1") -> float:
    """Planning estimate of peak training VRAM per image for ``variant``."""
    return _APPROX_GIB_PER_IMAGE * _VRAM_SCALE_VS_B1.get(variant, 1.0)


def _check_memory_budget(cfg: dict, total_gb: float) -> None:
    """Warn before a micro-batch that is unlikely to fit is attempted.

    Advisory only — it never changes the config. A run that OOMs after an hour
    of data loading is worse than a warning that is occasionally pessimistic.
    """
    micro = cfg["batch_size"]
    accum = cfg.get("accumulate_grad_batches", 1)
    val_micro = micro * cfg.get("val_batch_multiplier", 1)
    print(f"[env] batch: {micro} micro x {accum} accum = "
          f"{cfg.get('effective_batch_size', micro * accum)} effective "
          f"| val {val_micro}")

    # Whatever the desktop/compositor already holds is not available to us.
    free_gb = torch.cuda.mem_get_info(0)[0] / 1024**3
    variant = cfg.get("model", {}).get("variant", "b1")
    per_image = gib_per_image(variant)
    estimate = micro * per_image
    if estimate > free_gb * 0.9:
        fits = max(16, int(free_gb * 0.9 / per_image) // 16 * 16)
        print(f"[env] WARNING: micro-batch {micro} of variant {variant} needs "
              f"roughly {estimate:.1f} GiB but only {free_gb:.1f} GiB is free. "
              f"Try batch_size={fits} (raise accumulate_grad_batches to keep "
              f"the same effective batch).")
    if sys.platform == "win32" and cfg.get("num_workers", 0) > 8:
        print(f"[env] NOTE: num_workers={cfg['num_workers']} on Windows — "
              "workers spawn (no fork), so each re-imports the module and "
              "holds its own copy. 4-8 is usually the sweet spot on 32 GB.")


def _setup_secrets(cfg: dict, interactive: bool) -> None:
    """Read HF/W&B tokens from env vars; optionally prompt when missing."""
    hf_token = os.getenv("HF_TOKEN")
    if hf_token:
        try:
            from huggingface_hub import login

            login(token=hf_token, add_to_git_credential=False)
            print("[env] Hugging Face: logged in from HF_TOKEN")
        except Exception as e:  # non-fatal: public models still download
            print(f"[env] Hugging Face login skipped: {e}")

    if cfg.get("use_wandb"):
        if not os.getenv("WANDB_API_KEY") and interactive:
            import getpass

            key = getpass.getpass("WANDB_API_KEY (empty to disable W&B): ").strip()
            if key:
                os.environ["WANDB_API_KEY"] = key
            else:
                cfg["use_wandb"] = False
                print("[env] W&B disabled for this run")
        elif not os.getenv("WANDB_API_KEY"):
            print("[env] WANDB_API_KEY not set — W&B will prompt or fail at trainer start. "
                  "Export it or set use_wandb=False.")


In [ ]:
# ── environment: TF32, cudnn, memory budget, seed ───────────────────────────
device = setup_environment(cfg)
import pytorch_lightning as pl
pl.seed_everything(cfg["seed"], workers=True)


## Δ since v10 — data pipeline

**Augmentation — torchvision ops → the timm DeiT stack:**
<pre>
<del>transforms.RandAugment(2, 9),                     # torchvision variant</del>
rand_augment_transform("rand-m9-mstd0.5-inc1", hparams)   # timm; PVT v2's actual recipe
transforms.RandomErasing(p=0.25)                          # was absent in v10
</pre>

**Sampling — plain shuffle → repeated augmentation:**
<pre>
<del>DataLoader(train_ds, shuffle=True, ...)</del>
RepeatAugSampler(train_ds, num_repeats=3, num_replicas=1, rank=0)
</pre>
&nbsp;&nbsp;*why:* the DeiT recipe the LR is calibrated for; the explicit
`num_replicas=1, rank=0` works around timm calling `dist.get_world_size()`
on a single process. Note the cost: 3 repeats ⇒ ~427k distinct images per
"epoch", so early epochs look slower than v10's — that is expected, not a bug.

**Loading — the hard-won rules are now enforced, not remembered:**
- a missing Arrow snapshot **raises with the build command** instead of
  triggering a silent 160 GB rebuild;
- the image/label columns are found by *feature type*, not by name (HF vs TFDS
  builds name them differently);
- fork workers + persistent workers + prefetch;
- `val_batch_multiplier`, and a val split falling back to `test` only with a
  loud warning.

The v10 cells that hand-built `load_dataset(...)` paths are replaced by
`build_dataloaders(cfg)`.


In [ ]:
# ═══ pvt_moe/data/imagenet.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""Image data pipeline (HF Arrow, map-style): ImageNet 1k / 22k, PASS (SSL), small sets.

Hard-won rules from this project's history (do not regress):

- **Map-style only.** ``load_from_disk`` memory-maps the Arrow files; a
  map-style ``Dataset`` gives O(1) random access and truly parallel workers.
  ``streaming=True`` (IterableDataset) was tried and was drastically slower.
- **Never rebuild silently.** A missing Arrow snapshot raises with
  instructions instead of kicking off a ~160 GB rebuild.
- Workers died (batch 1024 x 12 workers) when prefetch was oversized; the
  loader settings below are the stable configuration.

Augmentation follows the DeiT-1 stack that PVT v2 inherits. Two pieces need
timm rather than torchvision:

- **RandAugment** is specified as timm's config string
  (``rand-m9-mstd0.5-inc1``): magnitude 9, magnitude-std 0.5, *increasing*
  severity. torchvision's ``RandAugment`` supports neither the magnitude
  jitter nor the increasing-severity op set, so it is only a fallback
  (``dataset.randaugment: None``).
- **Repeated augmentation** (3 repeats) is a *sampler*, not a transform. The
  epoch keeps its LENGTH (same number of steps); what changes is that it draws
  only ~1/3 as many distinct images, each 3 times with different augmentation.
  It therefore costs no extra time and buys none either — the effect is on
  gradient variance, not throughput. ``dataset.repeated_aug: 1`` disables it.

  The sampler shuffles deterministically from ``self.epoch``; Lightning's fit
  loop advances it via ``_set_sampler_epoch``. If that ever stopped happening,
  every epoch would redraw the SAME third of the dataset —
  ``tests/test_data_sampler.py`` runs a real fit to catch that.

ImageNet-22k: same Arrow layout expected at ``dataset.arrow_dirs['imagenet-22k']``
with 21841 classes (fall11 full-tag convention). Build it once with
``datasets``' parquet loader + ``save_to_disk`` (see README). The label
column name is probed, not assumed.
"""

from __future__ import annotations

import os
import sys

import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

_LABEL_KEYS = ("label", "labels", "cls", "fine_label")


def _build_randaugment(ds: dict):
    """RandAugment op: timm's config string when given, else torchvision."""
    spec = ds.get("randaugment")
    if isinstance(spec, str):
        from timm.data import rand_augment_transform  # lazy

        # timm needs the target size + fill colour to place its geometric ops.
        hparams = {
            "translate_const": int(ds["img_size"] * 0.45),
            "img_mean": tuple(round(255 * c) for c in IMAGENET_MEAN),
        }
        return rand_augment_transform(spec, hparams)
    if isinstance(spec, (list, tuple)):  # legacy [ops, magnitude] form
        return transforms.RandAugment(*spec)
    return transforms.RandAugment(
        ds.get("randaugment_ops", 2), ds.get("randaugment_magnitude", 9)
    )


def build_transforms(cfg: dict):
    """(train, val) transforms — the DeiT-1 stack PVT v2 uses."""
    ds = cfg["dataset"]
    img_size = ds["img_size"]
    train_tf = transforms.Compose(
        [
            transforms.RandomResizedCrop(img_size),
            transforms.RandomHorizontalFlip(),
            # RandAugment runs on the PIL image, before ToTensor.
            _build_randaugment(ds),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            transforms.RandomErasing(p=ds["random_erasing"]),
        ]
    )
    val_tf = transforms.Compose(
        [
            transforms.Resize(int(img_size / ds["crop_pct"])),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )
    return train_tf, val_tf


def _image_column(hf_split) -> str:
    """The column holding the image, found by FEATURE TYPE, not by name —
    the HF and TFDS builds of the same corpus name their fields differently."""
    from datasets import Image  # lazy

    cols = [c for c, f in hf_split.features.items() if isinstance(f, Image)]
    if len(cols) != 1:
        raise KeyError(
            f"expected exactly one Image column, found {cols} in {sorted(hf_split.column_names)}")
    return cols[0]


class HFImageDataset(Dataset):
    """Map-style wrapper over a HF Arrow split.

    ``labelled=True`` probes the label column and yields ``(image, label)``.
    ``labelled=False`` (PASS) keeps ONLY the image column — creator name,
    date and GPS metadata are never read — and yields ``(image, -1)`` so the
    batch shape stays what the SSL module expects (it discards the label).
    """

    def __init__(self, hf_split, transform=None, labelled: bool = True):
        self.transform = transform
        self.image_key = _image_column(hf_split)
        self.label_key = None
        self.dropped_columns = []
        if labelled:
            columns = set(hf_split.column_names)
            for key in _LABEL_KEYS:
                if key in columns:
                    self.label_key = key
                    break
            else:
                raise KeyError(
                    f"No label column found; columns={sorted(columns)}, tried {_LABEL_KEYS}"
                )
            self.dataset = hf_split
        else:
            self.dropped_columns = sorted(set(hf_split.column_names) - {self.image_key})
            self.dataset = hf_split.select_columns([self.image_key])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item[self.image_key]
        if image.mode != "RGB":
            image = image.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, (item[self.label_key] if self.label_key is not None else -1)


def build_datasets(cfg: dict):
    """Load the Arrow snapshot for ``cfg.dataset.name`` -> (train_ds, val_ds).

    ``val_ds`` is None for a corpus with no validation split (PASS has only
    ``train``): SSL then trains with no validation loader at all — the
    monitored quantity is the training ``ssl_loss`` and the real evaluation
    is the linear probe on a LABELLED dataset (docs/JEPA_GUIDE.md §5).
    """
    # [v12] inlined above: was `from pvt_moe.config import ...`

    ds_cfg = cfg["dataset"]
    labelled = DATASETS[ds_cfg["name"]]["labelled"]
    if not labelled and cfg.get("task") != "ssl":
        # validate_config refuses this already; this is the last line of
        # defence so an unlabelled corpus never reaches a classifier loader.
        raise ValueError(f"dataset {ds_cfg['name']!r} is unlabelled: SSL (task 'ssl') only")
    arrow_dir = ds_cfg["arrow_dirs"][ds_cfg["name"]]
    # Check the path BEFORE the heavy import: a missing snapshot should say so,
    # not surface as ModuleNotFoundError on a box where `datasets` is absent.
    if not os.path.isdir(arrow_dir) and ds_cfg["name"] == "pass":
        raise FileNotFoundError(
            f"No Arrow snapshot at {arrow_dir} for PASS. Build it once (no HF token, "
            "~166 GB final, ~333 GB free while building). PASS is NOT on the Hub any "
            "more — its repo ships a loading script datasets 5.0 cannot run — so the "
            "images come from Zenodo via the dataset's own script, in two steps:\n"
            "  git clone https://github.com/yukimasano/PASS\n"
            "  cd PASS && bash download.sh /data/pass_jpg\n"
            f"  python download_data.py --dataset pass --from-images /data/pass_jpg "
            f"--out {arrow_dir}\n"
            "then re-run (and `rm -rf /data/pass_jpg` once it prints done)."
        )
    if not os.path.isdir(arrow_dir):
        raise FileNotFoundError(
            f"No Arrow snapshot at {arrow_dir} for {ds_cfg['name']}.\n"
            "Deliberate: automatic rebuilds are disabled (a rebuild downloads/"
            "writes ~160 GB for 1k, and roughly 1.3 TB for 22k — and needs "
            "about TWICE that free while building, since `datasets` keeps the "
            "raw download and the Arrow cache at once). Build it once:\n"
            f"  python download_data.py --out {arrow_dir}\n"
            "which checks the licence, HF_TOKEN and free space first. By hand:\n"
            "  from datasets import load_dataset\n"
            "  d = load_dataset('ILSVRC/imagenet-1k')   # full namespace/name; a\n"
            "                                          # bare id is rejected\n"
            f"  d.save_to_disk({arrow_dir!r})\n"
            "then re-run."
        )

    from datasets import DatasetDict  # lazy — heavy import

    raw = DatasetDict.load_from_disk(arrow_dir)
    val_split = next((s for s in ("validation", "val") if s in raw), None)
    if val_split is None and labelled and "test" in raw:
        # A hand-built snapshot with train/test only. The per-epoch metric
        # then IS the test split; with the fixed per-dataset epoch budget
        # nothing is tuned on it, but the top-k checkpoint by val_acc is
        # selected on it. download_data.py carves a seeded validation split
        # for the small sets so this branch is never needed for them.
        val_split = "test"
        print(f"[data] WARNING: {ds_cfg['name']} snapshot has no validation split; using "
              "'test' for the per-epoch metric. Rebuild with download_data.py (seeded "
              "validation carve-out) for a clean protocol.")

    train_tf, val_tf = build_transforms(cfg)
    train_ds = HFImageDataset(raw["train"], transform=train_tf, labelled=labelled)
    val_ds = (HFImageDataset(raw[val_split], transform=val_tf, labelled=labelled)
              if val_split is not None else None)
    if labelled:
        native = raw["train"][0][train_ds.image_key].size if len(train_ds) else None
        print(f"[data] {ds_cfg['name']}: train={len(train_ds):,} val={len(val_ds):,} "
              f"({ds_cfg['num_classes']} classes, label column '{train_ds.label_key}', "
              f"val split '{val_split}')")
        if native is not None and max(native) < ds_cfg["img_size"]:
            print(f"[data] native {native[0]}x{native[1]} images are UPSAMPLED to "
                  f"{ds_cfg['img_size']}x{ds_cfg['img_size']} by the transforms: results on "
                  "this dataset partly measure interpolation (docs/GUIDE.md).")
        subset = ds_cfg.get("subset_file")
        if subset:
            # [v12] inlined above: was `from pvt_moe.eval.lowshot import ...`

            indices, meta = load_subset(subset, expect_dataset=ds_cfg["name"],
                                        expect_len=len(train_ds))
            train_ds = torch.utils.data.Subset(train_ds, indices)
            print(f"[data] low-shot subset {subset}: {len(indices):,} of {meta['total']:,} train "
                  f"images ({meta['fraction']:.1%}, seed {meta['seed']}, class-balanced); the "
                  "validation split is untouched")
    else:
        print(f"[data] {ds_cfg['name']}: train={len(train_ds):,} images, unlabelled | "
              f"snapshot features: {list(raw['train'].features)} | using only "
              f"'{train_ds.image_key}', dropped {train_ds.dropped_columns}")
        if val_ds is None:
            print("[data] no validation split in this corpus: SSL runs with no "
                  "validation loader; evaluation is the linear probe on a labelled set")
    return train_ds, val_ds


#: RandomResizedCrop scale range per SSL method. SimMIM: (0.67, 1) with the
#: default 3/4-4/3 aspect range (microsoft/SimMIM data/data_simmim.py);
#: JEPA: the wider (0.3, 1) this repo's I-JEPA recipe shipped with.
SSL_CROP_SCALE = {"simmim": (0.67, 1.0), "jepa": (0.3, 1.0)}


def build_ssl_transform(cfg: dict):
    """Pretraining transform: random resized crop + flip ONLY. Neither method
    uses heavy augmentation — the masking objective supplies the pressure."""
    img_size = cfg["dataset"]["img_size"]
    method = (cfg.get("ssl") or {}).get("method", "simmim")
    scale = SSL_CROP_SCALE.get(method, SSL_CROP_SCALE["simmim"])
    return transforms.Compose(
        [
            transforms.RandomResizedCrop(img_size, scale=scale, ratio=(3.0 / 4.0, 4.0 / 3.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )


def build_dataloaders(cfg: dict, ssl: bool | None = None):
    """(train_loader, val_loader) with the project's stable loader settings.

    ``ssl=True`` (default when ``cfg["task"] == "ssl"``) swaps the train
    transform for the SSL method's crop + flip recipe (labels are still
    returned — the SSL module ignores them; the val loader keeps the standard
    eval transform for linear probing). ``val_loader`` is None when the corpus
    has no validation split.
    """
    if ssl is None:
        ssl = cfg.get("task") == "ssl"
    train_ds, val_ds = build_datasets(cfg)
    if ssl:
        base = train_ds.dataset if isinstance(train_ds, torch.utils.data.Subset) else train_ds
        base.transform = build_ssl_transform(cfg)

    # Repeated augmentation: a sampler, not a transform. Disabled for SSL —
    # neither SimMIM nor JEPA uses it; the masking objective supplies the
    # pressure and seeing the same image 3x per batch would weaken the signal.
    repeats = 1 if ssl else int(cfg["dataset"].get("repeated_aug", 1) or 1)
    train_sampler = None
    if repeats > 1:
        from timm.data.distributed_sampler import RepeatAugSampler  # lazy

        # RepeatAugSampler calls dist.get_world_size() unconditionally when
        # num_replicas is None, which raises on a single-process run (no
        # process group). Supply the degenerate values ourselves unless
        # torch.distributed is actually up.
        if torch.distributed.is_available() and torch.distributed.is_initialized():
            replicas, rank = None, None
        else:
            replicas, rank = 1, 0
        train_sampler = RepeatAugSampler(
            train_ds, num_replicas=replicas, rank=rank, num_repeats=repeats
        )
        print(f"[data] repeated augmentation: {repeats} repeats | "
              f"{len(train_sampler):,} samples/epoch "
              f"(~{len(train_sampler) // repeats:,} distinct images)")

    num_workers = cfg["num_workers"]
    # fork is measurably faster for HF Arrow datasets, but is Linux-only.
    mp_ctx = "fork" if sys.platform == "linux" and num_workers > 0 else None
    common = dict(
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=num_workers > 0,
        prefetch_factor=2 if num_workers > 0 else None,
        multiprocessing_context=mp_ctx,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg["batch_size"],
        # A sampler and shuffle=True are mutually exclusive in DataLoader.
        sampler=train_sampler,
        shuffle=(train_sampler is None),
        drop_last=True,
        **common,
    )
    if val_ds is None:
        return train_loader, None
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg["batch_size"] * cfg["val_batch_multiplier"],
        shuffle=False,
        drop_last=False,
        **common,
    )
    return train_loader, val_loader


In [ ]:
# ── dataloaders (raises with the build command if the snapshot is missing) ──
train_loader, val_loader = build_dataloaders(cfg)


## Δ since v10 — norms and RoPE

**Norms.** v10 flirted with BN for stages 1–3 (the `ALL_EDITS` BN experiment)
and RMSNorm via a `NORM_LAYER_STAGE4` global; the ablation is now one config
key with the stage-4 guard in code:
<pre>
<del>NORM_LAYER = nn.LayerNorm; NORM_LAYER_STAGE4 = None   # globals wired by hand</del>
cfg["model"]["norm_type"] = "layernorm" | "rmsnorm"
cfg["model"]["stage4_keeps_layernorm"] = True   # Tutel's MoE block stays LN
</pre>

**RoPE.** v10: fixed axial frequencies, complex multiply, theta 50, stage 4
only, queries and keys on the same grid.
<pre>
<del># ── 2D Rotary Position Embedding (Axial RoPE — complex-mul, from rope-vit) ──</del>
<del>freqs = 1.0 / (theta ** (torch.arange(0, dim, 4)[: dim // 4] / dim))   # fixed ladder</del>
init_mixed_freqs(...)   # LEARNABLE per-head (ω_x, ω_y) pairs — RoPE-Mixed (rope-vit)
</pre>
- keys are rotated on the **reduced** SRA grid but in full-grid units
  (`scale_h = H / H_kv`), so q–k relative phases stay geometric when
  `sr_ratio > 1` — v10 only ever placed RoPE where `sr_ratio == 1`, so it never
  faced this;
- the frequencies are **parameters**: excluded from weight decay
  (`no_weight_decay`), snapshotted by `RopeFreqSnapshot` at step 0 and every
  epoch (`rope_freqs_init.pt` / `rope_freqs_final.pt`) so drift is plottable
  (`tools/plot_rope_freqs.py`);
- `mode="axial"` keeps v10's behaviour as the `-ax` ablation arm.


In [ ]:
# ═══ pvt_moe/models/norms.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""Normalization layers and the norm-type ablation factory.

RMSNorm uses the fused ``torch.nn.RMSNorm`` (PyTorch >= 2.4, CUDA-fused via
``F.rms_norm``) with a dtype-cast forward so the fused kernel dispatches
correctly under bf16-mixed autocast, and falls back to a hand-written
implementation on older PyTorch. Ported from the archive RMS_FullFT notebook.
"""

from __future__ import annotations

from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F

_TORCH_HAS_FUSED_RMSNORM = hasattr(nn, "RMSNorm")


if _TORCH_HAS_FUSED_RMSNORM:

    class RMSNorm(nn.RMSNorm):
        """Fused RMSNorm. Subclassed so ``isinstance`` checks keep working.

        Under bf16-mixed autocast the input arrives in bf16 while the weight
        is fp32; ``F.rms_norm`` requires matching dtypes for the fused CUDA
        kernel, so we cast the weight to the input dtype on mismatch.
        """

        def __init__(self, dim, eps: float = 1e-6, **kwargs):
            super().__init__(dim, eps=eps)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            if self.weight.dtype != x.dtype:
                return F.rms_norm(x, (self.weight.shape[0],), self.weight.to(x.dtype), self.eps)
            return super().forward(x)

else:

    class RMSNorm(nn.Module):
        """Reference RMSNorm for PyTorch < 2.4 (no fused kernel available)."""

        def __init__(self, dim, eps: float = 1e-6, **kwargs):
            super().__init__()
            self.eps = eps
            self.weight = nn.Parameter(torch.ones(dim))

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            rms = x.float().pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
            return (x.float() / rms).type_as(x) * self.weight


def rmsnorm_backend() -> str:
    """Human-readable description of the active RMSNorm implementation."""
    return (
        "torch.nn.RMSNorm (fused)" if _TORCH_HAS_FUSED_RMSNORM
        else "custom RMSNorm (PyTorch < 2.4 fallback)"
    )


def build_norm_layers(norm_type: str, eps: float, stage4_keeps_layernorm: bool):
    """Return ``(norm_main, norm_last_stage)`` layer factories for the model.

    - ``norm_type="layernorm"``: LayerNorm everywhere (v9 behavior).
    - ``norm_type="rmsnorm"``:   RMSNorm in stages 1..N-1; the last stage keeps
      LayerNorm when ``stage4_keeps_layernorm`` (archive precedent: the MoE
      stage stays closest to pretrained LN statistics and the router input
      stays mean-centered), else RMSNorm everywhere.
    """
    layer_norm = partial(nn.LayerNorm, eps=eps)
    if norm_type == "layernorm":
        return layer_norm, layer_norm
    if norm_type == "rmsnorm":
        rms = partial(RMSNorm, eps=eps)
        return rms, (layer_norm if stage4_keeps_layernorm else rms)
    raise ValueError(f"Unknown norm_type: {norm_type!r}")


In [ ]:
# ═══ pvt_moe/models/rope.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""2D Rotary Position Embedding (complex-multiplication form): Mixed or Axial.

Follows "Rotary Position Embedding for Vision Transformer" (Heo et al.,
ECCV 2024, "rope-vit"; reference: naver-ai/rope-vit ``deit/models_v2_rope.py``
@ 48d8df50). Two modes:

- **axial** — the head dimension is split so that half the frequency pairs
  rotate with the x coordinate and half with the y coordinate; frequencies
  are fixed (``theta`` = 100 in the paper, 50 here); caches per (H, W).
- **mixed** (RoPE-Mixed, the default) — every frequency channel *c* of every
  head *h* is a LEARNABLE 2D vector (ω_x, ω_y) and the phase at position
  (x, y) is ω_x·x + ω_y·y. The parameter is ``freqs`` of shape
  ``(2, num_heads, head_dim // 2)``, index 0 = ω_x, index 1 = ω_y — the same
  layout as the reference's ``init_random_2d_freqs`` (there the per-layer
  tensors are further stacked into one ``(2, depth, heads * head_dim // 2)``
  model-level parameter; here each attention module owns its own). Init:
  the axial magnitudes ``1 / theta ** (4k / head_dim)`` rotated by one random
  angle per head (first ``head_dim // 4`` channels at φ_h, the second at
  φ_h + π/2), ``theta`` = 10 as in the reference. Excluded from weight decay
  (reference ``no_weight_decay``). The phase is computed in fp32 every call.

Numerical note: the rotation is performed in fp32 (``x.float()``) and cast
back to the input dtype. Under bf16-mixed this is intentional — complex
multiplication needs fp32 phase accuracy; the cache stays complex64.

In this architecture RoPE exists primarily to reinject positional information
into MoE blocks: the Tutel/MegaBlocks expert FFN replaces the dense Mlp that
carried PVT v2's depthwise-conv positional encoding (DWConv), so MoE blocks
would otherwise be position-blind.
"""

from __future__ import annotations

import contextlib

import torch
import torch.nn as nn


def _init_t_xy(end_x: int, end_y: int, scale_x: float = 1.0, scale_y: float = 1.0):
    """Map flattened (row-major) sequence positions to 2D (x, y) coordinates.

    ``scale_*`` expresses the coordinates of a REDUCED grid in the units of
    the full grid: cell *i* of a grid reduced by factor *s* covers full-grid
    cells ``[i*s, (i+1)*s)`` and its center is ``(i + 0.5)*s - 0.5``. With
    scale 1 this is exactly ``i`` (bit-identical to the unscaled cache).
    """
    t = torch.arange(end_x * end_y, dtype=torch.float32)
    t_x = (t % end_x).float()
    t_y = torch.div(t, end_x, rounding_mode="floor").float()
    if scale_x != 1.0:
        t_x = (t_x + 0.5) * scale_x - 0.5
    if scale_y != 1.0:
        t_y = (t_y + 0.5) * scale_y - 0.5
    return t_x, t_y


def compute_axial_cis(
    dim: int,
    end_x: int,
    end_y: int,
    theta: float = 100.0,
    scale_x: float = 1.0,
    scale_y: float = 1.0,
) -> torch.Tensor:
    """Build the complex frequency cache for axial 2D RoPE.

    Returns a complex64 tensor of shape ``(end_x * end_y, dim // 2)``.
    """
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 4)[: (dim // 4)].float() / dim))
    t_x, t_y = _init_t_xy(end_x, end_y, scale_x, scale_y)
    freqs_x = torch.outer(t_x, freqs)
    freqs_y = torch.outer(t_y, freqs)
    return torch.cat(
        [
            torch.polar(torch.ones_like(freqs_x), freqs_x),
            torch.polar(torch.ones_like(freqs_y), freqs_y),
        ],
        dim=-1,
    )


def apply_rotary_emb(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """Rotate ``x`` (B, heads, seq, head_dim) by complex phases.

    ``freqs_cis`` is ``(seq, head_dim//2)`` (axial, shared by all heads) or
    ``(heads, seq, head_dim//2)`` (mixed, one set per head).
    """
    x_ = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    if freqs_cis.ndim == 2:
        freqs_cis = freqs_cis[None, None]            # (1, 1, seq, head_dim//2)
    elif freqs_cis.ndim == 3:
        if freqs_cis.shape[0] != x_.shape[1]:
            raise ValueError(
                f"per-head freqs_cis has {freqs_cis.shape[0]} heads, x has {x_.shape[1]}")
        freqs_cis = freqs_cis[None]                  # (1, heads, seq, head_dim//2)
    else:
        raise ValueError(f"freqs_cis must be 2-D or 3-D, got {freqs_cis.ndim}-D")
    return torch.view_as_real(x_ * freqs_cis.to(x_.device)).flatten(-2).type_as(x)


ROPE_MODES = ("mixed", "axial")


def init_mixed_freqs(
    head_dim: int, num_heads: int, theta: float = 10.0, rotate: bool = True
) -> torch.Tensor:
    """RoPE-Mixed initial frequencies, shape ``(2, num_heads, head_dim // 2)``.

    Port of rope-vit ``init_random_2d_freqs``. ``[0]`` is ω_x, ``[1]`` is
    ω_y. Magnitudes are the axial ladder ``1 / theta ** (4k / head_dim)``
    (``head_dim // 4`` of them); each head gets one random angle φ_h (from
    the global torch RNG — seed it) and the two halves of the channel axis
    sit at φ_h and φ_h + π/2. ``rotate=False`` gives φ_h = 0, which is
    exactly the axial layout (x-channels then y-channels).
    """
    if head_dim % 4 != 0:
        raise ValueError(f"head_dim must be divisible by 4 for 2D RoPE, got {head_dim}")
    mag = 1.0 / (theta ** (torch.arange(0, head_dim, 4)[: (head_dim // 4)].float() / head_dim))
    fx, fy = [], []
    for _ in range(num_heads):
        angle = torch.rand(1) * 2 * torch.pi if rotate else torch.zeros(1)
        fx.append(torch.cat([mag * torch.cos(angle), mag * torch.cos(torch.pi / 2 + angle)], dim=-1))
        fy.append(torch.cat([mag * torch.sin(angle), mag * torch.sin(torch.pi / 2 + angle)], dim=-1))
    return torch.stack([torch.stack(fx, dim=0), torch.stack(fy, dim=0)], dim=0)


def compute_mixed_cis(freqs: torch.Tensor, t_x: torch.Tensor, t_y: torch.Tensor) -> torch.Tensor:
    """Complex phases ``exp(i(ω_x·x + ω_y·y))`` for learnable ``freqs``.

    ``freqs`` is ``(2, num_heads, head_dim // 2)``; ``t_x``/``t_y`` are the
    ``N`` flattened coordinates. Returns complex64 ``(num_heads, N,
    head_dim // 2)``. Always fp32 — the phase range needs it (the reference
    disables autocast here for the same reason).
    """
    if freqs.ndim != 3 or freqs.shape[0] != 2:
        raise ValueError(f"mixed freqs must be (2, heads, head_dim//2), got {tuple(freqs.shape)}")
    # Explicit .float() upcasts keep the phase fp32 on their own; the autocast
    # guard is belt-and-braces on the device types that support it (a meta
    # device, used for shape tracing, does not).
    dev = freqs.device.type
    guard = (torch.autocast(device_type=dev, enabled=False)
             if dev in ("cpu", "cuda") else contextlib.nullcontext())
    with guard:
        f = freqs.float()
        phase = (t_x.float()[None, :, None] * f[0][:, None, :]
                 + t_y.float()[None, :, None] * f[1][:, None, :])   # (heads, N, D/2)
        return torch.polar(torch.ones_like(phase), phase)


class RotaryEmbedding2D(nn.Module):
    """2D RoPE for one attention module, ``mode`` = "mixed" (default) | "axial".

    axial: no parameters; the cache is keyed by (H, W, scale, device) and
    stored on the target device so repeated calls avoid host-to-device copies.
    mixed: one learnable ``freqs`` parameter ``(2, num_heads, head_dim // 2)``
    (see ``init_mixed_freqs``); the phases are recomputed every call because
    the frequencies change every step, and ``get`` returns a per-head tensor
    ``(num_heads, N, head_dim // 2)``.

    ``scale_h``/``scale_w`` express a REDUCED grid's coordinates in full-grid
    units — required so that Q (full grid) and K (SR-reduced grid) rotate in
    the SAME coordinate system and relative phases stay meaningful. With
    scale 1 the axial cache is bit-identical to the unscaled (v9) one.
    """

    def __init__(self, head_dim: int, theta: float = 10.0, mode: str = "mixed",
                 num_heads: int | None = None):
        super().__init__()
        if head_dim % 4 != 0:
            raise ValueError(f"head_dim must be divisible by 4 for 2D RoPE, got {head_dim}")
        if mode not in ROPE_MODES:
            raise ValueError(f"rope mode must be one of {ROPE_MODES}, got {mode!r}")
        self.head_dim = head_dim
        self.theta = theta
        self.mode = mode
        self._cache: dict = {}
        if mode == "mixed":
            if not num_heads:
                raise ValueError("mixed RoPE needs num_heads (one frequency set per head)")
            self.num_heads = num_heads
            self.freqs = nn.Parameter(init_mixed_freqs(head_dim, num_heads, theta))

    def get(
        self,
        H: int,
        W: int,
        device: torch.device,
        scale_h: float = 1.0,
        scale_w: float = 1.0,
    ) -> torch.Tensor:
        key = (
            H, W, float(scale_h), float(scale_w),
            device.index if device.type == "cuda" else str(device.type),
        )
        if self.mode == "mixed":
            # The coordinates are constants: cache them per grid/device like
            # the axial cis (the reference registers t_x/t_y as buffers).
            # Only the phase depends on the learnable freqs and is recomputed.
            if key not in self._cache:
                t_x, t_y = _init_t_xy(W, H, scale_x=scale_w, scale_y=scale_h)
                self._cache[key] = (t_x.to(device), t_y.to(device))
            t_x, t_y = self._cache[key]
            return compute_mixed_cis(self.freqs, t_x, t_y)
        if key not in self._cache:
            self._cache[key] = compute_axial_cis(
                self.head_dim, W, H, self.theta, scale_x=scale_w, scale_y=scale_h
            ).to(device)
        return self._cache[key]


## Δ since v10 — attention

<pre>
<del>class GQAttention(nn.Module):                       # grouped-query attention</del>
<del>    self.kv = nn.Linear(dim, 2 * num_kv_heads * self.head_dim, bias=qkv_bias)</del>
<del>    if self.num_kv_heads == self.num_heads:</del>
<del>        out = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p)</del>
<del>    elif _SDPA_HAS_GQA:                             # torch >= 2.5</del>
<del>        out = F.scaled_dot_product_attention(q, k, v, ..., enable_gqa=True)</del>
<del>    else:</del>
<del>        out = F.scaled_dot_product_attention(q, k.repeat_interleave(groups, 1), ...)</del>
class SRAttention(nn.Module):                       # plain multi-head
    self.kv = nn.Linear(dim, 2 * dim, bias=qkv_bias)
    out = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p)
</pre>
*why:* the ablation grid never runs GQA, mixed RoPE requires MHA, and the
single unmasked SDPA call is flash-eligible on CUDA under bf16 (head_dim
32/64, no mask) with nothing to install. Removal verified function-identical
on b1/b2 before landing; the q/kv projection names and shapes are unchanged,
so every existing checkpoint still loads. The spatial-reduction (`sr`) path
and PVT v2-li pooling are exactly v10's.


In [ ]:
# ═══ pvt_moe/models/attention.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""Spatial-reduction multi-head attention with optional 2D RoPE.

This is the attention used throughout the backbone:

- **SRA** (PVT v2): keys/values are computed on a spatially reduced feature
  map — a strided ``sr_ratio`` conv (standard mode) or adaptive 7x7 average
  pooling (``linear_attention`` mode, "PVT v2-li").
- **MHA through SDPA**: attention is the unmasked
  ``F.scaled_dot_product_attention`` call, one kv head per query head, which
  dispatches to the flash kernel on CUDA under bf16/fp16 (head_dim 32/64,
  no mask) — nothing to install or enable. Q and the fused KV keep separate
  projections (``q`` / ``kv``), the official PVT v2 layout the HF remap in
  ``models/pretrained.py`` loads into.
- **RoPE** (optional; ``rope_mode`` "mixed" = learnable per-head 2D
  frequencies, the default, or "axial" = fixed): queries are rotated on the
  full (H, W) grid, keys on the reduced (H_kv, W_kv) grid; values are never
  rotated. Coordinates scale correctly because the phases depend only on
  grid coordinates, and mixed frequencies are per head — keys carry the same
  head count as queries.
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

# [v12] inlined above: was `from pvt_moe.models.rope import ...`


class SRAttention(nn.Module):
    """SR-Attention (multi-head) with optional 2D RoPE."""

    def __init__(
        self,
        dim: int,
        num_heads: int,
        qkv_bias: bool = True,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
        sr_ratio: int = 1,
        linear_attention: bool = False,
        norm_layer=nn.LayerNorm,
        use_rope: bool = False,
        rope_theta: float = 10.0,
        rope_mode: str = "mixed",
    ):
        super().__init__()
        if dim % num_heads != 0:
            raise ValueError(f"dim {dim} must be divisible by num_heads {num_heads}")
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.use_rope = use_rope

        self.q = nn.Linear(dim, dim, bias=qkv_bias)
        self.kv = nn.Linear(dim, 2 * dim, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        self.linear_attention = linear_attention
        self.sr_ratio = sr_ratio
        if not linear_attention:
            if sr_ratio > 1:
                self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio)
                self.norm = norm_layer(dim)
        else:
            self.pool = nn.AdaptiveAvgPool2d(7)
            self.sr = nn.Conv2d(dim, dim, kernel_size=1, stride=1)
            self.norm = norm_layer(dim)
            self.act = nn.GELU()

        if use_rope:
            self.rope = RotaryEmbedding2D(self.head_dim, theta=rope_theta,
                                          mode=rope_mode, num_heads=num_heads)

    def forward(self, x: torch.Tensor, H: int, W: int) -> torch.Tensor:
        B, N, C = x.shape
        q = self.q(x).reshape(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        # Spatially reduced context for K/V (tracks H_kv, W_kv for RoPE).
        if not self.linear_attention:
            if self.sr_ratio > 1:
                x_ = x.transpose(1, 2).reshape(B, C, H, W)
                x_ = self.sr(x_).reshape(B, C, -1).transpose(1, 2)
                x_ = self.norm(x_)
                H_kv, W_kv = H // self.sr_ratio, W // self.sr_ratio
            else:
                x_ = x
                H_kv, W_kv = H, W
        else:
            x_ = x.transpose(1, 2).reshape(B, C, H, W)
            x_ = self.sr(self.pool(x_)).reshape(B, C, -1).transpose(1, 2)
            x_ = self.act(self.norm(x_))
            H_kv, W_kv = 7, 7

        kv = self.kv(x_).reshape(B, -1, 2, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]

        if self.use_rope:
            # Q rotates on the full grid; K on the reduced grid but expressed
            # in FULL-grid units (scale_h/scale_w) so q–k relative phases stay
            # geometrically meaningful when sr_ratio > 1 or in linear mode.
            # For sr_ratio == 1 the scales are 1 → identical to v9 behavior.
            q_cis = self.rope.get(H, W, x.device)
            q = apply_rotary_emb(q, q_cis)
            # Same grid (sr_ratio 1, e.g. stage 4): the key phases are the
            # query phases — no second computation.
            k_cis = q_cis if (H_kv, W_kv) == (H, W) else self.rope.get(
                H_kv, W_kv, x.device, scale_h=H / H_kv, scale_w=W / W_kv)
            k = apply_rotary_emb(k, k_cis)

        dropout_p = self.attn_drop.p if self.training else 0.0
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p)

        out = out.transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(out))


## Δ since v10 — the MoE FFN

**Kept from v10 (these were v10's own hard-won fixes):**
- `build_moe_ffn_layer` always passes `activation_fn` (Tutel's default branch
  references `F` without importing it — omitting the arg crashes inside
  Tutel's *forward*);
- `capacity_factor` / `gate_noise` ride **inside** `gate_type` — as plain
  kwargs Tutel silently ignores them;
- the always-on shared expert, and `moe_block_dwconv` riding it (the routed
  branch has no token grid to convolve over).

**New since v10:**
<pre>
<del># v10: nothing forced the gates back into train mode</del>
def force_tutel_gates_train(model):   # LOAD-BEARING — called from train() and
    ...                               # on_train_epoch_start; see the docstring
</pre>
&nbsp;&nbsp;*why:* after every Lightning validation pass Tutel's gate modules stay in
eval mode, silently disabling `gate_noise` — the exploration that keeps
experts balanced. v10 ran with this latent.

<pre>
<del># v10: Tutel or nothing</del>
class NativeMoEFFN(...)     # backend "native": pure-PyTorch top-k routing,
                            # same aux loss, no CUDA extension; what the CPU
                            # test suite exercises, and a fallback on any box
</pre>
- the native gate runs in **fp32 under autocast** (autocast would otherwise
  cast the `nn.Linear` itself — `x.float()` alone is not enough);
- dropped tokens are counted (`dropped_tokens`), feeding the RoutingMonitor's
  `train_drop_rate_realised`;
- upcycling (zero routed fc2 + seed the shared expert — v10's fix for the
  discarded stage-4 FFN) moved from model construction into the warm-start
  path (`upcycle_init`), where it also covers HF-pretrained starts, and is
  covered by function-preservation tests + `tools/verify_upcycling.py` on the
  real backend.


In [ ]:
# ═══ pvt_moe/models/moe_native.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""Pure-PyTorch MoE expert layer — the no-Tutel fallback.

Insurance, not a replacement. Tutel remains the default backend because it is
the one validated by the v9 lineage's runs; this exists so a teammate's
Windows/WSL2 box, a fresh GPU rental, or a Tutel build break cannot stop an
ablation. No custom CUDA, no NCCL, no compiler — just torch.

Drop-in contract
----------------
``NativeMoEFFN`` stands in for ``tutel_moe.moe_layer``: it takes flattened
``(tokens, model_dim)`` input and returns ``(output, aux_loss)``, so
``MoEMlp`` swaps backends without any other code changing. The shared expert
is NOT inside this module — it lives one level up in ``MoEMlp``, which is the
only place with the ``(B, N, C)`` grid its DWConv needs (a routed branch has
no grid; see docs/ARCHITECTURE.md section 2). Function preservation at init is
therefore the same story under both backends: shared expert carries the
pretrained FFN, routed fc2 starts at zero.

Checkpoint interchange
----------------------
Parameter names and shapes mirror Tutel's ``FusedExpertsNetwork`` exactly::

    experts.batched_fc1_w      (E, hidden, model_dim)
    experts.batched_fc2_w      (E, hidden, model_dim)   # stores fc2.T
    experts.batched_fc1_bias   (E, hidden)
    experts.batched_fc2_bias   (E, model_dim)
    gates.0.wg.weight          (E, model_dim)

so a run started on Tutel can be resumed on the native backend and vice
versa, and ``seed_moe_experts_from_dense`` / ``zero_routed_expert_output``
work against either without a special case. ``tests/test_native_moe.py``
asserts the key sets match.

Known deviations from Tutel, all deliberate
-------------------------------------------
- **top-1 only.** Our configs are top-1 and always have been; a top-k>1 path
  would be untested code shipped as a safety net, which is worse than a clear
  error.
- **No expert parallelism.** Every expert lives on one device. That is the
  point — no all-to-all, no NCCL.
- **Dense fallback loop over experts.** With E=4 the python loop costs less
  than the gather/scatter machinery it replaces, and it is readable.
"""

from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class Top1Router(nn.Module):
    """Linear gate -> softmax -> argmax, with Switch-style balancing loss.

    Kept as a submodule named ``wg`` so that ``gates.0.wg.weight`` matches
    Tutel's ``LinearTopKGate`` and the expert-utilization diagnostic works
    against either backend unchanged.
    """

    def __init__(self, model_dim: int, num_experts: int, gate_noise: float = 0.0):
        super().__init__()
        self.wg = nn.Linear(model_dim, num_experts, bias=False)
        self.num_experts = num_experts
        self.gate_noise = gate_noise
        # Sparse Upcycling B.9: random zero-mean normal, sigma 0.02.
        nn.init.normal_(self.wg.weight, mean=0.0, std=0.02)

    def forward(self, x: torch.Tensor):
        """Returns ``(expert_index, gate_value, aux_loss)`` for each token."""
        # Gate in fp32 regardless of autocast: an 8-way softmax in bf16 has
        # ~3 decimal digits, the routing decision is discrete, and the
        # load-balancing loss lives within ~1% of 1.0 where bf16's spacing is
        # 2^-8 = 0.0039 — quantising it to exactly 1.0.
        #
        # LOAD-BEARING: ``x.float()`` alone does NOT do this. torch.autocast
        # intercepts nn.Linear and casts the LAYER as well as the input, so
        # under bf16-mixed ``self.wg(x.float())`` returns bf16. Autocast has to
        # be turned off around the call. Tutel does the same thing around its
        # whole routing block (tutel/impls/moe_layer.py, `with
        # torch.amp.autocast('cuda', enabled=False): routing()`), which is why
        # its l_aux is fp32; this keeps the two backends comparable.
        if x.device.type in ("cpu", "cuda", "xpu"):
            with torch.autocast(device_type=x.device.type, enabled=False):
                logits = self.wg(x.float())
        else:                                     # a device autocast does not know
            logits = self.wg(x.float())

        if self.training and self.gate_noise > 0:
            logits = logits + self.gate_noise * torch.randn_like(logits) / self.num_experts

        probs = F.softmax(logits, dim=-1)
        gate, index = probs.max(dim=-1)

        # Switch Transformer load balancing:  E * sum_i f_i * P_i
        #   f_i = fraction of tokens routed to expert i   (mean of the one-hot)
        #   P_i = mean routing probability for expert i
        # Equals 1.0 under a perfectly uniform assignment, which is the same
        # scale as Tutel's gshard_loss — so `loss.aux_weight` (0.01) carries
        # over between backends without retuning.
        one_hot = F.one_hot(index, self.num_experts).to(probs.dtype)
        f = one_hot.mean(dim=0)
        p = probs.mean(dim=0)
        aux = self.num_experts * torch.sum(f * p)
        return index, gate, aux


class BatchedExperts(nn.Module):
    """E independent fc1 -> act -> fc2 FFNs held as batched tensors.

    Layout matches Tutel's ``FusedExpertsNetwork``, including the detail that
    ``batched_fc2_w`` stores fc2 **transposed** — Tutel's forward is
    ``matmul(h, batched_fc2_w)`` with no permute, so the stored tensor is
    ``fc2.weight.T``. Getting this wrong is silent: shapes match either way
    because hidden and model_dim differ only by a factor.
    """

    def __init__(self, model_dim: int, hidden: int, num_experts: int, activation_fn):
        super().__init__()
        self.model_dim, self.hidden, self.num_experts = model_dim, hidden, num_experts
        self.activation_fn = activation_fn

        self.batched_fc1_w = nn.Parameter(torch.empty(num_experts, hidden, model_dim))
        self.batched_fc2_w = nn.Parameter(torch.empty(num_experts, hidden, model_dim))
        self.batched_fc1_bias = nn.Parameter(torch.empty(num_experts, hidden))
        self.batched_fc2_bias = nn.Parameter(torch.empty(num_experts, model_dim))
        self.reset_parameters()

    @torch.no_grad()
    def reset_parameters(self):
        """Tutel's init for fc1; fc2 starts at ZERO.

        The zeroed output projection is the whole point: a routed expert emits
        exactly 0 whatever the gate says, so an upcycled block reproduces the
        pretrained dense FFN exactly at step 0 (`routed_zero_init`). fc2 still
        receives gradient from the first step (dL/dW2 = h^T g, and h != 0), so
        the experts are not frozen — `test_routed_experts_leave_zero_after_a_
        few_steps` proves that rather than assuming it.
        """
        stdv = 1.0 / math.sqrt(self.model_dim)
        self.batched_fc1_w.uniform_(-stdv, stdv)
        self.batched_fc1_bias.zero_()
        self.batched_fc2_w.zero_()
        self.batched_fc2_bias.zero_()

    def forward_one(self, x: torch.Tensor, expert: int) -> torch.Tensor:
        h = F.linear(x, self.batched_fc1_w[expert], self.batched_fc1_bias[expert])
        h = self.activation_fn(h)
        # batched_fc2_w[e] is (hidden, model_dim) == fc2.weight.T
        return h @ self.batched_fc2_w[expert] + self.batched_fc2_bias[expert]


class NativeMoEFFN(nn.Module):
    """Top-1 routed expert bank in pure PyTorch.

    Parameters
    ----------
    capacity_factor
        Each expert processes at most ``top_k * int(capacity_factor *
        ceil(tokens / E))`` tokens, matching Tutel's formula. ``<= 0`` means
        no cap (Tutel's dynamic capacity).

        **Overflow tokens are dropped**: they receive exactly zero from the
        routed branch. That is not just a quality loss, it changes gradient
        flow — a dropped token contributes nothing to any expert's gradient
        for that step, and its own upstream gradient arrives only through the
        shared expert and the residual. With a shared expert present (which
        this design requires) a dropped token still gets a full FFN, so
        overflow degrades gracefully instead of zeroing the block's output.
    """

    def __init__(self, model_dim: int, hidden_size_per_expert: int, num_experts: int,
                 top_k: int = 1, capacity_factor: float = 1.0, activation_fn=None,
                 gate_noise: float = 0.0):
        super().__init__()
        if top_k != 1:
            raise ValueError(
                f"NativeMoEFFN implements top-1 routing only, got top_k={top_k}. "
                "This is the fallback backend; use backend='tutel' for top-k > 1 "
                "rather than relying on an untested path."
            )
        self.model_dim = model_dim
        self.hidden = hidden_size_per_expert
        self.num_experts = num_experts
        self.top_k = top_k
        self.capacity_factor = capacity_factor

        self._dropped = None                  # realised overflow of the last forward
        self.experts = BatchedExperts(
            model_dim, hidden_size_per_expert, num_experts,
            activation_fn if activation_fn is not None else nn.GELU())
        self.gates = nn.ModuleList(
            [Top1Router(model_dim, num_experts, gate_noise=gate_noise)])
        # Mirrors Tutel's buffer so state_dicts interchange.
        self.register_buffer("_num_global_experts", torch.tensor(num_experts))

    def capacity_for(self, num_tokens: int) -> int:
        """Per-expert token cap. Tutel's formula; <= 0 disables the cap."""
        if self.capacity_factor <= 0:
            return num_tokens
        per_expert = math.ceil(num_tokens / self.num_experts)
        return max(1, self.top_k * int(self.capacity_factor * per_expert))

    def forward(self, x: torch.Tensor):
        """``(tokens, model_dim) -> ((tokens, model_dim), aux_loss)``."""
        if x.dim() != 2:
            raise ValueError(f"expected flattened (tokens, model_dim), got {tuple(x.shape)}")
        tokens = x.shape[0]
        index, gate, aux = self.gates[0](x)
        gate = gate.to(x.dtype)

        capacity = self.capacity_for(tokens)
        # Rank of each token within its own expert's queue, in token order.
        one_hot = F.one_hot(index, self.num_experts)
        rank = (one_hot.cumsum(dim=0) - 1).gather(1, index[:, None]).squeeze(1)
        kept = rank < capacity
        # Realised overflow, kept as a DEVICE TENSOR: `int(...)` here would be a
        # host-device sync on every routed forward for a number most steps never
        # read. `dropped_tokens` below materialises it on demand; RoutingMonitor
        # accumulates the tensor and syncs once per epoch.
        self._dropped = (~kept).sum()

        out = torch.zeros_like(x)
        for e in range(self.num_experts):
            sel = torch.nonzero((index == e) & kept, as_tuple=True)[0]
            if sel.numel() == 0:
                continue
            y = self.experts.forward_one(x[sel], e)
            # Scale by the raw softmax score, unnormalized — this is what
            # Tutel does at top_k=1 (normalize_gate only fires for top_k > 1,
            # see tutel/impls/fast_dispatch.py::extract_critical).
            out[sel] = y * gate[sel, None]
        return out, aux

    @property
    def dropped_tokens(self) -> int:
        """Tokens the last forward dropped for overflow (syncs on access)."""
        return 0 if self._dropped is None else int(self._dropped)

    def extra_repr(self) -> str:
        return (f"model_dim={self.model_dim}, hidden={self.hidden}, "
                f"num_experts={self.num_experts}, top_k={self.top_k}, "
                f"capacity_factor={self.capacity_factor}")


In [ ]:
# ═══ pvt_moe/models/ffn.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""Feed-forward layers: dense PVT v2 Mlp and the MoE replacement.

Two implementations sit behind one interface:

- ``Mlp``    — dense: fc1 -> DWConv (depthwise 3x3, PVT v2's positional
  encoding) -> GELU -> fc2. Returns a tensor.
- ``MoEMlp`` — sparse: a Tutel or MegaBlocks expert layer replacing the whole
  FFN. Returns ``(tensor, aux_loss)``.

ARCHITECTURAL INVARIANT: the routed MoE branch has **no DWConv**, so an MoE
block loses PVT v2's conv positional encoding. Enable RoPE in the same blocks
to reinject positional information (this is why the default config places RoPE
exactly where it places MoE) — or enable a shared expert, which can carry the
DWConv itself.

Shared expert (``moe.shared_expert``, DeepSeekMoE / Qwen-MoE style)
-------------------------------------------------------------------
An always-on dense FFN evaluated for EVERY token in parallel with the routed
experts; its output is added to theirs::

    y = routed_moe(x) + shared_expert(x)

It lives outside the backend layer (plain ``nn.Module``, ordinary data-parallel
parameter, no ``skip_allreduce``), so it works identically for both backends
and its weights survive whatever the backend does to its own experts. Two
reasons it matters here:

1. It is the only place a pretrained dense FFN can be kept *exactly* rather
   than copied into E experts — and with ``moe_block_dwconv`` it keeps the
   DWConv too, restoring the positional encoding the routed branch drops.
2. With ``upcycle_init="routed_zero"`` the routed experts' fc2 starts at zero,
   so at step 0 the block computes exactly the pretrained dense FFN and the
   routed experts learn a residual on top of it.

Cost: one extra dense FFN per token (no routing, no capacity), i.e. the block
goes from top-k to top-k+1 active FFNs per token.

Backend notes
-------------
Tutel (default):
  - gate: top-k with capacity_factor and gate_noise; the layer itself returns
    ``(output, l_aux)`` via ``result_func``.
  - Tutel gates revert themselves to eval mode after a Lightning validation
    pass, silently disabling gate_noise. ``pvt_moe.engine.classifier`` forces
    them back to train mode — that code is load-bearing.

MegaBlocks (dMoE, dropless — no capacity factor, no token dropping):
  - requires megablocks==0.10.0 (pins torch 2.7.x) + grouped_gemm==0.3.0;
    ``mlp_impl='grouped'`` is the only viable impl on modern torch (the
    'sparse' path was disabled upstream in v0.8.0).
  - ``bias`` is silently ignored by the grouped expert MLP, so we honestly
    set ``bias=False`` (the archive's failed attempt passed bias=True and
    silently lost all FFN biases).
  - the load-balancing loss lives in a module-global registry that is ONLY
    populated in training mode; the archive crashed by collecting it during
    Lightning's validation sanity check. We collect it only when
    ``self.training``.
  - ``capacity_factor`` and ``gate_noise`` from the config are no-ops here.
"""

from __future__ import annotations

import torch
import torch.nn as nn


class DWConv(nn.Module):
    """Depthwise 3x3 conv over the token grid — PVT v2's positional encoding."""

    def __init__(self, dim: int):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, 3, 1, 1, bias=True, groups=dim)

    def forward(self, x: torch.Tensor, H: int, W: int) -> torch.Tensor:
        B, N, C = x.shape
        x = x.transpose(1, 2).view(B, C, H, W)
        x = self.dwconv(x)
        return x.flatten(2).transpose(1, 2)


class Mlp(nn.Module):
    """Dense PVT v2 FFN: fc1 -> DWConv -> act -> fc2.

    ``use_dwconv=False`` drops the depthwise conv (and with it PVT v2's conv
    positional encoding), leaving a plain fc1 -> act -> fc2 FFN. Only the
    shared-expert branch of ``MoEMlp`` uses that form; the backbone's dense
    blocks always keep the DWConv.
    """

    def __init__(
        self,
        in_features: int,
        hidden_features: int,
        act_layer=nn.GELU,
        drop: float = 0.0,
        linear_attention: bool = False,
        use_dwconv: bool = True,
    ):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.dwconv = DWConv(hidden_features) if use_dwconv else None
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(drop)
        # PVT v2-li applies ReLU before the depthwise conv.
        self.relu = nn.ReLU(inplace=True) if linear_attention else None

    def forward(self, x: torch.Tensor, H: int, W: int) -> torch.Tensor:
        x = self.fc1(x)
        if self.relu is not None:
            x = self.relu(x)
        if self.dwconv is not None:
            x = self.dwconv(x, H, W)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        return self.drop(x)


class MoEMlp(nn.Module):
    """Mixture-of-experts FFN (Tutel or MegaBlocks behind one interface).

    ``forward`` returns ``(output, aux_loss)`` where ``aux_loss`` is the
    load-balancing loss for THIS layer (a scalar tensor; zero in eval mode
    for the megablocks backend).
    """

    def __init__(
        self,
        in_features: int,
        hidden_features: int,
        moe_cfg: dict,
        act_layer=nn.GELU,
        drop: float = 0.0,
    ):
        super().__init__()
        self.backend = moe_cfg["backend"]
        self.num_experts = moe_cfg["num_experts"]
        self.top_k = moe_cfg["top_k"]
        # Kept for the drop accounting in utils.diagnostics: Tutel's layer does
        # not expose the capacity it enforces, and megablocks has none at all.
        self.capacity_factor = moe_cfg.get("capacity_factor")
        self.in_features = in_features
        self.hidden_features = hidden_features
        self.drop = nn.Dropout(drop)

        # Always-on shared expert (None when disabled). Built with drop=0.0:
        # this module's single ``self.drop`` is applied once to the summed
        # output, so the shared branch must not drop twice.
        self.shared_expert = None
        if moe_cfg.get("shared_expert", False):
            self.shared_expert = Mlp(
                in_features,
                hidden_features,
                act_layer=act_layer,
                drop=0.0,
                use_dwconv=moe_cfg.get("moe_block_dwconv", True),
            )

        if self.backend == "tutel":
            self.moe_layer = self._build_tutel(in_features, hidden_features, moe_cfg, act_layer)
        elif self.backend == "native":
            self.moe_layer = self._build_native(in_features, hidden_features, moe_cfg, act_layer)
        elif self.backend == "megablocks":
            self.moe_layer, self.mb_args = self._build_megablocks(
                in_features, hidden_features, moe_cfg, act_layer
            )
        else:
            raise ValueError(f"Unknown MoE backend: {self.backend!r}")

        # Custom weight init must not touch expert/gate parameters — both
        # backends do their own init. pvt.py checks this attribute.
        self.moe_layer.is_moe_expert_container = True

    # -- builders -----------------------------------------------------------

    @staticmethod
    def _build_tutel(dim: int, hidden: int, moe_cfg: dict, act_layer):
        from tutel import moe as tutel_moe  # lazy: only needed for this backend

        return tutel_moe.moe_layer(
            gate_type={
                "type": "top",
                "k": moe_cfg["top_k"],
                "capacity_factor": moe_cfg["capacity_factor"],
                "gate_noise": moe_cfg["gate_noise"],
            },
            model_dim=dim,
            experts={
                "count_per_node": moe_cfg["num_experts"],
                "type": "ffn",
                "hidden_size_per_expert": hidden,
                # LOAD-BEARING: activation_fn must ALWAYS be passed explicitly.
                # tutel/experts/ffn.py uses `F.relu` in its default branch but
                # never imports torch.nn.functional, so omitting this raises
                #     NameError: name 'F' is not defined
                # from inside Tutel's own forward. Present on main as of
                # 9a70a681b7673ee23135aa54446ac2a03cf0a61d. Supplying it here
                # means that code path never runs.
                "activation_fn": act_layer(),
            },
            # Single-node training: exclude expert params from allreduce.
            scan_expert_func=lambda name, param: setattr(param, "skip_allreduce", True),
            result_func=lambda output: (output, output.l_aux),
        )

    @staticmethod
    def _build_native(dim: int, hidden: int, moe_cfg: dict, act_layer):
        """Pure-PyTorch fallback — same contract, no Tutel, no NCCL."""
        # [v12] inlined above: was `from pvt_moe.models.moe_native import ...`

        return NativeMoEFFN(
            model_dim=dim,
            hidden_size_per_expert=hidden,
            num_experts=moe_cfg["num_experts"],
            top_k=moe_cfg["top_k"],
            capacity_factor=moe_cfg["capacity_factor"],
            activation_fn=act_layer(),
            gate_noise=moe_cfg.get("gate_noise", 0.0),
        )

    @staticmethod
    def _build_megablocks(dim: int, hidden: int, moe_cfg: dict, act_layer):
        from megablocks.layers.arguments import Arguments  # lazy
        from megablocks.layers.dmoe import dMoE

        args = Arguments(
            hidden_size=dim,
            ffn_hidden_size=hidden,
            moe_num_experts=moe_cfg["num_experts"],
            moe_top_k=moe_cfg["top_k"],
            # Raw loss; the training loop applies loss.aux_weight itself.
            moe_loss_weight=1.0,
            # Grouped expert MLPs create no bias parameters; say so honestly.
            bias=False,
            activation_fn=act_layer(),
            mlp_impl="grouped",
            # One registry entry per layer — matches the clear/collect pattern
            # in forward() below.
            num_layers=1,
            # Arguments defaults to fp16=True (Megatron heritage) — wrong for
            # this pipeline. bf16 params match bf16-mixed autocast activations
            # (grouped_gemm kernels do not run fp32, so fp16=False alone would
            # still mismatch). MoE forwards must run under bf16 autocast.
            fp16=False,
            bf16=True,
            # Arguments' device default_factory CALLS torch.cuda.current_device()
            # at construction — crashes CPU-only boxes and silently puts expert
            # params on cuda:0 while the rest of the model is on CPU. Build on
            # CPU like every other module; Lightning/.to(device) moves it.
            device=torch.device("cpu"),
        )
        return dMoE(args), args

    # -- upcycling ------------------------------------------------------------

    @torch.no_grad()
    def load_from_dense_ffn(self, dense_state_dict: dict) -> int:
        """Load a pretrained dense FFN into the SHARED expert. Backend-agnostic.

        The routed experts are deliberately left at their zero-fc2 init: they
        must emit exactly zero so the block reproduces the dense checkpoint at
        step 0 (``upcycle_init: "routed_zero"``). Cloning the dense weights
        into them instead is the ``"shared_zero"`` scheme, which is NOT
        function-preserving at top_k=1 — Tutel normalizes combine weights only
        when top_k > 1, so the routed branch would be scaled by an untrained
        softmax score. See docs/HPARAMS.md section 3.

        ``dense_state_dict`` is a plain PVT v2 ``Mlp`` state dict
        (``fc1.weight``, ``fc1.bias``, ``fc2.weight``, ``fc2.bias``, and
        ``dwconv.dwconv.*`` when the block keeps its conv). Returns the number
        of tensors loaded; raises on any shape mismatch rather than quietly
        leaving a layer at random init.
        """
        if self.shared_expert is None:
            raise RuntimeError(
                "load_from_dense_ffn needs a shared expert to load into "
                "(moe.shared_expert is False). Without one there is no branch "
                "that can carry the pretrained FFN."
            )
        target = self.shared_expert.state_dict()
        loaded, skipped = 0, []
        for key, value in dense_state_dict.items():
            name = key.split("mlp.")[-1] if "mlp." in key else key
            if name not in target:
                skipped.append(name)
                continue
            if tuple(target[name].shape) != tuple(value.shape):
                raise ValueError(
                    f"shape mismatch loading dense FFN into the shared expert: "
                    f"{name} is {tuple(value.shape)} in the checkpoint but "
                    f"{tuple(target[name].shape)} in the model."
                )
            target[name].copy_(value)
            loaded += 1

        missing = sorted(set(target) - {k.split("mlp.")[-1] for k in dense_state_dict})
        if missing:
            raise ValueError(
                f"dense FFN state dict is missing {missing}; the shared expert "
                "would be left partly at random init. Pass the full Mlp "
                "state_dict, or rebuild with moe_block_dwconv matching the source."
            )
        if skipped:
            conv_only = all(k.startswith("dwconv.") for k in skipped)
            note = ("this block has no DWConv (moe_block_dwconv=False)"
                    if conv_only and self.shared_expert.dwconv is None
                    else "NO DESTINATION — check the source")
            print(f"[load_from_dense_ffn] skipped {skipped}: {note}")
        return loaded

    # -- forward ------------------------------------------------------------

    def forward(self, x: torch.Tensor, H: int, W: int):
        B, N, C = x.shape
        x_flat = x.reshape(B * N, C).contiguous()

        if self.backend in ("tutel", "native"):
            out, aux = self.moe_layer(x_flat)
        else:  # megablocks
            from megablocks.layers.moe import (  # lazy
                batched_load_balancing_loss,
                clear_load_balancing_loss,
            )

            clear_load_balancing_loss()
            out = self.moe_layer(x_flat)
            if self.training:
                # Registry is only populated in training mode; collecting it
                # in eval crashes (the archive attempt's failure mode).
                aux = batched_load_balancing_loss(self.mb_args)
            else:
                aux = torch.zeros((), device=x.device, dtype=x.dtype)
            clear_load_balancing_loss()

        out = out.reshape(B, N, C)
        if self.shared_expert is not None:
            # The shared expert sees the ORIGINAL (B, N, C) input — it needs
            # H/W for its DWConv, which the flattened routed path cannot use.
            out = out + self.shared_expert(x, H, W)
        return self.drop(out), aux


def force_tutel_gates_train(model: nn.Module) -> None:
    """Put every Tutel gate back into train mode (LOAD-BEARING).

    Tutel gate modules revert themselves to eval mode after a Lightning
    validation pass, silently disabling ``gate_noise`` and with it the
    exploration that keeps the experts balanced. Both training modules call
    this from ``train()`` and ``on_train_epoch_start``; do not remove.
    """
    for module in model.modules():
        if hasattr(module, "moe_layer"):
            module.moe_layer.train()
            for gate in getattr(module.moe_layer, "gates", []):
                if hasattr(gate, "train"):
                    gate.train()
                gate.training = True
        if hasattr(module, "gate_noise"):
            module.training = True


## Δ since v10 — the backbone

<pre>
<del>"moe_last_n_stages": 1, "rope_last_n_stages": 1     # only "last N stages"</del>
"moe_placement": [[], [], [], [-1]]   # per-stage BLOCK lists; -1 = last block
"rope_placement": [[], [], [], [-1]]  # of the stage for any variant's depth
</pre>

- MoE blocks return `(x, aux)`; the model returns
  `(logits, mean aux over MoE blocks | None)` — the tuple contract every
  caller (and the tests) rely on;
- optional per-stage **gradient checkpointing** (`grad_checkpointing`), with
  `use_reentrant=False` so tuple-returning MoE blocks work;
- `freeze_stages` + re-freeze after `.train()` (v10 froze by hand once);
- an SSL hook (`stage1_token_mask` / `mask_token`) used by SimMIM — inert in
  supervised runs;
- init unchanged (trunc-normal 0.02 linears, fan-out convs) and verified
  equal to timm's `pvt_v2_b2` — forward and every gradient ≤ 3e-6 with shared
  weights, so <b>the backbone is not a suspect for a run that will not learn</b>;
- `pretrained.py` maps the official OpenGVLab HF checkpoints onto these names
  and refuses another variant's weights instead of part-loading them.


In [ ]:
# ═══ pvt_moe/models/pvt.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""PVT v2 backbone with per-block MoE and RoPE placement.

A 4-stage pyramid vision transformer (Wang et al., "PVT v2: Improved
Baselines with Pyramid Vision Transformer") extended with:

- SRA multi-head attention (``pvt_moe.models.attention.SRAttention``)
- per-block Mixture-of-Experts FFN (``pvt_moe.models.ffn.MoEMlp``)
- per-block 2D RoPE, mixed (learnable per-head frequencies, default) or
  axial (``pvt_moe.models.rope``)
- a LayerNorm/RMSNorm toggle (``pvt_moe.models.norms``)

The overlapping patch-embedding stems are fixed at the official PVT v2
geometry — 7x7/stride-4 for stage 1 and 3x3/stride-2 for stages 2-4 — and are
deliberately not configurable (the old notebooks carried a misleading
``patch_size`` config knob that was silently ignored).

Auxiliary-loss contract (the "fixed aux" semantics from the v9 lineage):
every MoE block returns its own load-balancing loss; ``forward_features``
averages them over the number of MoE blocks; the model returns
``(logits, aux)`` where ``aux`` is None when no MoE block ran. Clamping,
weighting, and the NaN guard belong to the training loop, not the model.
"""

from __future__ import annotations

import math

import torch
import torch.nn as nn
from torch.nn.init import trunc_normal_

# [v12] inlined above: was `from pvt_moe.models.attention import ...`
# [v12] inlined above: was `from pvt_moe.models.ffn import ...`
# [v12] inlined above: was `from pvt_moe.models.norms import ...`


def _to_2tuple(x):
    return x if isinstance(x, (tuple, list)) else (x, x)


class DropPath(nn.Module):
    """Stochastic depth per sample (residual-branch dropout)."""

    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        mask_shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = x.new_empty(mask_shape).bernoulli_(keep_prob)
        return x * mask / keep_prob

    def extra_repr(self) -> str:
        return f"drop_prob={self.drop_prob:.3f}"


class OverlapPatchEmbed(nn.Module):
    """Overlapping conv patch embedding (conv -> flatten -> norm)."""

    def __init__(self, patch_size, stride, in_chans, embed_dim, norm_layer=nn.LayerNorm):
        super().__init__()
        patch_size = _to_2tuple(patch_size)
        if max(patch_size) <= stride:
            raise ValueError("patch_size must exceed stride for overlapping embedding")
        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=patch_size,
            stride=stride,
            padding=(patch_size[0] // 2, patch_size[1] // 2),
        )
        self.norm = norm_layer(embed_dim)

    def forward(self, x: torch.Tensor):
        x = self.proj(x)
        _, _, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x, H, W


class Block(nn.Module):
    """Pre-norm transformer block: SRA attention + (dense | MoE) FFN.

    ``forward`` returns a tensor for dense blocks and ``(tensor, aux_loss)``
    for MoE blocks.
    """

    def __init__(
        self,
        dim: int,
        num_heads: int,
        mlp_ratio: float,
        qkv_bias: bool,
        drop: float,
        attn_drop: float,
        drop_path: float,
        norm_layer,
        sr_ratio: int,
        linear_attention: bool,
        act_layer=nn.GELU,
        use_moe: bool = False,
        moe_cfg: dict | None = None,
        use_rope: bool = False,
        rope_theta: float = 10.0,
        rope_mode: str = "mixed",
        dense_dwconv: bool = True,
    ):
        super().__init__()
        self.use_moe = use_moe
        self.norm1 = norm_layer(dim)
        self.attn = SRAttention(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
            sr_ratio=sr_ratio,
            linear_attention=linear_attention,
            norm_layer=norm_layer,
            use_rope=use_rope,
            rope_theta=rope_theta,
            rope_mode=rope_mode,
        )
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.norm2 = norm_layer(dim)
        hidden = int(dim * mlp_ratio)
        if use_moe:
            self.mlp = MoEMlp(dim, hidden, moe_cfg=moe_cfg, act_layer=act_layer, drop=drop)
        else:
            self.mlp = Mlp(
                dim, hidden, act_layer=act_layer, drop=drop,
                linear_attention=linear_attention, use_dwconv=dense_dwconv,
            )

    def forward(self, x: torch.Tensor, H: int, W: int):
        x = x + self.drop_path(self.attn(self.norm1(x), H, W))
        if self.use_moe:
            mlp_out, aux = self.mlp(self.norm2(x), H, W)
            return x + self.drop_path(mlp_out), aux
        return x + self.drop_path(self.mlp(self.norm2(x), H, W))


class PyramidVisionTransformerV2(nn.Module):
    """PVT v2 with per-block MoE/RoPE placement.

    ``moe_placement`` / ``rope_placement`` are lists (one entry per stage) of
    block indices, e.g. ``[[], [], [], [0, 1]]`` enables both blocks of
    stage 4. Use ``pvt_moe.config.resolve_placement`` to build them.
    """

    def __init__(
        self,
        in_chans: int = 3,
        num_classes: int = 1000,
        embed_dims=(64, 128, 320, 512),
        num_heads=(1, 2, 5, 8),
        mlp_ratios=(8, 8, 4, 4),
        depths=(2, 2, 2, 2),
        sr_ratios=(8, 4, 2, 1),
        qkv_bias: bool = True,
        drop_rate: float = 0.0,
        attn_drop_rate: float = 0.0,
        drop_path_rate: float = 0.0,
        linear_attention: bool = False,
        norm_layer=nn.LayerNorm,
        norm_layer_last_stage=None,
        moe_placement=None,
        rope_placement=None,
        moe_cfg: dict | None = None,
        rope_theta: float = 10.0,
        rope_mode: str = "mixed",
        act_layer=nn.GELU,
        dense_dwconv: bool = True,
        grad_checkpointing=(),
    ):
        super().__init__()
        # 1-based stage numbers to recompute in the backward pass.
        self.grad_checkpointing = set(grad_checkpointing or ())
        self.num_classes = num_classes
        self.depths = list(depths)
        self.num_stages = len(depths)
        self.embed_dims = list(embed_dims)
        moe_placement = moe_placement or [[] for _ in depths]
        rope_placement = rope_placement or [[] for _ in depths]
        self.moe_placement = [list(b) for b in moe_placement]
        self.rope_placement = [list(b) for b in rope_placement]
        norm_last = norm_layer_last_stage or norm_layer

        # Stochastic depth: linear ramp over the full block sequence.
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        cur = 0

        for i in range(self.num_stages):
            stage_norm = norm_last if i == self.num_stages - 1 else norm_layer
            patch_embed = OverlapPatchEmbed(
                patch_size=7 if i == 0 else 3,
                stride=4 if i == 0 else 2,
                in_chans=in_chans if i == 0 else embed_dims[i - 1],
                embed_dim=embed_dims[i],
                norm_layer=stage_norm,
            )
            blocks = nn.ModuleList(
                [
                    Block(
                        dim=embed_dims[i],
                        num_heads=num_heads[i],
                        mlp_ratio=mlp_ratios[i],
                        qkv_bias=qkv_bias,
                        drop=drop_rate,
                        attn_drop=attn_drop_rate,
                        drop_path=dpr[cur + j],
                        norm_layer=stage_norm,
                        sr_ratio=sr_ratios[i],
                        linear_attention=linear_attention,
                        act_layer=act_layer,
                        use_moe=(j in self.moe_placement[i]),
                        moe_cfg=moe_cfg,
                        use_rope=(j in self.rope_placement[i]),
                        rope_theta=rope_theta,
                        rope_mode=rope_mode,
                        dense_dwconv=dense_dwconv,
                    )
                    for j in range(depths[i])
                ]
            )
            norm = stage_norm(embed_dims[i])
            cur += depths[i]
            # PVT-official attribute naming — the HF pretrained remap
            # (pvt_moe.models.pretrained) depends on these names.
            setattr(self, f"patch_embed{i + 1}", patch_embed)
            setattr(self, f"block{i + 1}", blocks)
            setattr(self, f"norm{i + 1}", norm)

        self.head = nn.Linear(embed_dims[-1], num_classes) if num_classes > 0 else nn.Identity()
        self._init_all_weights()

    # -- init ----------------------------------------------------------------

    def _init_all_weights(self):
        """Init every module ONCE, skipping MoE expert/gate internals."""
        skip = set()
        for m in self.modules():
            if getattr(m, "is_moe_expert_container", False):
                skip.update(id(s) for s in m.modules())
        for m in self.modules():
            if id(m) not in skip:
                self._init_weights(m)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, RMSNorm)):
            if getattr(m, "bias", None) is not None:
                nn.init.constant_(m.bias, 0)
            if getattr(m, "weight", None) is not None:
                nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels // m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    # -- utilities -------------------------------------------------------------

    @torch.jit.ignore
    def no_weight_decay(self) -> set:
        """Parameter names to exclude from weight decay, beyond the ndim<=1
        rule (norms/biases) the optimizer factories apply themselves.

        The learnable RoPE-Mixed frequencies: decaying them pulls every
        frequency toward zero, i.e. toward position blindness (rope-vit lists
        ``freqs`` under ``no_weight_decay`` for the same reason). Both
        ``LitClassifier`` and ``LitJEPA`` consult this.
        """
        return {n for n, _ in self.named_parameters() if n.endswith("rope.freqs")}

    def get_classifier(self):
        return self.head

    def reset_classifier(self, num_classes: int):
        self.num_classes = num_classes
        self.head = (
            nn.Linear(self.embed_dims[-1], num_classes) if num_classes > 0 else nn.Identity()
        )
        if num_classes > 0:
            self._init_weights(self.head)  # keep the trunc_normal(0.02) contract

    def freeze_stages(self, num_frozen_stages: int):
        """Freeze (eval + requires_grad=False) the first N stages.

        RoPE-Mixed frequencies inside a frozen stage stay trainable: no
        pretrained checkpoint carries them, so freezing would pin a random
        positional encoding that the stage was never trained with.
        """
        for i in range(num_frozen_stages):
            for attr in (f"patch_embed{i + 1}", f"block{i + 1}", f"norm{i + 1}"):
                module = getattr(self, attr)
                module.eval()
                for name, p in module.named_parameters():
                    p.requires_grad = name.endswith("rope.freqs")

    # -- forward ---------------------------------------------------------------

    def forward_features(
        self,
        x: torch.Tensor,
        return_tokens: bool = False,
        stage1_token_mask: torch.Tensor | None = None,
        mask_token: torch.Tensor | None = None,
    ):
        """Run the 4-stage backbone.

        Returns ``(features, aux)`` where ``features`` is the mean-pooled
        embedding (B, C) — or the final token sequence (B, N, C) when
        ``return_tokens`` — and ``aux`` is the mean MoE load-balancing loss
        over MoE blocks (None when no MoE block ran).

        SSL hooks (used by pvt_moe.ssl): ``stage1_token_mask`` is a (B, N1)
        bool tensor over the stage-1 token grid; masked tokens are replaced by
        the learnable ``mask_token`` (C1,) right after the first patch
        embedding (SimMIM-style masking for hierarchical backbones).
        """
        B = x.shape[0]
        aux_total = 0.0
        num_moe_blocks = 0

        for i in range(self.num_stages):
            patch_embed = getattr(self, f"patch_embed{i + 1}")
            blocks = getattr(self, f"block{i + 1}")
            norm = getattr(self, f"norm{i + 1}")

            x, H, W = patch_embed(x)
            if i == 0 and stage1_token_mask is not None:
                if mask_token is None:
                    raise ValueError("stage1_token_mask requires mask_token")
                x = torch.where(
                    stage1_token_mask[..., None], mask_token.to(x.dtype).expand_as(x), x
                )

            checkpointed = (
                self.training
                and torch.is_grad_enabled()
                and (i + 1) in self.grad_checkpointing
            )
            for blk in blocks:
                if checkpointed:
                    # use_reentrant=False keeps this compatible with blocks that
                    # return tuples (MoE blocks return (x, aux)).
                    out = torch.utils.checkpoint.checkpoint(
                        blk, x, H, W, use_reentrant=False)
                else:
                    out = blk(x, H, W)
                if isinstance(out, tuple):
                    x, blk_aux = out
                    aux_total = aux_total + blk_aux
                    num_moe_blocks += 1
                else:
                    x = out

            x = norm(x)
            if i != self.num_stages - 1:
                x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()

        aux = aux_total / num_moe_blocks if num_moe_blocks > 0 else None
        feats = x if return_tokens else x.mean(dim=1)
        return feats, aux

    def forward(self, x: torch.Tensor):
        """Return ``(logits, aux)``; ``aux`` is None when no MoE block ran."""
        feats, aux = self.forward_features(x)
        return self.head(feats), aux


def build_model(cfg: dict) -> PyramidVisionTransformerV2:
    """Construct the backbone from a validated config dict.

    Warm starting (HF weights / SSL checkpoints / expert seeding) is handled
    separately by ``pvt_moe.models.pretrained`` — this builds architecture
    only.
    """
    m = cfg["model"]
    abl = m["ablation"]
    norm_main, norm_last = build_norm_layers(
        m["norm_type"], m["norm_eps"], m["stage4_keeps_layernorm"]
    )
    return PyramidVisionTransformerV2(
        in_chans=m["in_chans"],
        num_classes=cfg["dataset"]["num_classes"],
        embed_dims=m["embed_dims"],
        num_heads=m["num_heads"],
        mlp_ratios=m["mlp_ratios"],
        depths=m["depths"],
        sr_ratios=m["sr_ratios"],
        qkv_bias=m["qkv_bias"],
        drop_rate=m["drop_rate"],
        attn_drop_rate=m["attn_drop_rate"],
        drop_path_rate=m["drop_path_rate"],
        linear_attention=m["linear_attention"],
        norm_layer=norm_main,
        norm_layer_last_stage=norm_last,
        moe_placement=abl["moe_placement"] if abl["use_moe"] else None,
        rope_placement=abl["rope_placement"] if abl["use_rope"] else None,
        moe_cfg=m["moe"],
        rope_theta=abl["rope_theta"],
        rope_mode=abl["rope_mode"],
        dense_dwconv=m.get("dense_dwconv", True),
        grad_checkpointing=m.get("grad_checkpointing", ()),
    )


In [ ]:
# ═══ pvt_moe/models/pretrained.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""Warm-start utilities: HF weight remapping, expert seeding, SSL init.

Three entry points:

- ``load_hf_pretrained(model, hf_id, ...)`` — load ``OpenGVLab/pvt_v2_b*``
  (or any HF PVT v2) into our backbone: remaps HF key names, fuses the HF
  key/value projections into our ``attn.kv`` layout, skips the dense FFN of
  MoE blocks, and optionally seeds MoE experts from those skipped dense
  weights (sparse upcycling, Komatsuzaki et al. 2023).
- ``load_backbone_checkpoint(model, path, ...)`` — load a backbone
  ``state_dict`` saved by this package (e.g. a JEPA-pretrained encoder).
- ``seed_moe_experts_from_dense(...)`` — copy dense fc1/fc2 into every
  expert. Layout-aware for both Tutel and MegaBlocks; refuses to guess.
- ``seed_shared_expert_from_dense(...)`` — copy the dense FFN (fc1/fc2 AND
  the DWConv) into ``MoEMlp.shared_expert``, which is a plain PVT v2 ``Mlp``
  and therefore takes the weights verbatim.
- ``zero_routed_expert_output(...)`` — zero every routed expert's fc2 so a
  shared-expert block starts out computing EXACTLY the pretrained dense FFN.

Norm-type interop: when the target model uses RMSNorm, LayerNorm ``.bias``
keys from the source simply have no destination parameter and are dropped
(reported in the load stats). LN gamma transfers to RMSNorm weight directly.

Every loader returns a stats dict — print it and READ it. A silent
0-weights-loaded bug cost this project a full failed training run (8.9%
accuracy) before the remap patterns were fixed.
"""

from __future__ import annotations

import re

import torch
import torch.nn as nn


# ---------------------------------------------------------------------------
# HF key remapping
# ---------------------------------------------------------------------------

def _remap_hf_key(hf_key: str):
    """Map a HF transformers PvtV2 key to this package's naming (or None)."""
    # layers.{N}.patch_embedding.* -> patch_embed{N+1}.*
    m = re.match(r"pvt_v2\.encoder\.layers\.(\d+)\.patch_embedding\.(.*)", hf_key)
    if m:
        n, rest = int(m.group(1)), m.group(2)
        rest = re.sub(r"^layer_norm\.", "norm.", rest)
        rest = re.sub(r"^projection\.", "proj.", rest)
        return f"patch_embed{n + 1}.{rest}"

    # layers.{N}.layer_norm.* -> norm{N+1}.*  (stage output norm)
    m = re.match(r"pvt_v2\.encoder\.layers\.(\d+)\.layer_norm\.(.*)", hf_key)
    if m:
        return f"norm{int(m.group(1)) + 1}.{m.group(2)}"

    # layers.{N}.blocks.{M}.* -> block{N+1}.{M}.*
    m = re.match(r"pvt_v2\.encoder\.layers\.(\d+)\.blocks\.(\d+)\.(.*)", hf_key)
    if m:
        n, blk, rest = int(m.group(1)), int(m.group(2)), m.group(3)
        if rest.startswith(("attention.key.", "attention.value.")):
            return None  # fused into attn.kv by the loader, not remapped 1:1
        rest = re.sub(r"^layer_norm_1\.", "norm1.", rest)
        rest = re.sub(r"^layer_norm_2\.", "norm2.", rest)
        rest = re.sub(r"^attention\.query\.", "attn.q.", rest)
        rest = re.sub(r"^attention\.proj\.", "attn.proj.", rest)
        rest = re.sub(r"^attention\.spatial_reduction\.", "attn.sr.", rest)
        rest = re.sub(r"^attention\.layer_norm\.", "attn.norm.", rest)
        # attention.key / attention.value are fused separately (see below).
        rest = re.sub(r"^mlp\.dense1\.", "mlp.fc1.", rest)
        rest = re.sub(r"^mlp\.dense2\.", "mlp.fc2.", rest)
        rest = re.sub(r"^mlp\.dwconv\.dwconv\.", "mlp.dwconv.dwconv.", rest)
        return f"block{n + 1}.{blk}.{rest}"

    m = re.match(r"classifier\.(.*)", hf_key)
    if m:
        return f"head.{m.group(1)}"
    return None


_HF_KV_RE = re.compile(
    r"pvt_v2\.encoder\.layers\.(\d+)\.blocks\.(\d+)\.attention\.(key|value)\.(.*)"
)


def _assert_hf_architecture_matches(model, hf_config, hf_model_id: str) -> None:
    """Refuse a checkpoint whose depths/widths differ from the built model.

    ``load_state_dict(strict=False)`` would otherwise load the blocks that
    exist in both and silently leave the rest at random init — exactly the
    "B2 depths with B1 weights" run this guard exists to make impossible.
    Compared: ``depths`` and ``hidden_sizes`` (the HF PvtV2Config names).
    """
    want = {"depths": [int(d) for d in getattr(model, "depths", [])],
            "embed_dims": [int(d) for d in getattr(model, "embed_dims", [])]}
    have = {"depths": [int(d) for d in getattr(hf_config, "depths", [])],
            "embed_dims": [int(d) for d in getattr(hf_config, "hidden_sizes", [])]}
    bad = [k for k in want if want[k] and have[k] and want[k] != have[k]]
    if bad:
        detail = "; ".join(f"{k}: model {want[k]} vs checkpoint {have[k]}" for k in bad)
        raise ValueError(
            f"pretrained checkpoint {hf_model_id!r} does not fit this model "
            f"({detail}). Pick the --variant whose architecture matches the "
            f"checkpoint, or the checkpoint that matches the variant."
        )


def load_hf_pretrained(
    model: nn.Module,
    hf_model_id: str = "OpenGVLab/pvt_v2_b1",
    seed_moe_experts: bool = True,
    upcycle_init: str = "none",
    verbose: bool = True,
) -> dict:
    """Load HF PVT v2 weights into the backbone. Returns a stats dict.

    MoE blocks (from ``model.moe_placement``) skip their dense ``mlp.*``
    weights; when ``seed_moe_experts`` those weights seed the experts instead.
    A block with a shared expert additionally gets the dense FFN loaded into
    that shared branch verbatim. ``upcycle_init`` then says which branch starts
    at zero so the block does not emit ~2x the dense layer at step 0:

    - ``"routed_zero"`` zeros the routed experts' fc2 — the shared branch
      carries the pretrained FFN, exact at any top_k;
    - ``"shared_zero"`` zeros the shared expert's fc2 instead — the routed
      experts carry it (the spec's Sparse-Upcycling-style init);
    - ``"none"`` zeros nothing.
    """
    from transformers import AutoModelForImageClassification  # lazy

    hf_model = AutoModelForImageClassification.from_pretrained(hf_model_id)
    _assert_hf_architecture_matches(model, hf_model.config, hf_model_id)
    hf_state = hf_model.state_dict()
    model_state = model.state_dict()

    moe_placement = getattr(model, "moe_placement", [[] for _ in range(4)])
    moe_block_prefixes = {
        f"block{i + 1}.{j}." for i, blocks in enumerate(moe_placement) for j in blocks
    }

    def _is_moe_mlp(custom_key: str) -> bool:
        return any(
            custom_key.startswith(p) and custom_key[len(p):].startswith("mlp.")
            for p in moe_block_prefixes
        )

    filtered, kv_pending, dense_mlp_for_seeding = {}, {}, {}
    stats = {
        "loaded": 0, "kv_fused": 0, "kv_skipped": 0, "skipped_moe_mlp": 0,
        "skipped_shape": 0, "dropped_no_target": 0, "unmapped": [],
        "seeded_moe_blocks": 0, "seeded_shared_experts": 0,
        "zeroed_routed_fc2": 0, "zeroed_shared_fc2": 0,
    }

    for hf_key, value in hf_state.items():
        kv_match = _HF_KV_RE.match(hf_key)
        if kv_match:
            n, blk, kind, suffix = kv_match.groups()
            kv_pending.setdefault((f"block{int(n) + 1}.{blk}", suffix), {})[kind] = value
            continue

        custom_key = _remap_hf_key(hf_key)
        if custom_key is None:
            stats["unmapped"].append(hf_key)
            continue
        if _is_moe_mlp(custom_key):
            stats["skipped_moe_mlp"] += 1
            dense_mlp_for_seeding[custom_key] = value
            continue
        if custom_key not in model_state:
            stats["dropped_no_target"] += 1  # e.g. LN bias -> RMSNorm target
            continue
        if model_state[custom_key].shape != value.shape:
            stats["skipped_shape"] += 1
            continue
        filtered[custom_key] = value

    # Fuse HF's separate key/value projections into our attn.kv layout.
    for (block_prefix, suffix), pair in kv_pending.items():
        if "key" in pair and "value" in pair:
            fused = torch.cat([pair["key"], pair["value"]], dim=0)
            custom_key = f"{block_prefix}.attn.kv.{suffix}"
            if custom_key in model_state and model_state[custom_key].shape == fused.shape:
                filtered[custom_key] = fused
                stats["kv_fused"] += 1
            else:
                stats["kv_skipped"] += 1  # fused kv shape mismatch (a non-official head layout)

    missing, unexpected = model.load_state_dict(filtered, strict=False)
    stats["loaded"] = len(filtered)
    stats["missing"] = list(missing)
    stats["unexpected"] = list(unexpected)

    if seed_moe_experts and moe_block_prefixes:
        upcycle_moe_blocks(model, dense_mlp_for_seeding, moe_block_prefixes, upcycle_init, stats)

    if verbose:
        print(
            f"[HF pretrained] loaded={stats['loaded']} kv_fused={stats['kv_fused']} "
            f"kv_skipped={stats['kv_skipped']} moe_mlp_skipped={stats['skipped_moe_mlp']} "
            f"shape_skipped={stats['skipped_shape']} no_target={stats['dropped_no_target']} "
            f"seeded_moe_blocks={stats['seeded_moe_blocks']} "
            f"seeded_shared={stats['seeded_shared_experts']} "
            f"zeroed_routed_fc2={stats['zeroed_routed_fc2']} "
            f"zeroed_shared_fc2={stats['zeroed_shared_fc2']}"
        )
        if stats["loaded"] < 50:
            print("  !! Very few weights loaded — remap patterns are probably stale. "
                  "Inspect stats['unmapped'] and stats['missing'].")
        if stats["unmapped"]:
            print(f"  unmapped HF keys ({len(stats['unmapped'])}): {stats['unmapped'][:5]} ...")
        if stats["missing"]:
            print(f"  missing model keys ({len(stats['missing'])}, expected for MoE experts, "
                  f"and RoPE-Mixed freqs, which stay at init): "
                  f"{stats['missing'][:6]} ...")
    return stats


# ---------------------------------------------------------------------------
# Expert seeding (sparse upcycling)
# ---------------------------------------------------------------------------

def moe_block_prefixes_of(model) -> set:
    """``{"block4.1.", ...}`` for every block the model converted to MoE."""
    placement = getattr(model, "moe_placement", None) or []
    return {f"block{i + 1}.{j}." for i, blocks in enumerate(placement) for j in blocks}


def upcycle_moe_blocks(model, dense_mlp: dict, moe_block_prefixes, upcycle_init: str,
                       stats: dict | None = None) -> dict:
    """Seed every MoE'd block from the dense FFN it replaced — the ONE
    upcycling routine, shared by the HF and the ssl_init warm starts.

    ``dense_mlp`` maps full dense-model keys (``block4.1.mlp.fc1.weight``,
    ``...mlp.dwconv.dwconv.weight``, ...) to tensors. Per block: the routed
    experts are replicated from fc1/fc2, the shared expert (when present)
    takes the FFN verbatim (DWConv included), and ``upcycle_init`` picks the
    branch that starts at zero so the block reproduces the dense FFN exactly
    at step 0 (``routed_zero``: the shared branch carries it, exact at any
    top_k). Returns / updates the stats counters.
    """
    stats = stats if stats is not None else {}
    for key in ("seeded_moe_blocks", "seeded_shared_experts",
                "zeroed_routed_fc2", "zeroed_shared_fc2"):
        stats.setdefault(key, 0)
    for prefix in sorted(moe_block_prefixes):
        fc1_w = dense_mlp.get(f"{prefix}mlp.fc1.weight")
        fc1_b = dense_mlp.get(f"{prefix}mlp.fc1.bias")
        fc2_w = dense_mlp.get(f"{prefix}mlp.fc2.weight")
        fc2_b = dense_mlp.get(f"{prefix}mlp.fc2.bias")
        if fc1_w is None or fc2_w is None:
            continue
        stage = int(prefix.split(".")[0].removeprefix("block")) - 1
        blk = int(prefix.split(".")[1])
        moe_mlp = getattr(model, f"block{stage + 1}")[blk].mlp
        seed_moe_experts_from_dense(moe_mlp, fc1_w, fc1_b, fc2_w, fc2_b)
        stats["seeded_moe_blocks"] += 1

        # Shared expert (when enabled): takes the dense FFN verbatim,
        # DWConv included, so the pretrained function is kept exactly
        # rather than replicated across experts.
        if getattr(moe_mlp, "shared_expert", None) is not None:
            seed_shared_expert_from_dense(moe_mlp, dense_mlp, prefix)
            stats["seeded_shared_experts"] += 1
            if upcycle_init == "routed_zero":
                stats["zeroed_routed_fc2"] += zero_routed_expert_output(moe_mlp)
            elif upcycle_init == "shared_zero":
                stats["zeroed_shared_fc2"] += zero_shared_expert_output(moe_mlp)
    return stats


@torch.no_grad()
def seed_moe_experts_from_dense(moe_mlp, fc1_w, fc1_b, fc2_w, fc2_b) -> int:
    """Copy dense FFN weights into every expert of a ``MoEMlp``.

    Dense shapes: ``fc1_w (hidden, dim)``, ``fc1_b (hidden,)``,
    ``fc2_w (dim, hidden)``, ``fc2_b (dim,)``.

    Expert layouts handled explicitly (no shape guessing — the archived
    MegaBlocks attempt silently seeded nothing by guessing):

    - Tutel FusedExpertsNetwork: ``batched_fc1_w (E, hidden, dim)``,
      ``batched_fc2_w (E, hidden, dim)`` (stored TRANSPOSED — it equals
      ``fc2_w.T`` per expert), ``batched_fc1_bias (E, hidden)``-ish,
      ``batched_fc2_bias (..., dim)``.
    - MegaBlocks GroupedMLP: ``w1 (E*hidden, dim)``, ``w2 (E*hidden, dim)``
      (``w2`` rows are ``fc2_w.T`` per expert). No biases (bias=False).

    All experts start identical; the router's noise/jitter breaks symmetry.
    Returns the number of parameters seeded; raises if the layout was not
    recognized.
    """
    E = moe_mlp.num_experts
    hidden, dim = fc1_w.shape
    if fc2_w.shape != (dim, hidden):
        raise ValueError(f"fc2_w shape {tuple(fc2_w.shape)} != ({dim}, {hidden})")
    fc2_w_t = fc2_w.t().contiguous()  # (hidden, dim)

    seeded = 0
    unrecognized = []
    for name, param in moe_mlp.moe_layer.named_parameters():
        lname = name.lower()
        if "gate" in lname or re.search(r"(^|\.)wg", lname) or "router" in lname:
            continue  # never touch the router
        shape = tuple(param.shape)

        if ("fc1" in lname or re.search(r"(^|\.)w1$", lname)) and "bias" not in lname:
            if shape == (E, hidden, dim):                     # tutel batched
                param.copy_(fc1_w.unsqueeze(0).expand_as(param))
            elif shape == (E * hidden, dim):                  # megablocks flattened
                param.copy_(fc1_w.repeat(E, 1))
            elif shape == (E, dim, hidden):                   # transposed variant
                param.copy_(fc1_w.t().unsqueeze(0).expand_as(param))
            else:
                unrecognized.append((name, shape))
                continue
            seeded += 1

        elif ("fc2" in lname or re.search(r"(^|\.)w2$", lname)) and "bias" not in lname:
            if shape == (E, hidden, dim):                     # tutel: stores fc2.T
                param.copy_(fc2_w_t.unsqueeze(0).expand_as(param))
            elif shape == (E * hidden, dim):                  # megablocks: rows are fc2.T
                param.copy_(fc2_w_t.repeat(E, 1))
            elif shape == (E, dim, hidden):
                param.copy_(fc2_w.unsqueeze(0).expand_as(param))
            else:
                unrecognized.append((name, shape))
                continue
            seeded += 1

        elif "bias" in lname and fc1_b is not None and "fc1" in lname:
            flat = param.reshape(-1)
            if flat.numel() == E * hidden:
                param.copy_(fc1_b.repeat(E).reshape(param.shape))
                seeded += 1
            elif flat.numel() == hidden:
                param.copy_(fc1_b.reshape(param.shape))
                seeded += 1
            else:
                unrecognized.append((name, shape))

        elif "bias" in lname and fc2_b is not None and "fc2" in lname:
            flat = param.reshape(-1)
            if flat.numel() == E * dim:
                param.copy_(fc2_b.repeat(E).reshape(param.shape))
                seeded += 1
            elif flat.numel() == dim:
                param.copy_(fc2_b.reshape(param.shape))
                seeded += 1
            else:
                unrecognized.append((name, shape))

    if seeded == 0:
        listing = [(n, tuple(p.shape)) for n, p in moe_mlp.moe_layer.named_parameters()]
        raise RuntimeError(
            "seed_moe_experts_from_dense: recognized no expert parameters. "
            f"Backend={moe_mlp.backend}. named_parameters()={listing}. "
            "The expert layout has changed — update this function; do NOT shape-guess."
        )
    if unrecognized:
        print(f"[seed experts] warning — unrecognized params left at init: {unrecognized}")
    return seeded


def _is_gate_param(lname: str) -> bool:
    """True for router/gate parameters, which seeding must never touch."""
    return "gate" in lname or re.search(r"(^|\.)wg", lname) is not None or "router" in lname


@torch.no_grad()
def seed_shared_expert_from_dense(moe_mlp, dense_state: dict, prefix: str) -> int:
    """Load a block's dense ``mlp.*`` weights into ``moe_mlp.shared_expert``.

    ``dense_state`` maps full model keys (``block4.0.mlp.fc1.weight``, ...) to
    tensors; ``prefix`` is the block prefix (``"block4.0."``). The shared
    expert IS a PVT v2 ``Mlp``, so the weights transfer verbatim — including
    ``dwconv`` when the shared expert was built with it. Returns the number of
    tensors loaded (0 when there is no shared expert).
    """
    shared = getattr(moe_mlp, "shared_expert", None)
    if shared is None:
        return 0

    target = shared.state_dict()
    sub = {}
    for key, value in dense_state.items():
        if not key.startswith(prefix + "mlp."):
            continue
        local = key[len(prefix) + len("mlp."):]        # e.g. "fc1.weight"
        if local in target and target[local].shape == value.shape:
            sub[local] = value

    shared.load_state_dict(sub, strict=False)

    # Report BOTH directions. A source tensor with no destination is expected
    # when the block dropped its DWConv (moe_block_dwconv: False) and alarming
    # otherwise, so say which it is rather than dropping weights silently.
    source_keys = {k[len(prefix) + len("mlp."):] for k in dense_state
                   if k.startswith(prefix + "mlp.")}
    no_destination = sorted(source_keys - set(target))
    if no_destination:
        conv_only = all(k.startswith("dwconv.") for k in no_destination)
        if conv_only and shared.dwconv is None:
            print(f"[shared expert] {prefix}: this block has no DWConv "
                  f"(moe_block_dwconv=False) — skipped {len(no_destination)} conv "
                  f"tensor(s); fc1/fc2 transferred in full.")
        else:
            print(f"[shared expert] WARNING — {prefix}: {len(no_destination)} source "
                  f"tensor(s) had no destination and were DROPPED: {no_destination}")

    left_at_init = sorted(set(target) - set(sub))
    if left_at_init:
        print(f"[shared expert] WARNING — {prefix}: loaded {len(sub)}/{len(target)} "
              f"tensors, left at random init: {left_at_init}")
    return len(sub)


@torch.no_grad()
def zero_shared_expert_output(moe_mlp) -> int:
    """Zero the SHARED expert's output projection (fc2 weight and bias).

    The counterpart to ``zero_routed_expert_output``: here the ROUTED experts
    carry the replicated pretrained FFN and the shared expert grows from zero.
    This is the Sparse-Upcycling-style init in docs/HPARAMS.md, whose stated
    purpose is to stop shared + routed both copying the FFN and emitting ~2x
    the dense layer at step 0.

    Caveat (see docs/HPARAMS.md): that scheme is only function-preserving if
    the router's combine weights are normalized per token to sum to 1. Tutel
    normalizes gates ONLY when ``top_k > 1`` (``impls/fast_dispatch.py``,
    ``extract_critical``), so at the default ``top_k: 1`` the routed branch is
    scaled by the raw softmax score (<1) and the block does NOT reproduce the
    dense FFN exactly. ``upcycle_init="routed_zero"`` does, at any top_k.

    Returns the number of tensors zeroed (0 when there is no shared expert).
    """
    shared = getattr(moe_mlp, "shared_expert", None)
    if shared is None:
        return 0
    shared.fc2.weight.zero_()
    zeroed = 1
    if shared.fc2.bias is not None:
        shared.fc2.bias.zero_()
        zeroed += 1
    return zeroed


@torch.no_grad()
def zero_routed_expert_output(moe_mlp) -> int:
    """Zero every routed expert's fc2 (weight and bias).

    With a shared expert holding the pretrained FFN, this makes the block's
    output at step 0 exactly ``shared_expert(x)`` — i.e. exactly the
    pretrained dense FFN — while the routed experts stay fully trainable
    (fc2 receives gradient from the first step, then fc1 through it). This is
    the residual-upcycling init (``upcycle_init="routed_zero"``); without a
    shared expert it would zero the block's entire output, which is why
    ``validate_config`` resolves that case to ``"none"``.

    Returns the number of tensors zeroed; raises if none were recognized.
    """
    zeroed = 0
    for name, param in moe_mlp.moe_layer.named_parameters():
        lname = name.lower()
        if _is_gate_param(lname):
            continue
        if "fc2" in lname or re.search(r"(^|\.)w2($|_)", lname):
            param.zero_()
            zeroed += 1
    if zeroed == 0:
        listing = [(n, tuple(p.shape)) for n, p in moe_mlp.moe_layer.named_parameters()]
        raise RuntimeError(
            "zero_routed_expert_output: found no fc2/w2 expert parameters. "
            f"Backend={moe_mlp.backend}. named_parameters()={listing}."
        )
    return zeroed


# ---------------------------------------------------------------------------
# Backbone checkpoints (SSL init, official .pth files, our own saves)
# ---------------------------------------------------------------------------

def _checkpoint_cfg(ckpt) -> dict | None:
    """The config a checkpoint was trained with, if it carries one.

    ``LitJEPA.save_backbone`` writes ``{"state_dict", "cfg"}``; a Lightning
    checkpoint keeps it under ``hyper_parameters["cfg"]``.
    """
    if not isinstance(ckpt, dict):
        return None
    if isinstance(ckpt.get("cfg"), dict):
        return ckpt["cfg"]
    hp = ckpt.get("hyper_parameters")
    if isinstance(hp, dict) and isinstance(hp.get("cfg"), dict):
        return hp["cfg"]
    return None


def check_backbone_architecture(ckpt_cfg: dict | None, cfg: dict, state_keys,
                                path: str = "<checkpoint>") -> list:
    """Compare a checkpoint's saved architecture with the run being built.

    Returns the list of mismatch descriptions (empty = compatible). Checked:
    variant, depths / embed_dims / num_heads / mlp_ratios / sr_ratios, RoPE
    on/off, RoPE mode and resolved placement, and MoE placement WHEN the
    checkpoint itself has MoE weights (a dense checkpoint feeding a MoE run
    is sparse upcycling, which is allowed). A checkpoint without a saved
    config cannot be checked; the caller decides how loud to be.
    """
    # [v12] inlined above: was `from pvt_moe.config import ...`

    if ckpt_cfg is None:
        return [f"{path} carries no config; architecture cannot be verified"]
    want, have = cfg["model"], ckpt_cfg.get("model", {})
    out = []
    if have.get("variant") not in (None, want.get("variant")):
        out.append(f"variant: checkpoint {have.get('variant')!r} vs run {want.get('variant')!r}")
    for key in VARIANT_ARCH_KEYS:
        if have.get(key) is not None and list(have[key]) != list(want[key]):
            out.append(f"model.{key}: checkpoint {list(have[key])} vs run {list(want[key])}")
    if out:                      # different depths: placements are not comparable
        return out
    depths = list(want["depths"])
    ha, wa = have.get("ablation", {}), want["ablation"]

    def _resolved(abl, kind):
        pl_ = abl.get(f"{kind}_placement")
        if pl_ is None:
            return None
        return resolve_placement(pl_, abl.get(f"{kind}_last_n_stages"), depths)

    h_rope = bool(ha.get("use_rope")) if "use_rope" in ha else None
    if h_rope is not None and h_rope != bool(wa["use_rope"]):
        out.append(f"use_rope: checkpoint {h_rope} vs run {bool(wa['use_rope'])}")
    elif h_rope:
        if ha.get("rope_mode") is not None and ha["rope_mode"] != wa["rope_mode"]:
            out.append(f"rope_mode: checkpoint {ha['rope_mode']!r} vs run {wa['rope_mode']!r}"
                       " (mixed frequencies exist only in a mixed checkpoint)")
        hp, wp = _resolved(ha, "rope"), _resolved(wa, "rope")
        if hp is not None and hp != wp:
            out.append(f"rope_placement: checkpoint {hp} vs run {wp} — blocks with RoPE in "
                       "only one of the two get dropped or random RoPE-Mixed frequencies")
    ckpt_has_moe = any(".mlp.moe_layer." in k or ".mlp.shared_expert." in k for k in state_keys)
    if ckpt_has_moe:
        hp = _resolved(ha, "moe") if ha.get("use_moe") else [[] for _ in depths]
        wp = _resolved(wa, "moe") if wa["use_moe"] else [[] for _ in depths]
        if hp != wp:
            out.append(f"moe_placement: checkpoint {hp} vs run {wp}")
        for key in ("num_experts", "shared_expert", "backend"):
            hv = ckpt_cfg.get("model", {}).get("moe", {}).get(key)
            if hv is not None and hv != want["moe"].get(key):
                out.append(f"model.moe.{key}: checkpoint {hv!r} vs run {want['moe'].get(key)!r}")
    return out


def load_backbone_checkpoint(
    model: nn.Module,
    path: str,
    skip_head: bool = True,
    verbose: bool = True,
    expected_cfg: dict | None = None,
    check_arch: bool = True,
    seed_moe_experts: bool = True,
    upcycle_init: str | None = None,
) -> dict:
    """Load a backbone state_dict from ``path`` into ``model``.

    Accepts raw state_dicts, ``{"state_dict": ...}`` / ``{"model": ...}``
    wrappers, and Lightning checkpoints; strips ``model.`` / ``module.``
    prefixes. Skips ``head.*`` by default (class count may differ). Keys with
    no destination or mismatched shapes are dropped and counted.

    Sparse upcycling, exactly as ``load_hf_pretrained``: a dense checkpoint's
    ``mlp.*`` tensors for blocks the model converted to MoE are not dropped
    but handed to ``upcycle_moe_blocks`` (when ``seed_moe_experts``), which
    seeds the routed experts and the shared expert and applies
    ``upcycle_init`` — so a JEPA backbone's own stage-4 FFN is the dense
    teacher and the MoE'd block reproduces it at step 0.

    With ``expected_cfg`` (the run's validated config) the checkpoint's saved
    architecture is compared first (``check_backbone_architecture``): a
    mismatch raises when ``check_arch`` is True and is printed as a loud
    warning otherwise. A checkpoint with no saved config is always a warning.
    """
    if upcycle_init is None:
        # Taken from the run's config so a caller cannot forget it and get
        # the pre-fix behaviour (a doubled FFN) by omission.
        upcycle_init = (expected_cfg["model"]["moe"].get("upcycle_init", "none")
                        if expected_cfg is not None else "none")
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    state = ckpt
    for wrapper in ("state_dict", "model"):
        if isinstance(state, dict) and wrapper in state and isinstance(state[wrapper], dict):
            state = state[wrapper]

    cleaned = {}
    for k, v in state.items():
        # "context." covers Lightning checkpoints written by LitJEPA (its
        # encoder attribute is self.context) and "encoder." those of LitSimMIM;
        # trailing dots keep the prefixes unambiguous. target./predictor./
        # head.0. keys intentionally get no prefix match and drop out.
        for prefix in ("model.", "module.", "backbone.", "context_encoder.", "context.",
                       "encoder."):
            if k.startswith(prefix):
                k = k[len(prefix):]
        cleaned[k] = v

    stats = {"loaded": 0, "skipped_head": 0, "dropped_no_target": 0, "skipped_shape": 0,
             "arch_mismatches": [], "dense_mlp_for_seeding": 0, "seeded_moe_blocks": 0,
             "seeded_shared_experts": 0, "zeroed_routed_fc2": 0, "zeroed_shared_fc2": 0}
    # Provenance of the parent run, for cfg["chain"] / results.json.
    parent_cfg = _checkpoint_cfg(ckpt) or {}
    stats["parent_chain"] = list(parent_cfg.get("chain") or [])
    stats["parent_run_name"] = parent_cfg.get("run_name")
    stats["parent_method"] = ckpt.get("method") if isinstance(ckpt, dict) else None
    moe_prefixes = moe_block_prefixes_of(model)
    dense_mlp_for_seeding = {}
    if expected_cfg is not None:
        problems = check_backbone_architecture(_checkpoint_cfg(ckpt), expected_cfg, cleaned, path)
        stats["arch_mismatches"] = problems
        if problems:
            text = "\n  - ".join(problems)
            if check_arch and _checkpoint_cfg(ckpt) is not None:
                raise ValueError(
                    f"ssl_init checkpoint {path} was trained with a different architecture:"
                    f"\n  - {text}\nMatch the run to the checkpoint (--variant / --rope-mode / "
                    f"--rope-placement ...) or set model.ssl_init_check_arch: false to load "
                    f"what fits and leave the rest at random init.")
            print(f"[backbone ckpt] WARNING: {text}")

    model_state = model.state_dict()
    filtered = {}
    for k, v in cleaned.items():
        if skip_head and k.startswith("head."):
            stats["skipped_head"] += 1
            continue
        if k not in model_state and any(k.startswith(p + "mlp.") for p in moe_prefixes):
            dense_mlp_for_seeding[k] = v          # the dense teacher of a MoE'd block
            stats["dense_mlp_for_seeding"] += 1
            continue
        if k not in model_state:
            stats["dropped_no_target"] += 1
            continue
        if model_state[k].shape != v.shape:
            stats["skipped_shape"] += 1
            continue
        filtered[k] = v

    missing, unexpected = model.load_state_dict(filtered, strict=False)
    stats["loaded"] = len(filtered)
    stats["missing"] = list(missing)
    stats["unexpected"] = list(unexpected)
    if seed_moe_experts and dense_mlp_for_seeding:
        upcycle_moe_blocks(model, dense_mlp_for_seeding, moe_prefixes, upcycle_init, stats)
    elif moe_prefixes and upcycle_init != "none" and verbose:
        # A MoE-carrying checkpoint loaded straight (pretraining path 2): the
        # routed experts and the shared expert came from the file, so the
        # zero-init rule had nothing to act on. Say so rather than leave the
        # printed upcycle_init looking like it did something.
        print(f"[backbone ckpt] checkpoint already carries the MoE weights for "
              f"{len(moe_prefixes)} block(s): loaded as trained, upcycle_init="
              f"{upcycle_init!r} not applied (nothing to upcycle)")
    if verbose:
        print(
            f"[backbone ckpt] loaded={stats['loaded']} head_skipped={stats['skipped_head']} "
            f"no_target={stats['dropped_no_target']} shape_skipped={stats['skipped_shape']} "
            f"missing={len(stats['missing'])} | upcycled: dense_mlp={stats['dense_mlp_for_seeding']} "
            f"seeded_moe_blocks={stats['seeded_moe_blocks']} "
            f"seeded_shared={stats['seeded_shared_experts']} "
            f"zeroed_routed_fc2={stats['zeroed_routed_fc2']} "
            f"zeroed_shared_fc2={stats['zeroed_shared_fc2']} (upcycle_init={upcycle_init})"
        )
        if stats["loaded"] == 0:
            print("  !! 0 weights loaded — wrong file or key prefix. First source keys: "
                  f"{list(cleaned)[:5]}")
        rope_random = [k for k in stats["missing"] if k.endswith("rope.freqs")]
        rope_dropped = [k for k in cleaned if k.endswith("rope.freqs") and k not in filtered]
        if rope_random or rope_dropped:
            print(f"  RoPE-Mixed frequencies left at RANDOM init: {rope_random or 'none'}; "
                  f"in the checkpoint but unused: {rope_dropped or 'none'}")
        moe_random = [k for k in stats["missing"] if ".mlp.moe_layer." in k or ".mlp.shared_expert." in k]
        if moe_random and not stats["seeded_moe_blocks"]:
            print(f"  MoE'd blocks start at RANDOM init ({len(moe_random)} tensors): the "
                  "checkpoint had no dense FFN for them or seed_moe_experts is off.")
    return stats


## Δ since v10 — LitModel → LitClassifier

**Optimizer:**
<pre>
<del>params = [{"params": stages123, "lr": lr}, {"params": stage4, "lr": lr * 10}]</del>
4 groups: stages123/stage4 × decay/no-decay          # wd 0.05 vs 0.0
  no-decay = every ndim<=1 param + model.no_weight_decay()   # norms, biases,
  stage4 multiplier still exists but is 1.0 in every recipe  # RoPE freqs
optional optim.layer_decay < 1 → per-block LR ladder (BEiT/SimMIM scheme)
</pre>
&nbsp;&nbsp;*why:* wd on norm weights and on the learnable RoPE frequencies is
actively harmful; v10's ×10 stage-4 multiplier belonged to its
frozen-backbone resume, not to from-scratch training.

**Schedule:** unchanged shape (LinearLR warmup → cosine, stepped per epoch),
but driven by the recipe (5 ep from 1e-6 → 1e-3 → cosine to 1e-6) instead of
<del>warmup 0 / start_factor 1e-6 / lr 1e-4</del>.

**Training step:** same mixup → `SoftTargetCrossEntropy` core as v10, plus
<pre>
aux = torch.clamp(aux, max=self.aux_clamp)          # spike guard (10.0)
if torch.isnan(loss) or torch.isinf(loss): loss = ce_loss   # drop aux, keep CE
</pre>

**Kept from v10:** the per-epoch confusion-matrix reset (hand-updated
torchmetrics are not auto-reset — v9's matrix accumulated forever).

**New:** `save_hyperparameters({"cfg": cfg})` — the checkpoint carries its own
resolved config, which is what lets `tools/probe_checkpoint.py` read a run's
identity, LR and Adam moments off disk without rebuilding the model (rebuilding
under a guessed config with `strict=False` zero-fills missing experts and
probes a network that was never trained). Plus `chain` provenance: a warm
start prepends the parent's stages, so results.json names the whole path.


In [ ]:
# ═══ pvt_moe/utils/flops.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""MoE-aware compute accounting.

All numbers use fvcore's convention: **1 multiply-add (MAC) = 1 "FLOP"** —
the same convention as the PVT/Swin papers' GFLOPs tables. fvcore traces the
dense compute; MoE expert FFNs are stubbed during tracing (fvcore cannot
trace Tutel/MegaBlocks kernels) and added back analytically in the SAME MAC
convention:

    moe_macs = seq_len * [ k * (dim*hidden + hidden*dim)   (routed expert FFNs)
                           + dim * num_experts              (router)
                           + (dim*hidden + hidden*dim)      (shared expert, if any)
                           + 9 * hidden ]                   (its DWConv, if any)

The shared expert is counted here rather than by fvcore because the ENTIRE
``MoEMlp.forward`` — shared branch included — is stubbed during tracing.
Forgetting it under-reports total GFLOPs, which is exactly the number an
ablation table compares.

Stage geometry: stage i tokens sit on an ``img_size / (4 * 2^i)`` grid.
"""

from __future__ import annotations

import re
import types

import torch

_STAGE_STRIDES = (4, 8, 16, 32)


def _analytic_moe_flops(model, img_size: int) -> int:
    # [v12] inlined above: was `from pvt_moe.models.ffn import ...`

    total = 0
    for name, module in model.named_modules():
        if not isinstance(module, MoEMlp):
            continue
        stage_match = re.search(r"block(\d+)", name)
        if not stage_match:
            raise ValueError(f"Cannot infer stage index from module name {name!r}")
        stage = int(stage_match.group(1)) - 1
        seq_len = (img_size // _STAGE_STRIDES[stage]) ** 2
        dim, hidden = module.in_features, module.hidden_features
        # MAC convention (1 multiply-add = 1), matching fvcore's dense count.
        expert_ffn = dim * hidden + hidden * dim                # fc1 + fc2
        router = dim * module.num_experts
        per_token = module.top_k * expert_ffn + router

        # The shared expert (when enabled) runs for EVERY token — it is dense
        # compute, not routed, so it carries no top_k factor.
        shared = getattr(module, "shared_expert", None)
        if shared is not None:
            per_token += expert_ffn
            if shared.dwconv is not None:
                # depthwise 3x3 over `hidden` channels: 9 MACs per channel
                # per token (groups == channels, so no cross-channel term).
                per_token += 9 * hidden

        total += seq_len * per_token
    return total


def _has_shared(model) -> bool:
    # [v12] inlined above: was `from pvt_moe.models.ffn import ...`

    return any(
        getattr(m, "shared_expert", None) is not None
        for m in model.modules()
        if isinstance(m, MoEMlp)
    )


def count_flops(model, img_size: int = 224, verbose: bool = True) -> dict:
    """Count model FLOPs for one image. Returns dict of GFLOPs components."""
    from fvcore.nn import FlopCountAnalysis, flop_count_table  # lazy

    # [v12] inlined above: was `from pvt_moe.models.ffn import ...`

    was_training = model.training
    model.eval()
    device = next(model.parameters()).device
    x = torch.randn(1, 3, img_size, img_size, device=device)

    # Stub MoE forwards during tracing (fvcore can't trace expert kernels).
    originals = {}
    for name, module in model.named_modules():
        if isinstance(module, MoEMlp):
            originals[name] = module.forward

            def _stub(self, x, H, W):
                return torch.zeros_like(x), torch.zeros((), device=x.device)

            module.forward = types.MethodType(_stub, module)

    try:
        with torch.no_grad():
            analysis = FlopCountAnalysis(model, x)
            analysis.unsupported_ops_warnings(False)
            analysis.uncalled_modules_warnings(False)
            dense_flops = analysis.total()
        table = flop_count_table(analysis, max_depth=3)
    finally:
        for name, module in model.named_modules():
            if name in originals:
                module.forward = originals[name]
        model.train(was_training)

    moe_flops = _analytic_moe_flops(model, img_size)
    result = {
        "dense_gflops": dense_flops / 1e9,
        "moe_gflops": moe_flops / 1e9,
        "total_gflops": (dense_flops + moe_flops) / 1e9,
    }
    if verbose:
        print("(MAC convention: 1 multiply-add = 1 FLOP, as in the PVT/Swin papers)")
        print(f"Dense: {result['dense_gflops']:.3f} G")
        print(f"MoE:   {result['moe_gflops']:.3f} G "
              f"(top-k active experts + router{' + shared expert' if _has_shared(model) else ''})")
        print(f"Total: {result['total_gflops']:.3f} G")
        print(table)
    return result


def count_params(model) -> dict:
    """Total / trainable / MoE-expert parameter counts (M)."""
    # [v12] inlined above: was `from pvt_moe.models.ffn import ...`

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    moe = sum(
        p.numel()
        for m in model.modules()
        if isinstance(m, MoEMlp)
        for p in m.parameters()
    )
    # Shared-expert params are always-on dense compute, so they are reported
    # separately from the routed bank (of which only top_k/num_experts is
    # active per token) — the two numbers mean different things in a table.
    shared = sum(
        p.numel()
        for m in model.modules()
        if isinstance(m, MoEMlp) and getattr(m, "shared_expert", None) is not None
        for p in m.shared_expert.parameters()
    )
    result = {
        "total_m": total / 1e6,
        "trainable_m": trainable / 1e6,
        "moe_m": moe / 1e6,
        "shared_expert_m": shared / 1e6,
        "routed_expert_m": (moe - shared) / 1e6,
        "dense_m": (total - moe) / 1e6,
    }
    print(
        f"Params: {result['total_m']:.1f}M total | {result['trainable_m']:.1f}M trainable | "
        f"{result['routed_expert_m']:.1f}M routed experts | "
        f"{result['shared_expert_m']:.1f}M shared expert | {result['dense_m']:.1f}M dense"
    )
    return result


In [ ]:
# ═══ pvt_moe/utils/diagnostics.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""Diagnostics: MoE routing statistics, expert utilization, training-curve plots.

Expert utilization is THE first thing to check when an MoE run underperforms:
top-1 routing can collapse onto a few experts, at which point the extra
capacity is dead weight. Healthy top-1 routing with 8 experts shows every
expert between ~5% and ~25% token share and entropy near log(8) = 2.08.

The SECOND thing to check is whether tokens were dropped. An expert takes at
most ``capacity_factor * ceil(tokens / E)`` tokens per forward (``top_k`` x
that when routing k-way); everything past that gets exactly zero from the
routed branch. A collapsed router and a starved capacity look identical in
the loss and opposite in the fix, so ``routing_stats`` measures both in one
pass: "MoE did not help" and "the tokens never reached an expert" are
different results.

**Do not use the load-balancing loss (`train_aux`) as the balance metric.**
Both backends compute ``aux = E * sum_i f_i * p_i`` where ``f`` is the token
share per expert and ``p`` the mean gate probability. Writing ``f = 1/E + a``
and ``p = 1/E + b`` (both deviations sum to zero) that is exactly::

    aux = 1 + E * <a, b>

so ``aux - 1`` is a product of TWO deviations — second order in the imbalance,
and identically zero whenever either factor vanishes. Two consequences:

* **Low resolution.** A router whose worst expert holds 26% of the tokens
  reads about 1.0004; you need roughly 37% before it reaches 1.03. Most of the
  interesting range of imbalance lives in the fourth decimal place.
* **A blind spot.** It reads exactly 1.0 whenever the mean gate probability is
  uniform, *no matter how skewed the assignment is* — a router that argmaxes
  every token onto one expert with a near-flat softmax reads 1.0. And the
  loss's own gradient, ``(E/T) * p_j * (f_j - <f, p>)`` per logit, drives ``p``
  toward uniform, so a working balancer walks into its own blind spot. The
  gradient still sees the imbalance ``a`` (it vanishes exactly when ``f`` is
  uniform); only the reported *number* collapses to the correlation ``<a, b>``.

Dtype is NOT part of the story on the default backend, though it is close:
Tutel runs its whole routing block with autocast disabled
(``tutel/impls/moe_layer.py``), so ``l_aux`` is fp32 even under bf16-mixed.
The native backend has to turn autocast off explicitly to match — ``x.float()``
alone does not, because autocast casts the ``nn.Linear`` itself
(``moe_native.py``). Were the loss ever computed in bf16, round-to-nearest
would absorb the whole interval ``[1 - 2^-9, 1 + 2^-8] = [0.998047, 1.003906]``
into exactly 1.0 — asymmetric, because bf16's spacing is 2^-8 below 1.0 and
2^-7 above it.

One thing 1.0 is NOT: a floor. ``<a, b>`` is a correlation, and the argmax
constraint does not force it positive. A router where 90% of tokens pick one
expert by a hair while the remaining 10% pick another with confidence has the
share and the mean probability ANTI-correlated, and reads ``aux = 0.94``
(``tests/test_routing_diagnostics.py``). The attainable range is roughly
``[E/(2(E-1)), E]``; 1.0 is where the correlation crosses zero, which is one
more reason not to read health off the number.

``logit_routing_stats`` below reports what the aux value cannot: the token share,
the drop rate, and the two entropies that tell a confidently-balanced router
apart from a uniformly-undecided one.
"""

from __future__ import annotations

import math

import torch


def capacity_of(num_tokens: int, num_experts: int, capacity_factor: float,
                top_k: int = 1) -> int:
    """Per-expert token cap — Tutel's formula, mirrored by the native backend.

    ``capacity_factor <= 0`` means no cap (Tutel's dynamic capacity).
    """
    if capacity_factor <= 0:
        return num_tokens
    per_expert = math.ceil(num_tokens / num_experts)
    return max(1, top_k * int(capacity_factor * per_expert))


def logit_routing_stats(logits: torch.Tensor, capacity_factor: float = 1.0, top_k: int = 1,
                  dropless: bool = False) -> dict:
    """What top-1 routing actually did, from one MoE layer's gate logits.

    ``logits`` is ``(tokens, num_experts)``. Everything is computed in fp64 so
    the numbers are not themselves quantised (see the module docstring).

    Returned keys, and why each one exists:

    ``share``            token fraction per expert (``f``). The headline.
    ``imbalance``        ``sum_i max(0, f_i - 1/E)`` — the total-variation
                         distance between the routing distribution and uniform.
                         0 = perfect balance, ``1 - 1/E`` = full collapse.
    ``drop_rate``        fraction of tokens over capacity, i.e. tokens that get
                         NOTHING from the routed branch. The metric with
                         physical consequence. At ``capacity_factor == 1.0``
                         it equals ``imbalance`` EXACTLY when E divides the
                         token count (the production case: 6272 tokens over 4
                         experts gives capacity 1568 = T/E). Otherwise
                         ``capacity = ceil(T/E) > T/E`` and this is a
                         THRESHOLDED total variation that under-reads, with a
                         dead zone near balance — at T=49 (one image's stage-4
                         grid) it reads 0.143 where ``imbalance`` reads 0.158,
                         and at ``capacity_factor > 1`` the dead zone is large
                         by design. Always 0 for a dropless backend
                         (megablocks). Prefer ``imbalance`` as the balance
                         measure and this as the cost measure.
    ``route_entropy``    entropy of ``f`` in nats; ``max_entropy`` is ``log E``.
    ``gate_entropy``     mean per-token entropy of the gate softmax. This is the
                         one that separates the two cases a flat ``aux`` cannot:
                         near ``log E`` means the router is undecided (and then
                         ``aux`` is pinned at 1 whatever the share does), well
                         below it means the router is confident.
    ``mean_gate_prob``   ``p``. ``aux`` is blind to the share whenever this is
                         uniform.
    ``aux``              the load-balancing loss recomputed in fp64 from these
                         same logits, so it can be compared against the logged
                         ``train_aux``; ``aux_excess`` is ``aux - 1``.

    The decision is recomputed from the logits the layer was given, WITHOUT
    the gate noise the backend may have added: this measures the router's
    policy, not the realised noisy sample. Both backends add zero-mean
    GAUSSIAN noise scaled by ``gate_noise / num_experts`` (σ = 0.125 at the
    defaults), so with ``gate_noise > 0`` the two differ by however much that
    moves tokens across the capacity line — a few tenths of a percent at the
    measured stage-4 logit spread. ``RoutingMonitor`` reports both.
    """
    if logits.ndim != 2:
        raise ValueError(f"expected (tokens, experts) gate logits, got {tuple(logits.shape)}")
    lg = logits.detach().double()
    tokens, num_experts = lg.shape
    probs = lg.softmax(dim=-1)
    index = lg.argmax(dim=-1)
    counts = torch.bincount(index, minlength=num_experts).double()
    share = counts / max(1, tokens)
    mean_p = probs.mean(dim=0)

    capacity = capacity_of(tokens, num_experts, capacity_factor, top_k)
    dropped = 0.0 if dropless else float((counts - capacity).clamp(min=0).sum())

    def _entropy(q):
        q = q[q > 0]
        return float(-(q * q.log()).sum()) if q.numel() else 0.0

    per_token_entropy = -(probs.clamp_min(1e-12).log() * probs).sum(dim=-1)
    aux = float(num_experts * (share * mean_p).sum())
    return {
        "tokens": int(tokens),
        "num_experts": int(num_experts),
        "counts": [int(c) for c in counts],
        "share": [round(float(v), 6) for v in share],
        "imbalance": round(float((share - 1.0 / num_experts).clamp(min=0).sum()), 6),
        "capacity": int(capacity),
        "dropped_tokens": int(dropped),
        "drop_rate": round(dropped / max(1, tokens), 6),
        "dropless": bool(dropless),
        "route_entropy": round(_entropy(share), 6),
        "gate_entropy": round(float(per_token_entropy.mean()), 6),
        "max_entropy": round(math.log(num_experts), 6),
        "mean_gate_prob": [round(float(v), 6) for v in mean_p],
        "aux": round(aux, 8),
        "aux_excess": round(aux - 1.0, 8),
    }


def gate_logits(moe_mlp, x_flat: torch.Tensor) -> torch.Tensor:
    """Router logits ``(tokens, E)`` of one MoE layer on its actual input.

    Tutel and the native backend keep the gate at ``moe_layer.gates[0].wg``;
    megablocks' dMoE has ``router.layer``; the test suite's fake Tutel layer
    carries a bare ``gate_wg`` weight.
    """
    layer = moe_mlp.moe_layer
    if hasattr(layer, "gates"):
        gate = layer.gates[0]
        return gate.wg(x_flat.to(gate.wg.weight.dtype))
    if hasattr(layer, "gate_wg"):
        return x_flat.to(layer.gate_wg.dtype) @ layer.gate_wg.t()
    # megablocks: dMoE.router is a LearnedRouter with a .layer Linear.
    # Its weights are bf16 (never fp32) — cast the input to match.
    router = layer.router
    lin = getattr(router, "layer", router)
    return lin(x_flat.to(lin.weight.dtype))


def expert_capacity(moe_mlp, tokens: int) -> int | None:
    """Per-expert token cap for ONE forward over ``tokens`` tokens.

    The native backend owns the formula (``NativeMoEFFN.capacity_for``) and is
    asked directly, so this can never drift from what that layer actually
    enforces. Tutel uses the same formula
    (``top_k * int(capacity_factor * ceil(tokens / E))``), replicated here
    because its layer does not expose it.

    ``None`` means no cap applies and nothing can be dropped: the megablocks
    backend is dropless by construction, and ``capacity_factor <= 0`` is
    Tutel's dynamic capacity.
    """
    layer = getattr(moe_mlp, "moe_layer", None)
    if getattr(moe_mlp, "backend", None) == "megablocks":
        return None
    cap_f = getattr(moe_mlp, "capacity_factor", None)
    if cap_f is not None and cap_f <= 0:
        return None
    fn = getattr(layer, "capacity_for", None)
    if callable(fn):
        return int(fn(tokens))
    if cap_f is None:
        return None
    per_expert = math.ceil(tokens / moe_mlp.num_experts)
    return max(1, moe_mlp.top_k * int(cap_f * per_expert))


@torch.no_grad()
def routing_stats(model, dataloader, num_batches: int = 50, device=None) -> dict:
    """Route ``num_batches`` of data and measure, per MoE block, BOTH which
    experts the router picked and how many tokens capacity threw away.

    Works with every backend via forward-pre-hooks on each ``MoEMlp`` (the
    hook recomputes the router decision on the layer's actual input — no
    manual forward re-implementation to drift out of sync).

    Returns ``{block_name: {"counts": LongTensor[E], "routed": int,
    "dropped": int, "drop_fraction": float, "capacity": int | None,
    "tokens_per_forward": int, "forwards": int}}``.

    Capacity is enforced per FORWARD over the whole flattened micro-batch
    (``MoEMlp`` reshapes (B, N, C) -> (B*N, C)), not per image, so the drop
    accounting is done per forward and then summed — aggregating counts first
    and applying a cap afterwards would understate drops on a peaked router
    and overstate them on a flat one.

    ``dropped`` counts routing SLOTS, not tokens: at ``top_k`` k each token
    makes k requests and ``routed`` is ``k * tokens``. At the top_k=1 every
    shipped arm uses, a slot is a token and the two coincide. The per-expert
    queue is filled in token order, exactly as ``NativeMoEFFN.forward`` does
    it, which makes the count exact there; for Tutel at top_k > 1 it is close
    but not exact, since Tutel ranks each k-slot separately.
    """
    # [v12] inlined above: was `from pvt_moe.models.ffn import ...`

    device = device or next(model.parameters()).device
    was_training = model.training
    model.eval()

    moe_modules = [
        (name, m) for name, m in model.named_modules() if isinstance(m, MoEMlp)
    ]
    if not moe_modules:
        print("No MoE modules in this model.")
        return {}

    stats = {
        name: {"counts": torch.zeros(m.num_experts, dtype=torch.long), "routed": 0,
               "dropped": 0, "capacity": None, "tokens_per_forward": 0, "forwards": 0}
        for name, m in moe_modules
    }

    hooks = []

    def _make_hook(name, moe_mlp):
        def hook(module, args):
            x = args[0]
            x_flat = x.reshape(-1, x.shape[-1])
            tokens, k = x_flat.shape[0], moe_mlp.top_k
            logits = gate_logits(moe_mlp, x_flat)
            assign = logits.topk(k, dim=-1).indices if k > 1 else logits.argmax(dim=-1)[:, None]
            s = stats[name]
            # counts stay the TOP-1 choice at any k, so the share/entropy
            # numbers mean the same thing they always did.
            s["counts"] += torch.bincount(assign[:, 0].cpu(), minlength=moe_mlp.num_experts)
            s["forwards"] += 1
            s["tokens_per_forward"] = tokens
            capacity = expert_capacity(moe_mlp, tokens)
            s["capacity"] = capacity
            s["routed"] += tokens * k
            if capacity is not None:
                per_expert = torch.bincount(assign.reshape(-1).cpu(),
                                            minlength=moe_mlp.num_experts)
                s["dropped"] += int((per_expert - capacity).clamp(min=0).sum())

        return hook

    for name, m in moe_modules:
        hooks.append(m.register_forward_pre_hook(_make_hook(name, m)))

    # The megablocks backend stores expert/router weights in bf16 and its
    # grouped kernels never run fp32 — the forward must happen under autocast.
    # Harmless (and representative of training) for tutel too.
    use_autocast = device.type == "cuda"
    try:
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_autocast):
            for batch_idx, (x, _) in enumerate(dataloader):
                if batch_idx >= num_batches:
                    break
                model(x.to(device))
    finally:
        for h in hooks:
            h.remove()
        model.train(was_training)

    for s in stats.values():
        s["drop_fraction"] = (round(s["dropped"] / s["routed"], 6) if s["routed"] else 0.0)
    return stats


@torch.no_grad()
def expert_utilization(model, dataloader, num_batches: int = 50, device=None) -> dict:
    """Token counts per expert per MoE block: ``{block_name: LongTensor[E]}``.

    The counts half of ``routing_stats`` (same single pass, same numbers),
    kept as its own name because the notebooks and ``plot_expert_utilization``
    take exactly this shape.
    """
    return {name: s["counts"]
            for name, s in routing_stats(model, dataloader, num_batches, device).items()}


def plot_expert_utilization(counts: dict):
    """Bar chart per MoE block: token share per expert + routing entropy."""
    import matplotlib.pyplot as plt

    if not counts:
        return
    n = len(counts)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 5), squeeze=False)
    for ax, (name, c) in zip(axes[0], counts.items()):
        c = c.float()
        total = c.sum().clamp(min=1)
        num_experts = c.numel()
        pcts = (c / total * 100).numpy()
        ideal = 100.0 / num_experts

        bars = ax.bar(range(num_experts), pcts, color="steelblue", edgecolor="black")
        for bar, pct in zip(bars, pcts):
            if pct < ideal * 0.5:
                bar.set_color("tomato")   # underloaded — possible collapse
            elif pct > ideal * 1.5:
                bar.set_color("gold")     # overloaded
        ax.axhline(ideal, color="red", linestyle="--", label=f"ideal {ideal:.1f}%")

        probs = c / total
        probs = probs[probs > 0]
        entropy = -(probs * probs.log()).sum().item()
        ax.set_title(f"{name}\nentropy {entropy:.2f} / {math.log(num_experts):.2f}")
        ax.set_xlabel("expert")
        ax.set_ylabel("token share (%)")
        ax.set_xticks(range(num_experts))
        ax.legend()
        ax.grid(True, alpha=0.3, axis="y")
    plt.suptitle("MoE expert utilization")
    plt.tight_layout()
    plt.show()

    for name, c in counts.items():
        c = c.float()
        pcts = c / c.sum().clamp(min=1) * 100
        print(f"\n{name}: {int(c.sum())} tokens")
        for i, p in enumerate(pcts):
            print(f"  expert {i}: {p:5.1f}% {'█' * int(p * 2)}")


def plot_training_curves(metrics_csv: str, save_path: str | None = None):
    """Plot accuracy / loss / aux curves from a Lightning CSVLogger file."""
    import matplotlib.pyplot as plt
    import pandas as pd

    df = pd.read_csv(metrics_csv)
    epoch_df = df.groupby("epoch").mean(numeric_only=True).reset_index()

    panels = [
        ("accuracy", ["val_acc", "val_acc_top5", "train_acc_mixed"]),
        ("loss", ["train_loss", "val_loss", "train_ce"]),
        ("aux loss", ["train_aux"]),
    ]
    present = [
        (title, [c for c in cols if c in epoch_df.columns]) for title, cols in panels
    ]
    present = [(t, c) for t, c in present if c]

    fig, axes = plt.subplots(1, len(present), figsize=(6 * len(present), 4), squeeze=False)
    for ax, (title, cols) in zip(axes[0], present):
        for col in cols:
            series = epoch_df[["epoch", col]].dropna()
            ax.plot(series["epoch"], series[col], marker=".", label=col)
        ax.set_title(title)
        ax.set_xlabel("epoch")
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"saved -> {save_path}")
    plt.show()


In [ ]:
# ═══ pvt_moe/engine/results.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""``results.json`` + ``results.md``: one record per run, refreshed every epoch.

Every training run — supervised, intermediate fine-tune, downstream, SSL —
writes ``<checkpoint_root>/<run_name>/results.json`` at every epoch boundary
(and once more at the end), so a killed run still leaves its numbers and a
thesis table never needs a W&B export. ``tools/compare_runs.py`` reads these
files. The record has five parts:

- ``identity``  — run name, version, variant, task / recipe / mode, dataset
  and resolution, seed, the full ``chain`` of stages that produced the
  weights (e.g. ``simmim_pretrain@pass_r224 -> ssl_finetune@imagenet-1k_r224
  -> downstream@eurosat_r224``), the parent checkpoint, the git commit and
  a hash of the resolved config;
- ``accuracy``  — the latest and the best validation top-1 / top-5, macro
  precision / recall, losses; for SSL runs the pretraining losses and an
  explicit note that probe / k-NN accuracy is expected to be low under MIM;
- ``efficiency`` — MEASURED: seconds per epoch, images per second, peak VRAM;
  plus parameter counts and GFLOPs (fvcore, when installed);
- ``moe``       — the aux loss and expert utilisation (token share and
  routing entropy per MoE block on a few validation batches); for MoE
  pretraining the mask-vs-visible routing split (``pvt_moe.ssl.diagnostics``);
- ``environment`` — torch / CUDA / Lightning versions, GPU name, platform.

``history`` keeps one row per epoch. ``eval`` is reserved for ``evaluate.py``
(k-NN, linear probe, test-split numbers), which merges into the same file.
The callback's state (history, best) travels inside every checkpoint, so a
resume on another machine continues the record instead of restarting it.
"""

from __future__ import annotations

import hashlib
import json
import os
import platform
import subprocess
import sys
import time

import pytorch_lightning as pl
import torch

SCHEMA = "pvt_moe.results/1"
JSON_NAME = "results.json"
MD_NAME = "results.md"

_METRIC_KEYS = (
    "train_loss", "train_ce", "train_aux", "train_acc_mixed",
    "val_loss", "val_acc", "val_acc_top5", "val_precision_macro", "val_recall_macro",
    "ssl_loss", "recon_loss", "mask_ratio", "target_std", "pred_std", "ema_momentum",
    # Routing (RoutingMonitor). These belong in the per-epoch history, not just
    # in the latest snapshot: collapse is something you watch DEVELOP, and
    # train_aux cannot show it (AUX_NOTE).
    "train_drop_rate", "train_drop_rate_realised", "train_moe_imbalance",
    "train_route_entropy", "train_gate_entropy",
)

AUX_NOTE = ("train_aux is a poor balance metric: aux = E*sum_i f_i*p_i is identically "
            "1 + E*<f - 1/E, p - 1/E>, a product of two deviations. It is therefore SECOND "
            "ORDER in the imbalance (a 26% worst-expert share reads ~1.0004; ~37% is needed "
            "for 1.03), and it reads exactly 1.0 whenever the mean gate probability p is "
            "uniform however skewed the token share f is — while the aux gradient itself "
            "drives p toward uniform. Read moe.routing.*.drop_rate (first order, no floor), "
            ".share, and .gate_entropy (near log E = an undecided router, which is exactly "
            "when aux is pinned) instead.")

MIM_PROBE_NOTE = ("Linear-probe / k-NN accuracy is EXPECTED to be low for a masked-image-"
                  "modelling encoder (SimMIM, MAE, BEiT all report weak probes and strong "
                  "fine-tuning); here they are collapse detectors. The headline number of a "
                  "SimMIM arm is the fine-tuned top-1 (docs/SIMMIM_GUIDE.md §7).")


def _float(v):
    if isinstance(v, torch.Tensor):
        return float(v.detach().cpu())
    return v


def git_commit(repo_dir: str | None = None) -> str | None:
    """Short commit hash of the code that ran, or None outside a git checkout."""
    try:
        here = repo_dir or os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
        out = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=here, capture_output=True,
                             text=True, timeout=5)
        return out.stdout.strip() or None
    except Exception:  # noqa: BLE001 — provenance is best-effort
        return None


def config_hash(cfg: dict) -> str:
    return hashlib.sha1(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:12]


def environment_info() -> dict:
    info = {"python": sys.version.split()[0], "torch": torch.__version__,
            "cuda": torch.version.cuda, "lightning": pl.__version__,
            "platform": platform.platform()}
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        info["gpu"] = props.name
        info["gpu_vram_gib"] = round(props.total_memory / 1024 ** 3, 1)
        info["gpu_arch"] = f"sm_{props.major}{props.minor}"
    else:
        info["gpu"] = None
    return info


def run_identity(cfg: dict) -> dict:
    m, abl, moe = cfg["model"], cfg["model"]["ablation"], cfg["model"]["moe"]
    ident = {
        "run_name": cfg["run_name"], "version": cfg.get("version"), "variant": m["variant"],
        "task": cfg.get("task"), "recipe": cfg.get("recipe"), "mode": cfg.get("mode"),
        "dataset": cfg["dataset"]["name"], "img_size": cfg["dataset"]["img_size"],
        "num_classes": cfg["dataset"].get("num_classes"), "seed": cfg.get("seed"),
        "chain": list(cfg.get("chain") or []), "parent_ckpt": cfg.get("ckpt_path"),
        "epochs": cfg["ssl"]["epochs"] if cfg.get("task") == "ssl" else cfg["epochs"],
        "effective_batch_size": cfg.get("effective_batch_size"),
        "precision": cfg.get("precision"),
        "norm": m["norm_type"], "dense_dwconv": m.get("dense_dwconv", True),
        # Stochastic depth leaves NO trace in the checkpoint (DropPath holds no
        # parameters and no buffers) and none in the run name, so this record
        # is the only place a resume can check it against — see
        # assert_resume_identity.
        "drop_path_rate": m["drop_path_rate"],
        "rope": ({"mode": abl["rope_mode"], "placement": abl["rope_placement"],
                  "theta": abl["rope_theta"]} if abl["use_rope"] else None),
        # capacity_factor and gate_noise change what the router does and are in
        # neither the run name nor the state dict: a finished run that does not
        # record them cannot say whether an arm underperformed from capacity
        # starvation or from expert count.
        "moe": ({"placement": abl["moe_placement"], "num_experts": moe["num_experts"],
                 "top_k": moe["top_k"], "shared_expert": moe["shared_expert"],
                 "backend": moe["backend"], "upcycle_init": moe.get("upcycle_init"),
                 "capacity_factor": moe.get("capacity_factor"),
                 "gate_noise": moe.get("gate_noise"),
                 "aux_weight": (cfg.get("loss") or {}).get("aux_weight")}
                if abl["use_moe"] and any(abl["moe_placement"]) else None),
        "git_commit": git_commit(), "config_sha1": config_hash(cfg),
    }
    if cfg.get("task") == "ssl":
        s = cfg["ssl"]
        ident["ssl"] = {"method": s["method"], "lr": s["lr"], "base_lr": s["base_lr"],
                        "mask_patch_size": s["mask_patch_size"], "mask_ratio": s["mask_ratio"],
                        "mask_space": s["mask_space"], "grad_clip": s.get("grad_clip")}
    else:
        o = cfg["optim"]
        ident["optim"] = {"lr": o["lr"], "base_lr": o.get("base_lr"),
                          "layer_decay": o.get("layer_decay"), "warmup_epochs": o["warmup_epochs"],
                          "weight_decay": o["weight_decay"], "grad_clip": o.get("grad_clip")}
    return ident


class ResultsWriter(pl.Callback):
    """Write ``results.json`` / ``results.md`` into the run directory every epoch."""

    def __init__(self, cfg: dict, dirpath: str, utilization_batches: int = 8):
        self.cfg = cfg
        self.dirpath = dirpath
        self.utilization_batches = utilization_batches
        self.history: list = []
        self.best: dict = {}
        self._static: dict = {}
        self._t_epoch = None
        self._t_fit = None

    # -- persisted with every checkpoint ------------------------------------
    def state_dict(self) -> dict:
        return {"history": self.history, "best": self.best}

    def load_state_dict(self, state_dict: dict) -> None:
        self.history = list(state_dict.get("history") or [])
        self.best = dict(state_dict.get("best") or {})

    # -- static facts, once per fit ------------------------------------------
    def on_fit_start(self, trainer, pl_module):
        self._t_fit = time.time()
        model = getattr(pl_module, "model", None) or getattr(pl_module, "encoder", None) \
            or getattr(pl_module, "context", None) or pl_module
        static = {"environment": environment_info()}
        try:
            # [v12] inlined above: was `from pvt_moe.utils.flops import ...`

            import contextlib
            import io
            with contextlib.redirect_stdout(io.StringIO()):
                static["params"] = count_params(model)
        except Exception as e:  # noqa: BLE001
            static["params"] = {"error": str(e)}
        try:
            # [v12] inlined above: was `from pvt_moe.utils.flops import ...`

            static["gflops"] = count_flops(model, self.cfg["dataset"]["img_size"], verbose=False)
        except Exception as e:  # noqa: BLE001 — fvcore is optional
            static["gflops"] = {"unavailable": f"{type(e).__name__}: {e}"[:200]}
        self._static = static

    def on_train_epoch_start(self, trainer, pl_module):
        self._t_epoch = time.time()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    # -- one row per epoch -------------------------------------------------------
    def _metrics(self, trainer) -> dict:
        m = trainer.callback_metrics
        out = {}
        for k in _METRIC_KEYS:
            if k in m:
                out[k] = _float(m[k])
        return out

    def _moe_block(self, trainer, pl_module) -> dict | None:
        cfg = self.cfg
        abl = cfg["model"]["ablation"]
        if not (abl["use_moe"] and any(abl["moe_placement"])):
            return None
        moe_cfg = cfg["model"]["moe"]
        block = {"aux_weight": cfg["loss"]["aux_weight"],
                 "capacity_factor": moe_cfg.get("capacity_factor"),
                 "gate_noise": moe_cfg.get("gate_noise")}
        # Per-epoch training-time routing stats (RoutingMonitor), and the
        # warning that goes with the aux number so a reader of this file
        # cannot mistake a pinned 1.0 for a healthy router.
        monitor = next((cb for cb in getattr(trainer, "callbacks", [])
                        if type(cb).__name__ == "RoutingMonitor"), None)
        if monitor is not None and getattr(monitor, "last_stats", None):
            block["routing"] = monitor.last_stats
            block["aux_note"] = AUX_NOTE
        model = getattr(pl_module, "model", None)
        loader = getattr(trainer, "val_dataloaders", None)
        if model is not None and loader is not None and self.utilization_batches > 0:
            try:
                # [v12] inlined above: was `from pvt_moe.utils.diagnostics import ...`
                import contextlib
                import io
                import math

                with contextlib.redirect_stdout(io.StringIO()):
                    stats = routing_stats(model, loader, num_batches=self.utilization_batches)
                util, drops = {}, {}
                for name, st in stats.items():
                    c = st["counts"].float()
                    share = c / c.sum().clamp(min=1)
                    p = share[share > 0]
                    util[name] = {"share": [round(float(v), 4) for v in share],
                                  "entropy": round(float(-(p * p.log()).sum()), 4),
                                  "max_entropy": round(math.log(c.numel()), 4),
                                  "tokens": int(c.sum())}
                    # The direct measurement: what fraction of the routed
                    # tokens capacity threw away. Separates "MoE did not help"
                    # from "the tokens never reached an expert".
                    drops[name] = {"drop_fraction": st["drop_fraction"],
                                   "dropped": st["dropped"], "routed": st["routed"],
                                   "capacity": st["capacity"],
                                   "tokens_per_forward": st["tokens_per_forward"]}
                block["expert_utilization"] = util
                block["token_drops"] = drops
                block["utilization_batches"] = self.utilization_batches
            except Exception as e:  # noqa: BLE001 — diagnostics never kill a run
                block["expert_utilization"] = {"error": f"{type(e).__name__}: {e}"[:200]}
        return block

    def on_train_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        completed = trainer.current_epoch + 1
        elapsed = time.time() - self._t_epoch if self._t_epoch else None
        metrics = self._metrics(trainer)
        try:
            n_batches = trainer.num_training_batches
            images = (n_batches * self.cfg["batch_size"]
                      if n_batches not in (None, float("inf")) else None)
        except Exception:  # noqa: BLE001
            images = None
        row = {"epoch": completed, "seconds": round(elapsed, 1) if elapsed else None,
               "images_per_second": (round(images / elapsed, 1) if images and elapsed else None),
               "peak_vram_gib": (round(torch.cuda.max_memory_allocated() / 1024 ** 3, 2)
                                 if torch.cuda.is_available() else None),
               "lr": (trainer.optimizers[0].param_groups[0]["lr"] if trainer.optimizers else None),
               **metrics}
        # One row per epoch, even across a resume that replays the same epoch.
        self.history = [h for h in self.history if h.get("epoch") != completed] + [row]
        if "val_acc" in metrics and metrics["val_acc"] >= self.best.get("val_acc", -1.0):
            self.best = {"val_acc": metrics["val_acc"], "epoch": completed,
                         "val_acc_top5": metrics.get("val_acc_top5")}
        extra = {}
        if hasattr(pl_module, "results_extra"):
            try:
                extra = pl_module.results_extra(trainer) or {}
            except Exception as e:  # noqa: BLE001
                extra = {"results_extra_error": f"{type(e).__name__}: {e}"[:200]}
        self.write(trainer, pl_module, finished=False, extra=extra)

    def on_fit_end(self, trainer, pl_module):
        if self.history:
            extra = {}
            if hasattr(pl_module, "results_extra"):
                try:
                    extra = pl_module.results_extra(trainer) or {}
                except Exception:  # noqa: BLE001
                    extra = {}
            self.write(trainer, pl_module, finished=True, extra=extra)

    # -- assembling and writing ------------------------------------------------
    def record(self, trainer, pl_module, finished: bool, extra: dict | None = None) -> dict:
        cfg = self.cfg
        last = self.history[-1] if self.history else {}
        budget = cfg["ssl"]["epochs"] if cfg.get("task") == "ssl" else cfg["epochs"]
        accuracy = {k: last.get(k) for k in ("val_acc", "val_acc_top5", "val_precision_macro",
                                             "val_recall_macro", "val_loss", "train_loss",
                                             "train_ce", "train_acc_mixed")}
        accuracy["best_val_acc"] = self.best.get("val_acc")
        accuracy["best_epoch"] = self.best.get("epoch")
        rec = {
            "schema": SCHEMA,
            "identity": run_identity(cfg),
            "status": {"epochs_completed": last.get("epoch", 0), "epoch_budget": budget,
                       "finished": finished, "updated": time.strftime("%Y-%m-%dT%H:%M:%S"),
                       "stop_at_epoch": cfg.get("stop_at_epoch")},
            "accuracy": accuracy,
            "efficiency": {
                "epoch_seconds": last.get("seconds"),
                "images_per_second": last.get("images_per_second"),
                "peak_vram_gib": last.get("peak_vram_gib"),
                "fit_seconds": (round(time.time() - self._t_fit, 1) if self._t_fit else None),
                "params": self._static.get("params"),
                "gflops": self._static.get("gflops"),
                "batch_size": cfg["batch_size"],
                "accumulate_grad_batches": cfg.get("accumulate_grad_batches"),
                "precision": cfg.get("precision"),
            },
            "moe": self._moe_block(trainer, pl_module),
            "environment": self._static.get("environment") or environment_info(),
            "history": self.history,
        }
        if cfg.get("task") == "ssl":
            rec["ssl"] = {"ssl_loss": last.get("ssl_loss"), "recon_loss": last.get("recon_loss"),
                          "mask_ratio": last.get("mask_ratio"), "target_std": last.get("target_std"),
                          "note": MIM_PROBE_NOTE}
        if extra:
            for k, v in extra.items():
                if isinstance(v, dict) and isinstance(rec.get(k), dict):
                    rec[k].update(v)
                else:
                    rec[k] = v
        # Keep anything evaluate.py merged in earlier (k-NN, probe, test split).
        existing = read_results(self.dirpath)
        if existing and isinstance(existing.get("eval"), dict):
            rec["eval"] = existing["eval"]
        return rec

    def write(self, trainer, pl_module, finished: bool, extra: dict | None = None) -> None:
        if not getattr(trainer, "is_global_zero", True):
            return
        rec = self.record(trainer, pl_module, finished, extra)
        write_results(self.dirpath, rec)


def read_results(dirpath: str) -> dict | None:
    path = os.path.join(dirpath, JSON_NAME)
    if not os.path.exists(path):
        return None
    try:
        with open(path) as fh:
            return json.load(fh)
    except (OSError, ValueError):
        return None


#: ``(config path, identity path)`` for every resolved value that CHANGES
#: TRAINING, leaves no trace in the checkpoint, and is absent from the run
#: name — so a resume could silently continue one run under two settings.
#: Each is compared against the identity block results.json recorded for the
#: run being resumed; a side that is absent or None is skipped, which is what
#: makes the MoE-only and SSL-only entries no-ops elsewhere.
#:
#: Deliberately NOT here: optim.layer_decay. Changing it between 1.0 and a
#: decay changes the optimizer's param-group COUNT, which makes
#: ``load_state_dict`` raise on its own; changing it between two decays is a
#: silent NO-OP, because the restored base_lrs win. Guarding a no-op is worse
#: than not guarding it. What a resume silently IGNORES is reported instead,
#: by ``resume_provenance``.
RESUME_IDENTITY_FIELDS = (
    ("model.drop_path_rate", "drop_path_rate"),
    ("effective_batch_size", "effective_batch_size"),
    ("model.moe.capacity_factor", "moe.capacity_factor"),
    ("model.moe.gate_noise", "moe.gate_noise"),
    ("loss.aux_weight", "moe.aux_weight"),
    ("optim.grad_clip", "optim.grad_clip"),
    ("ssl.grad_clip", "ssl.grad_clip"),
    ("ssl.mask_ratio", "ssl.mask_ratio"),
)


def _dig(node, dotted: str):
    """``_dig(cfg, "model.moe.gate_noise")`` -> the value, or None if any hop
    is missing or not a dict (an SSL field on a supervised run, say)."""
    for part in dotted.split("."):
        if not isinstance(node, dict):
            return None
        node = node.get(part)
    return node


def resume_identity_mismatches(cfg: dict, ckpt_path: str) -> list:
    """Compare this run against the results.json beside ``ckpt_path``.

    Returns a list of human-readable mismatches; empty means consistent, or
    that there is nothing to compare against (a run from before results.json
    existed, or a checkpoint moved out of its run directory — a resume is
    still allowed then, it just cannot be verified).
    """
    rec = read_results(os.path.dirname(os.path.abspath(ckpt_path))) or {}
    ident = rec.get("identity") or {}
    out = []
    for cfg_path, ident_path in RESUME_IDENTITY_FIELDS:
        have, want = _dig(ident, ident_path), _dig(cfg, cfg_path)
        if have is None or want is None or have == want:
            continue
        out.append(f"{cfg_path}: the run being resumed trained at {have}, "
                   f"this command resolves {want}")
    return out


def resume_provenance(cfg: dict, ckpt_path: str) -> list:
    """What a resume takes FROM THE CHECKPOINT rather than the command line.

    Lightning restores the optimizer's ``initial_lr`` and the scheduler's
    ``base_lrs``, so a changed ``--lr`` on a resume is silently ignored: the
    run continues on the original schedule. That wastes a run rather than
    corrupting one, which is precisely why it is easy to miss — so it is
    printed, with the two values side by side, whether or not they differ.

    Returns display lines; empty when the checkpoint cannot be read (never a
    reason to stop a resume).
    """
    import torch

    try:
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    except Exception as e:                       # noqa: BLE001 — never block a resume
        return [f"  (could not read {ckpt_path} to report it: {type(e).__name__}: {e})"]
    if not isinstance(ck, dict):
        return []

    lines = []
    epoch, step = ck.get("epoch"), ck.get("global_step")
    if epoch is not None:
        lines.append(f"  loop state: resuming after epoch {epoch} (0-based), global step {step}")

    # base_lrs is the number the scheduler rebuilds every LR from. SequentialLR
    # nests its children, which is the shape LitClassifier already handles.
    base_lrs = []
    for sched in ck.get("lr_schedulers") or []:
        if isinstance(sched, dict):
            if "_schedulers" in sched:
                for sub in sched["_schedulers"]:
                    base_lrs += list((sub or {}).get("base_lrs") or [])
            base_lrs += list(sched.get("base_lrs") or [])
    if base_lrs:
        ckpt_lr = max(base_lrs)
        asked = (cfg["ssl"]["lr"] if cfg.get("task") == "ssl" else cfg["optim"]["lr"])
        key = "ssl.lr" if cfg.get("task") == "ssl" else "optim.lr"
        same = asked is not None and abs(ckpt_lr - asked) <= 1e-12 * max(1.0, abs(asked))
        lines.append(
            f"  {key}: this command resolves {asked:.3e}, the checkpoint restores "
            f"{ckpt_lr:.3e}" + ("  (same)" if same else "  <-- the checkpoint WINS; "
                                "your value is ignored"))
    opt_states = ck.get("optimizer_states") or []
    if opt_states and isinstance(opt_states[0], dict):
        groups = opt_states[0].get("param_groups") or []
        wds = {g.get("weight_decay") for g in groups if isinstance(g, dict)}
        wds.discard(None)
        asked_wd = (cfg["ssl"] if cfg.get("task") == "ssl" else cfg["optim"]).get("weight_decay")
        if wds and asked_wd is not None and asked_wd not in wds:
            lines.append(f"  weight_decay: this command resolves {asked_wd}, the checkpoint "
                         f"restores {sorted(wds)}  <-- the checkpoint WINS")
        lines.append(f"  optimizer: {len(groups)} param group(s) restored "
                     f"(momentum/variance included)")
    return lines


def assert_resume_identity(cfg: dict) -> list:
    """Raise if a resume would silently change one of those fields.

    Called by ``build_trainer`` / ``build_ssl_trainer`` (so the CLI and the
    notebooks are both covered) before any compute. Same shape as the
    ``ssl_init`` architecture guard: it names the fix and can be switched off
    with ``model.resume_check_identity: false`` when the change is deliberate.
    """
    if cfg.get("mode") != "resume" or not cfg.get("ckpt_path"):
        return []
    problems = resume_identity_mismatches(cfg, cfg["ckpt_path"])
    provenance = resume_provenance(cfg, cfg["ckpt_path"])
    if provenance:
        print(f"[resume] {cfg['ckpt_path']} — what comes from the checkpoint, "
              f"not the command line:")
        for line in provenance:
            print(line)
    if problems and cfg["model"].get("resume_check_identity", True):
        text = "\n  - ".join(problems)
        raise ValueError(
            f"resuming {cfg['ckpt_path']} would change settings the checkpoint cannot "
            f"carry:\n  - {text}\nHalf the run would train at each value. Pass the "
            f"recorded value explicitly (e.g. --drop-path <recorded>), or set "
            f"model.resume_check_identity: false if the change is deliberate."
        )
    return problems


def write_results(dirpath: str, rec: dict) -> str:
    """Atomically write results.json and the markdown rendering next to it."""
    os.makedirs(dirpath, exist_ok=True)
    path = os.path.join(dirpath, JSON_NAME)
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(rec, fh, indent=2, sort_keys=False, default=str)
    os.replace(tmp, path)
    with open(os.path.join(dirpath, MD_NAME), "w") as fh:
        fh.write(render_markdown(rec))
    return path


def _fmt(v, pct=False, nd=2):
    if v is None:
        return "—"
    if isinstance(v, float):
        return f"{100 * v:.{nd}f}%" if pct else f"{v:.{nd}f}"
    return str(v)


def render_markdown(rec: dict) -> str:
    ident, acc, eff = rec["identity"], rec["accuracy"], rec["efficiency"]
    st = rec["status"]
    lines = [f"# {ident['run_name']}", ""]
    lines.append(f"**chain**: {' -> '.join(ident['chain']) or '(this run only)'}  ")
    lines.append(f"**variant** {ident['variant']} | **task** {ident['task']} | **recipe** "
                 f"{ident['recipe']} | **mode** {ident['mode']} | **dataset** {ident['dataset']} "
                 f"@ {ident['img_size']}px | **seed** {ident['seed']} | **commit** "
                 f"{ident.get('git_commit') or '?'} | **config** {ident['config_sha1']}  ")
    lines.append(f"**status**: {st['epochs_completed']} / {st['epoch_budget']} epochs"
                 f"{' (finished)' if st['finished'] else ''}, updated {st['updated']}")
    lines += ["", "## Accuracy (validation split)", "",
              "| latest top-1 | latest top-5 | best top-1 (epoch) | prec. macro | rec. macro | val loss | train loss |",
              "|---|---|---|---|---|---|---|",
              f"| {_fmt(acc.get('val_acc'), pct=True)} | {_fmt(acc.get('val_acc_top5'), pct=True)} | "
              f"{_fmt(acc.get('best_val_acc'), pct=True)} ({_fmt(acc.get('best_epoch'))}) | "
              f"{_fmt(acc.get('val_precision_macro'), pct=True)} | "
              f"{_fmt(acc.get('val_recall_macro'), pct=True)} | {_fmt(acc.get('val_loss'), nd=4)} | "
              f"{_fmt(acc.get('train_loss'), nd=4)} |"]
    if rec.get("ssl"):
        s = rec["ssl"]
        lines += ["", "## SSL pretraining", "",
                  f"ssl_loss {_fmt(s.get('ssl_loss'), nd=4)} | recon {_fmt(s.get('recon_loss'), nd=4)} | "
                  f"mask ratio {_fmt(s.get('mask_ratio'), nd=3)} measured"
                  + (f" / {_fmt(s.get('mask_ratio_configured'), nd=3)} configured"
                     if s.get("mask_ratio_configured") is not None else "")
                  + (f" | target_std {_fmt(s.get('target_std'), nd=3)}" if s.get("target_std") is not None else ""),
                  "", f"> {s.get('note', MIM_PROBE_NOTE)}"]
    params, gfl = eff.get("params") or {}, eff.get("gflops") or {}
    lines += ["", "## Measured efficiency", "",
              "| s / epoch | images / s | peak VRAM (GiB) | params (M) | GFLOPs | batch | precision |",
              "|---|---|---|---|---|---|---|",
              f"| {_fmt(eff.get('epoch_seconds'), nd=1)} | {_fmt(eff.get('images_per_second'), nd=1)} | "
              f"{_fmt(eff.get('peak_vram_gib'))} | {_fmt(params.get('total_m'), nd=1)} | "
              f"{_fmt(gfl.get('total_gflops'))} | {eff.get('batch_size')} x "
              f"{eff.get('accumulate_grad_batches')} | {eff.get('precision')} |"]
    moe = rec.get("moe")
    if moe:
        lines += ["", "## MoE", "", f"aux weight {moe.get('aux_weight')} | train_aux "
                  f"{_fmt(acc.get('train_aux') if 'train_aux' in acc else (rec['history'][-1].get('train_aux') if rec.get('history') else None), nd=4)}"]
        lines[-1] += (f" | capacity_factor {moe.get('capacity_factor')} "
                      f"| gate_noise {moe.get('gate_noise')}")
        routing = moe.get("routing") or {}
        if routing:
            lines += ["", "Training-token routing (RoutingMonitor) — the metrics `train_aux` "
                      "cannot show:", "",
                      "| block | token share per expert | drop rate | imbalance | H(route) | H(gate) / max |",
                      "|---|---|---|---|---|---|"]
            for name, r in routing.items():
                drop = f"{r['drop_rate']:.1%}"
                if "drop_rate_realised" in r:
                    drop += f" ({r['drop_rate_realised']:.1%} realised)"
                lines.append(f"| {name} | {[round(v, 3) for v in r['share']]} | {drop} | "
                             f"{r['imbalance']:.3f} | {r['route_entropy']:.2f} | "
                             f"{r['gate_entropy']:.2f} / {r['max_entropy']:.2f} |")
            lines += ["", f"> {moe.get('aux_note', AUX_NOTE)}"]
        util = moe.get("expert_utilization") or {}
        drops = moe.get("token_drops") or {}
        if util and "error" not in util:
            lines += ["", "Validation-batch utilisation (`expert_utilization`, eval mode, "
                      "noiseless logits — a different population from the row above):", "",
                      "| block | token share per expert | entropy / max | tokens dropped |",
                      "|---|---|---|---|"]
            for name, u in util.items():
                d = drops.get(name) or {}
                cap = d.get("capacity")
                dropped = ("n/a (no cap)" if cap is None and d
                           else f"{d['drop_fraction'] * 100:.2f}% ({d['dropped']}/{d['routed']}, "
                                f"cap {cap}/expert per fwd)" if d else "not measured")
                lines.append(f"| {name} | {u['share']} | "
                             f"{u['entropy']:.2f} / {u['max_entropy']:.2f} | {dropped} |")
    mr = rec.get("mask_routing")
    if mr and "error" not in mr:
        lines += ["", "## Mask-token vs visible-token routing (MoE pretraining)", "",
                  "| block | masked share | visible share | H masked / visible / max | concentration | gap |",
                  "|---|---|---|---|---|---|"]
        for name, s in mr.items():
            lines.append(f"| {name} | {s['masked_share']} | {s['visible_share']} | "
                         f"{s['masked_entropy']:.2f} / {s['visible_entropy']:.2f} / {s['max_entropy']:.2f} | "
                         f"{s['mask_token_concentration']:.2f} | {s['share_gap']:.2f} |")
        lines.append("")
        lines.append("> The load-balancing loss counts the masked positions: every token of the "
                     "stage is routed, so ~60% of what it balances is mask-token derived.")
    ev = rec.get("eval")
    if ev:
        lines += ["", "## Evaluation (evaluate.py)", "", "```json", json.dumps(ev, indent=2), "```"]
    env = rec.get("environment") or {}
    lines += ["", "## Environment", "",
              f"torch {env.get('torch')} | CUDA {env.get('cuda')} | lightning {env.get('lightning')} | "
              f"python {env.get('python')} | GPU {env.get('gpu')} | {env.get('platform')}", ""]
    return "\n".join(lines)


In [ ]:
# ═══ pvt_moe/engine/classifier.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""LightningModule for supervised ImageNet training.

Carries the v9 lineage's load-bearing training semantics:

- **Aux loss handling**: model returns ``(logits, aux)`` with aux already
  averaged over MoE blocks; here it is clamped (spike guard), weighted by
  ``loss.aux_weight``, added to the CE loss, and dropped for the step if the
  total goes NaN/Inf.
- **Tutel gate train-forcing**: Tutel gate modules revert themselves to eval
  mode after Lightning's validation pass, silently disabling ``gate_noise``
  (and with it the exploration that keeps experts balanced). ``train()`` is
  overridden and ``on_train_epoch_start`` re-forces every gate. Do not
  remove. (MegaBlocks needs none of this — plain ``self.training`` gates its
  loss registry.)
- **Discriminative LR + weight-decay hygiene**: 4 parameter groups —
  {stages 1-3, stage 4 + head} x {decay, no-decay}, where the no-decay split
  is the timm rule (``p.ndim <= 1``: biases and all norm weights).
- **Layer-wise LR decay** (``optim.layer_decay`` < 1, the ssl_finetune /
  downstream recipes): every block's LR is the peak scaled by
  ``decay ** (top - layer_id)`` compounding from the head down, the scheme of
  BEiT / SimMIM fine-tuning (``microsoft/SimMIM optimizer.py
  get_swin_layer``, mapped onto PVT v2's attribute names in
  ``layer_id_of``). ``1.0`` keeps the 4-group layout above byte-for-byte.
- **Chain provenance**: a warm start prepends the parent checkpoint's
  ``chain`` (an SSL run's stage tag) or the HF id to ``cfg["chain"]`` BEFORE
  the hyperparameters are saved, so the checkpoint and results.json name the
  whole path that produced the run.

The mixup pipeline follows DeiT/Swin practice: timm ``Mixup`` (mixup+cutmix,
label smoothing inside) with ``SoftTargetCrossEntropy`` for training and
plain CE on hard labels for validation. ``train_acc_mixed`` is measured
against ``argmax`` of the soft targets — a proxy that reads low; use
``val_acc`` for reporting.
"""

from __future__ import annotations

import gc
import re

import pytorch_lightning as pl
import torch
import torch.nn as nn
from torchmetrics import MetricCollection
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassPrecision,
    MulticlassRecall,
)

# [v12] inlined above: was `from pvt_moe.models.ffn import ...`
# [v12] inlined above: was `from pvt_moe.models.pretrained import ...`
# [v12] inlined above: was `from pvt_moe.models.pvt import ...`


class LitClassifier(pl.LightningModule):
    """Supervised classifier around ``PyramidVisionTransformerV2``."""

    def __init__(self, cfg: dict):
        super().__init__()
        self.cfg = cfg
        self.model = build_model(cfg)
        self._apply_warm_start()          # may prepend the parent's chain
        # cfg is JSON-safe by construction (validate_config enforces it), so
        # it can be checkpointed / logged verbatim — saved AFTER the warm
        # start so the stored copy carries the full provenance chain.
        self.save_hyperparameters({"cfg": cfg})

        self._uses_tutel_moe = (
            cfg["model"]["ablation"]["use_moe"]
            and cfg["model"]["moe"]["backend"] == "tutel"
            and any(cfg["model"]["ablation"]["moe_placement"])
        )

        num_classes = cfg["dataset"]["num_classes"]
        loss_cfg = cfg["loss"]
        self.aux_weight = loss_cfg["aux_weight"]
        self.aux_clamp = loss_cfg["aux_clamp"]

        from timm.data import Mixup
        from timm.loss import SoftTargetCrossEntropy

        self.mixup_fn = Mixup(
            mixup_alpha=loss_cfg["mixup_alpha"],
            cutmix_alpha=loss_cfg["cutmix_alpha"],
            prob=loss_cfg["mixup_prob"],
            switch_prob=loss_cfg["mixup_switch_prob"],
            mode="batch",
            label_smoothing=loss_cfg["label_smoothing"],
            num_classes=num_classes,
        )
        self.train_loss_fn = SoftTargetCrossEntropy()
        self.val_loss_fn = nn.CrossEntropyLoss()

        # ImageNet convention: micro (= overall) top-1 / top-5 accuracy.
        def _acc(top_k: int):
            return MulticlassAccuracy(num_classes=num_classes, top_k=top_k, average="micro")

        # Macro precision/recall (v9 lineage) surface per-class collapse that
        # micro accuracy hides — an MoE that serves the head classes well and
        # starves the tail reads fine on top-1 and badly here.
        #
        # Deliberately NOT on the train split: training metrics are computed
        # against argmax of MIXUP'd soft targets, where per-class precision is
        # noise. Use the val numbers.
        self.train_metrics = MetricCollection({"acc_mixed": _acc(1)}, prefix="train_")
        self.val_metrics = MetricCollection(
            {
                "acc": _acc(1),
                "acc_top5": _acc(5),
                "precision_macro": MulticlassPrecision(
                    num_classes=num_classes, average="macro"),
                "recall_macro": MulticlassRecall(
                    num_classes=num_classes, average="macro"),
            },
            prefix="val_",
        )
        self.test_metrics = self.val_metrics.clone(prefix="test_")

        # A 21841^2 confusion matrix is neither computable nor plottable —
        # gate it to the 1k dataset.
        self.val_confmat = None
        if num_classes <= 1000:
            from torchmetrics.classification import MulticlassConfusionMatrix

            self.val_confmat = MulticlassConfusionMatrix(
                num_classes=num_classes, normalize="true"
            )

    # -- warm start -----------------------------------------------------------

    def _apply_warm_start(self):
        cfg = self.cfg
        mode = cfg["mode"]
        if mode == "hf_pretrained" and cfg["model"]["pretrained_hf_id"]:
            load_hf_pretrained(
                self.model,
                cfg["model"]["pretrained_hf_id"],
                seed_moe_experts=cfg["model"]["seed_moe_from_dense"],
                upcycle_init=cfg["model"]["moe"].get("upcycle_init", "none"),
            )
            self._extend_chain([f"hf_pretrained@{cfg['model']['pretrained_hf_id']}"])
        elif mode == "ssl_init":
            stats = load_backbone_checkpoint(
                self.model, cfg["ckpt_path"], skip_head=True, expected_cfg=cfg,
                check_arch=cfg["model"].get("ssl_init_check_arch", True),
                seed_moe_experts=cfg["model"]["seed_moe_from_dense"],
                upcycle_init=cfg["model"]["moe"].get("upcycle_init", "none"))
            self._extend_chain(stats.get("parent_chain") or [])
        elif mode == "resume":
            # Lightning restores the full state in trainer.fit(ckpt_path=...);
            # only the provenance is read here, so a resumed run keeps the
            # chain its earlier epochs recorded.
            try:
                parent = _checkpoint_cfg(torch.load(cfg["ckpt_path"], map_location="cpu",
                                                    weights_only=False)) or {}
            except Exception as e:  # the fit itself will report a bad file
                print(f"[chain] could not read the resume checkpoint's config: {e}")
                parent = {}
            saved = list(parent.get("chain") or [])
            own = self.cfg["chain"][-1] if self.cfg.get("chain") else None
            if saved and saved[-1] == own:
                self.cfg["chain"] = saved          # the same stage, continued
                print("[chain] " + " -> ".join(self.cfg["chain"]))
            else:
                self._extend_chain(saved)
        # mode == "scratch": nothing to load and nothing before this stage.

        n_frozen = cfg["model"]["num_frozen_stages"]
        if n_frozen > 0:
            self.model.freeze_stages(n_frozen)
            print(f"[freeze] Stages 1..{n_frozen} frozen")

    def _extend_chain(self, parent: list) -> None:
        """Prepend the parent stage(s) to ``cfg["chain"]`` unless already there."""
        chain = list(self.cfg.get("chain") or [])
        parent = list(parent)
        if parent and chain[: len(parent)] != parent:
            chain = parent + chain
        self.cfg["chain"] = chain
        print("[chain] " + " -> ".join(chain))

    # -- tutel gate forcing (load-bearing) -------------------------------------

    def _force_tutel_gates_train(self):
        force_tutel_gates_train(self.model)

    def train(self, mode: bool = True):
        super().train(mode)
        if mode and self._uses_tutel_moe:
            self._force_tutel_gates_train()
        return self

    def on_train_epoch_start(self):
        self.model.train()
        if self._uses_tutel_moe:
            self._force_tutel_gates_train()
        n_frozen = self.cfg["model"]["num_frozen_stages"]
        if n_frozen > 0:
            self.model.freeze_stages(n_frozen)  # re-freeze after .train()

    def on_train_epoch_end(self):
        # Free fragmented CUDA memory between epochs (large-batch runs OOM'd
        # at scheduler transitions without this).
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # -- optimization -----------------------------------------------------------

    def layer_id_of(self, name: str) -> int:
        """Layer id of a backbone parameter for layer-wise LR decay.

        The SimMIM / BEiT scheme (``microsoft/SimMIM optimizer.py``,
        ``get_swin_layer``) on PVT v2's names: the stage-1 patch embed is
        layer 0; block ``j`` of stage ``i`` is ``1 + sum(depths[:i-1]) + j``;
        a later patch embed and a stage's output norm take the id of the last
        block BEFORE them (Swin's downsample rule); the final norm and the
        head sit on top (``sum(depths) + 1``), where the scale is 1.
        """
        depths = self.model.depths
        top = sum(depths) + 1
        m = re.match(r"block(\d+)\.(\d+)\.", name)
        if m:
            stage, j = int(m.group(1)), int(m.group(2))
            return 1 + sum(depths[: stage - 1]) + j
        m = re.match(r"patch_embed(\d+)\.", name)
        if m:
            stage = int(m.group(1))
            return 0 if stage == 1 else sum(depths[: stage - 1])
        m = re.match(r"norm(\d+)\.", name)
        if m:
            stage = int(m.group(1))
            return top if stage == self.model.num_stages else sum(depths[:stage])
        return top                                    # head, anything else

    def configure_optimizers(self):
        opt_cfg = self.cfg["optim"]
        base_lr = opt_cfg["lr"]
        mult = opt_cfg["stage4_lr_multiplier"]
        wd = opt_cfg["weight_decay"]
        layer_decay = opt_cfg.get("layer_decay") or 1.0

        last = self.model.num_stages
        stage4_ids = set()
        for attr in (f"patch_embed{last}", f"block{last}", f"norm{last}", "head"):
            module = getattr(self.model, attr, None)
            if module is not None:
                stage4_ids.update(id(p) for p in module.parameters())

        no_decay_names = self.model.no_weight_decay()   # RoPE-Mixed freqs
        if layer_decay == 1.0:
            groups = {"s123_decay": [], "s123_nodecay": [], "s4_decay": [], "s4_nodecay": []}
            for name, p in self.model.named_parameters():
                if not p.requires_grad:
                    continue
                part = "s4" if id(p) in stage4_ids else "s123"
                # biases + norm weights (timm rule) plus the model's own list.
                nodecay = p.ndim <= 1 or name in no_decay_names
                groups[f"{part}_{'nodecay' if nodecay else 'decay'}"].append(p)

            param_groups = [
                {"params": groups["s123_decay"], "lr": base_lr, "weight_decay": wd,
                 "name": "stages123_decay"},
                {"params": groups["s123_nodecay"], "lr": base_lr, "weight_decay": 0.0,
                 "name": "stages123_nodecay"},
                {"params": groups["s4_decay"], "lr": base_lr * mult, "weight_decay": wd,
                 "name": "stage4_decay"},
                {"params": groups["s4_nodecay"], "lr": base_lr * mult, "weight_decay": 0.0,
                 "name": "stage4_nodecay"},
            ]
            param_groups = [g for g in param_groups if g["params"]]
            for g in param_groups:
                print(f"[optimizer] {g['name']}: {len(g['params'])} tensors "
                      f"@ lr={g['lr']:.2e} wd={g['weight_decay']}")
        else:
            top = sum(self.model.depths) + 1
            by_key = {}
            for name, p in self.model.named_parameters():
                if not p.requires_grad:
                    continue
                lid = self.layer_id_of(name)
                part = "s4" if id(p) in stage4_ids else "s123"
                nodecay = p.ndim <= 1 or name in no_decay_names
                key = (lid, part, nodecay)
                if key not in by_key:
                    scale = layer_decay ** (top - lid)
                    by_key[key] = {
                        "params": [], "lr_scale": scale,
                        "lr": base_lr * scale * (mult if part == "s4" else 1.0),
                        "weight_decay": 0.0 if nodecay else wd,
                        "name": f"layer{lid:02d}_{part}_{'nodecay' if nodecay else 'decay'}",
                    }
                by_key[key]["params"].append(p)
            param_groups = [by_key[k] for k in sorted(by_key)]
            lrs = [g["lr"] for g in param_groups]
            print(f"[optimizer] layer_decay {layer_decay}: {top + 1} layer ids over "
                  f"{sum(self.model.depths)} blocks, {len(param_groups)} groups | "
                  f"lr {max(lrs):.2e} (head / final norm, scale 1) .. {min(lrs):.2e} "
                  f"(stage-1 patch embed, scale {layer_decay ** top:.3f}) | wd {wd}")

        optimizer = torch.optim.AdamW(param_groups, betas=tuple(opt_cfg["betas"]))

        warmup = opt_cfg["warmup_epochs"]
        cosine_epochs = max(1, self.cfg["epochs"] - warmup)
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cosine_epochs, eta_min=opt_cfg["eta_min"]
        )
        if warmup > 0:
            linear = torch.optim.lr_scheduler.LinearLR(
                optimizer, start_factor=opt_cfg["warmup_start_factor"], total_iters=warmup
            )
            scheduler = torch.optim.lr_scheduler.SequentialLR(
                optimizer, [linear, cosine], milestones=[warmup]
            )
        else:
            scheduler = cosine

        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "epoch", "name": "lr"},
        }

    def on_load_checkpoint(self, checkpoint):
        """When extending a run (larger cfg.epochs), patch the cosine T_max
        stored in the checkpoint so the resumed schedule matches."""
        warmup = self.cfg["optim"]["warmup_epochs"]
        new_t_max = max(1, self.cfg["epochs"] - warmup)
        for state in checkpoint.get("lr_schedulers", []):
            if "_schedulers" in state:  # SequentialLR
                for sub in state["_schedulers"]:
                    if "T_max" in sub:
                        sub["T_max"] = new_t_max
            elif "T_max" in state:
                state["T_max"] = new_t_max

    # -- steps ------------------------------------------------------------------

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        x, y_soft = self.mixup_fn(x, y)

        logits, aux = self.model(x)
        ce_loss = self.train_loss_fn(logits, y_soft)

        if aux is not None:
            aux = torch.clamp(aux, max=self.aux_clamp)  # load-balancing spike guard
            loss = ce_loss + self.aux_weight * aux
            if torch.isnan(loss) or torch.isinf(loss):
                loss = ce_loss  # drop aux for this step rather than poison the run
        else:
            loss = ce_loss
            aux = torch.zeros((), device=logits.device)

        self.log("train_loss_step", loss, on_step=True, on_epoch=False)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_ce", ce_loss, on_step=False, on_epoch=True)
        self.log("train_aux", aux, on_step=False, on_epoch=True, prog_bar=True)

        self.train_metrics(logits, y_soft.argmax(dim=1))
        self.log_dict(self.train_metrics, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_validation_epoch_start(self):
        # Manually-updated metrics are NOT auto-reset by Lightning (only
        # self.log-routed ones are). Without this, the "final" confusion
        # matrix would aggregate every epoch of the run + the sanity batches.
        if self.val_confmat is not None:
            self.val_confmat.reset()

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits, _ = self.model(x)
        self.log("val_loss", self.val_loss_fn(logits, y), on_epoch=True, prog_bar=True)
        self.val_metrics(logits, y)
        self.log_dict(self.val_metrics, on_epoch=True, prog_bar=True)
        if self.val_confmat is not None:
            self.val_confmat.update(logits, y)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits, _ = self.model(x)
        self.log("test_loss", self.val_loss_fn(logits, y), on_epoch=True)
        self.test_metrics(logits, y)
        self.log_dict(self.test_metrics, on_epoch=True)

    def on_train_end(self):
        """Log the final row-normalized confusion matrix to W&B (1k only)."""
        if self.val_confmat is None:
            return
        if not getattr(self.val_confmat, "_update_called", getattr(self.val_confmat, "update_called", True)):
            return
        try:
            import matplotlib.pyplot as plt
            import wandb

            cm = self.val_confmat.compute().cpu().numpy()
            fig, ax = plt.subplots(figsize=(12, 12))
            ax.imshow(cm, cmap="Blues", aspect="auto")
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
            ax.set_title(f"Val confusion matrix (epoch {self.current_epoch})")
            for logger in self.loggers or []:
                exp = getattr(logger, "experiment", None)
                if exp is not None and isinstance(exp, wandb.sdk.wandb_run.Run):
                    exp.log({"val/confmat_final": wandb.Image(fig)})
                    break
            plt.close(fig)
        except Exception as e:  # diagnostics must never kill a finished run
            print(f"[confmat] skipped: {e}")


## Δ since v10 — trainer, checkpoints, diagnostics

**Kept from v10:** milestone checkpoints (never pruned) + `stop_at_epoch`;
gradient accumulation applied by the Trainer (clipping once per optimizer
step); the `/`-free checkpoint filename; the per-epoch console line.

**Changed / new:**
<pre>
<del>trainer.fit(model, ..., weights_only=True)      # TypeError: not a fit() arg (v10 removed it)</del>
<del># v10: last.ckpt only via Lightning's save_last</del>
RollingCheckpoint     # last.ckpt rewritten every epoch, resume-safe
ModelCheckpoint       # top-2 by val_acc: epochNNN-valaccX.ckpt
RopeFreqSnapshot      # rope_freqs_init.pt (TRUE step-0, survives resumes) / _final.pt
RoutingMonitor        # per-block [routing] line: share, drop_rate (+realised),
                      # imbalance, route/gate entropies — because train_aux
                      # reads 1.0000 in a wide blind spot (see diagnostics cell)
ResultsWriter         # results.json + results.md rewritten EVERY epoch;
                      # tools/compare_runs.py aggregates them across runs
LearningRateMonitor   # the LR is also in every checkpoint (probe_checkpoint)
deterministic="warn"  # True would crash mid-run on nondeterministic ops
</pre>

`assert_resume_identity` refuses a resume whose config silently disagrees with
the run being resumed (drop_path, capacity_factor, gate_noise, effective
batch, grad clip, mask ratio — the fields that leave no trace in the
checkpoint), and `resume_provenance` prints what the checkpoint *overrides*
(a changed `--lr` on resume is ignored: the schedule's base_lrs win).


In [ ]:
# ═══ pvt_moe/engine/callbacks.py — inlined VERBATIM by tools/make_v12_notebook.py @ 7a720d7 ═══
# Edit the package file and regenerate; do not edit here.
"""Callbacks, loggers, and the Trainer factory."""

from __future__ import annotations

import os
import time

import pytorch_lightning as pl
import torch
from pytorch_lightning.callbacks import Checkpoint, LearningRateMonitor, ModelCheckpoint

# [v12] inlined above: was `from pvt_moe.engine.results import ...`


class MilestoneCheckpoint(pl.Callback):
    """Write a permanent full-state checkpoint at given epoch counts.

    Distinct from ``ModelCheckpoint`` in two ways that matter for a long run
    split across machines:

    - it is keyed on the EPOCH COUNT, not on a monitored metric, so the file
      you get back is the one you asked for;
    - it is never pruned by ``save_top_k``, so an epoch-90 snapshot survives
      another 200 epochs of better validation scores.

    The file holds model + optimizer + scheduler + epoch (Lightning's full
    checkpoint), so ``trainer.fit(..., ckpt_path=...)`` resumes exactly where
    it stopped, with the schedule still tied to the ORIGINAL epoch budget.

    Milestones count COMPLETED epochs: milestone 90 fires once the 90th epoch
    has finished, and the file is named ``milestone-epoch090.ckpt``.
    """

    def __init__(self, milestones, dirpath: str):
        self.milestones = sorted(set(milestones or []))
        self.dirpath = dirpath
        self.written = []

    @staticmethod
    def filename(epochs_completed: int) -> str:
        return f"milestone-epoch{epochs_completed:03d}.ckpt"

    def on_train_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        completed = trainer.current_epoch + 1  # current_epoch is 0-based
        if completed not in self.milestones:
            return
        path = os.path.join(self.dirpath, self.filename(completed))
        os.makedirs(self.dirpath, exist_ok=True)
        trainer.save_checkpoint(path)
        self.written.append(path)
        print(f"[milestone] epoch {completed}: saved full state -> {path}")


class RollingCheckpoint(Checkpoint):
    """Overwrite ``last.ckpt`` with the full training state after EVERY epoch.

    This is the file a killed run resumes from, so it must always hold the
    most recent completed epoch. ``ModelCheckpoint(save_last=True)`` does not
    guarantee that: Lightning (2.6) refreshes ``last.ckpt`` only in a step
    that also wrote a top-k file, so on any epoch whose ``val_acc`` does not
    enter the top-k — most epochs of a long run — ``last.ckpt`` is left at the
    last improvement, and a resume from it silently replays the epochs since.

    Subclasses ``Checkpoint`` (not ``Callback``) so Lightning runs it in the
    checkpoint pass, after ``ModelCheckpoint``: the saved state then already
    carries this epoch's top-k bookkeeping. Keep ``ModelCheckpoint`` first in
    the callback list so ``trainer.checkpoint_callback`` stays the val_acc one.
    """

    FILENAME = "last.ckpt"

    def __init__(self, dirpath: str):
        self.dirpath = dirpath

    def on_train_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        os.makedirs(self.dirpath, exist_ok=True)
        trainer.save_checkpoint(os.path.join(self.dirpath, self.FILENAME))


class RopeFreqSnapshot(pl.Callback):
    """Save the learnable RoPE-Mixed frequencies at step 0 and as they train.

    ``rope_freqs_init.pt`` is what the drift plot (tools/plot_rope_freqs.py)
    overlays the trained values on; without it the "spread init -> cluster"
    finding cannot be made. Keys are the model's parameter names
    (``block4.1.attn.rope.freqs``), values ``(2, heads, head_dim//2)`` fp32
    CPU tensors.

    The step-0 values are captured ONCE, the first time a run starts without
    a checkpoint, and kept in this callback's state, which Lightning stores
    inside every checkpoint (``ckpt["callbacks"]``). A resumed run, in the
    same directory or on another machine, therefore rewrites the init file
    from that state — never from the restored (already trained) weights,
    which is what a naive "save on fit start" would do, because Lightning
    restores the checkpoint before ``on_fit_start`` fires.

    ``rope_freqs_final.pt`` is refreshed after every epoch (so a killed run
    still leaves the latest values; ``last.ckpt`` carries them as well).
    """

    INIT = "rope_freqs_init.pt"
    FINAL = "rope_freqs_final.pt"

    def __init__(self, dirpath: str):
        self.dirpath = dirpath
        self._init: dict | None = None        # captured step-0 frequencies

    # -- persisted with every checkpoint --------------------------------------
    def state_dict(self) -> dict:
        return {"init": self._init}

    def load_state_dict(self, state_dict: dict) -> None:
        self._init = state_dict.get("init")

    @staticmethod
    def _freqs(pl_module) -> dict:
        model = getattr(pl_module, "model", pl_module)
        return {n: p.detach().float().cpu().clone()
                for n, p in model.named_parameters() if n.endswith("rope.freqs")}

    def _write(self, freqs: dict, name: str, overwrite: bool = True) -> None:
        if not freqs:
            return
        os.makedirs(self.dirpath, exist_ok=True)
        path = os.path.join(self.dirpath, name)
        if os.path.exists(path) and not overwrite:
            return                                # same-directory resume
        torch.save(freqs, path)
        print(f"[rope] saved {len(freqs)} frequency tensor(s) -> {path}")

    def on_fit_start(self, trainer, pl_module):
        if not self._freqs(pl_module):
            return                                # axial / no RoPE: nothing to track
        if trainer.ckpt_path is None:
            # A fresh run: these ARE the step-0 values (warm starts happen in
            # LitClassifier.__init__, before fit).
            if self._init is None:
                self._init = self._freqs(pl_module)
        elif self._init is None:
            # Resumed from a checkpoint written before this state existed:
            # the restored weights are trained, so no init file can be made.
            print("[rope] WARNING: checkpoint carries no step-0 frequencies; "
                  f"{self.INIT} is not written (the restored values are trained).")
            return
        if trainer.is_global_zero:
            self._write(self._init, self.INIT, overwrite=False)

    def on_train_epoch_end(self, trainer, pl_module):
        if trainer.is_global_zero and not trainer.sanity_checking:
            self._write(self._freqs(pl_module), self.FINAL)

    def on_fit_end(self, trainer, pl_module):
        if trainer.is_global_zero:
            self._write(self._freqs(pl_module), self.FINAL)


class RoutingMonitor(pl.Callback):
    """Per-epoch routing statistics for every MoE block, measured during TRAINING.

    This exists because ``train_aux`` cannot do the job. Both backends compute
    ``aux = E * sum_i f_i * p_i`` (token share x mean gate probability), which
    is identically ``1 + E * <f - 1/E, p - 1/E>``: it reads 1.0 whenever the
    mean gate probability is uniform, however skewed the assignment is, and the
    loss's own gradient pushes the probabilities toward uniform. A router that
    sends 96% of its tokens to one expert can log ``train_aux = 1.0000``. See
    ``pvt_moe.utils.diagnostics`` for the derivation and the worked case.

    What this logs instead, per epoch, averaged over every training token:

    ``train_drop_rate``      fraction of tokens over capacity — they receive
                             NOTHING from the routed branch. At
                             ``capacity_factor 1.0`` this is the total-variation
                             distance from uniform routing: 0 when balanced,
                             ``1 - 1/E`` at full collapse. Linear, no floor.
    ``train_moe_imbalance``  the same quantity computed from the shares alone
                             (identical at cf 1.0; they part company otherwise).
    ``train_route_entropy``  entropy of the token share, max ``log E``.
    ``train_gate_entropy``   mean per-token entropy of the gate softmax. Near
                             ``log E`` means the router is undecided — exactly
                             the regime where ``train_aux`` is pinned at 1.

    Cost: one ``(tokens, dim) x (dim, E)`` matmul per MoE block per step under
    ``no_grad``, with the counters kept on-device and synced ONCE per epoch.
    The share and the entropies are recomputed from the gate logits WITHOUT the
    backend's gate noise, so they describe the router's POLICY. The drop rate is
    reported both ways: ``train_drop_rate`` is the policy figure, and
    ``train_drop_rate_realised`` is what the layer actually did once the noise
    was added — read from ``NativeMoEFFN._dropped`` or, on Tutel, from
    ``moe_layer.dispatch_count`` (the per-expert pre-capacity counts Tutel
    stores on every forward). The two differ by however much the noise moves
    tokens across the capacity line. Both are accumulated on-device and synced
    once per epoch; the realised figure is simply absent for a backend that
    exposes neither.

    A failure inside the hook disables the monitor for the rest of the run with
    a warning: a diagnostic must never take a training run down with it.
    """

    def __init__(self, cfg: dict):
        self.cfg = cfg
        moe = cfg["model"]["moe"]
        self.capacity_factor = moe["capacity_factor"]
        self.top_k = moe["top_k"]
        self.dropless = moe["backend"] == "megablocks"   # dMoE never drops
        self.last_stats: dict = {}
        self._acc: dict = {}
        self._handles: list = []
        self._active = False
        self._failed = False

    # -- accumulation ---------------------------------------------------------

    def _hook(self, name: str, module):
        @torch.no_grad()
        def hook(mod, args):
            if not self._active or self._failed:
                return
            try:
                # Imported per call (a sys.modules lookup next to a matmul) so a
                # test can substitute it, and so an import error here cannot
                # take the run down at setup.
                # [v12] inlined above: was `from pvt_moe.utils.diagnostics import ...`

                x = args[0]
                flat = x.reshape(-1, x.shape[-1])
                logits = gate_logits(mod, flat).float()
                tokens, experts = logits.shape
                counts = torch.bincount(logits.argmax(dim=-1), minlength=experts)
                probs = logits.softmax(dim=-1)
                acc = self._acc.setdefault(name, {
                    "counts": torch.zeros(experts, dtype=torch.double, device=logits.device),
                    "prob_sum": torch.zeros(experts, dtype=torch.double, device=logits.device),
                    "entropy_sum": torch.zeros((), dtype=torch.double, device=logits.device),
                    "dropped": torch.zeros((), dtype=torch.double, device=logits.device),
                    "tokens": 0,
                })
                acc["counts"] += counts.double()
                acc["prob_sum"] += probs.sum(dim=0).double()
                acc["entropy_sum"] += -(probs.clamp_min(1e-12).log() * probs).sum().double()
                if not self.dropless:
                    cap = capacity_of(tokens, experts, self.capacity_factor, self.top_k)
                    acc["dropped"] += (counts - cap).clamp(min=0).sum().double()
                acc["tokens"] += int(tokens)
            except Exception as e:  # noqa: BLE001 — never kill a run for a diagnostic
                self._failed = True
                print(f"[routing] monitor disabled after an error in {name}: "
                      f"{type(e).__name__}: {e}")

        return hook

    def _post_hook(self, name: str):
        """Realised overflow, after the layer has actually routed."""
        # [v12] inlined above: was `from pvt_moe.utils.diagnostics import ...`

        @torch.no_grad()
        def hook(mod, args, output):
            if not self._active or self._failed or self.dropless:
                return
            layer = getattr(mod, "moe_layer", None)
            acc = self._acc.get(name)
            if layer is None or acc is None:
                return
            dropped = getattr(layer, "_dropped", None)          # native backend
            if dropped is None:
                counts = getattr(layer, "dispatch_count", None)  # tutel: per-expert counts
                if counts is None:
                    return
                counts = torch.as_tensor(counts)
                cap = capacity_of(int(counts.sum()), int(counts.numel()),
                                  self.capacity_factor, self.top_k)
                dropped = (counts - cap).clamp(min=0).sum()
            acc.setdefault("realised", torch.zeros((), dtype=torch.double,
                                                   device=torch.as_tensor(dropped).device))
            acc["realised"] += torch.as_tensor(dropped).double()
            acc["realised_seen"] = acc.get("realised_seen", 0) + 1

        return hook

    def setup(self, trainer, pl_module, stage=None):
        if self._handles or self._failed:
            return
        # [v12] inlined above: was `from pvt_moe.models.ffn import ...`

        root = getattr(pl_module, "model", None) or getattr(pl_module, "encoder", None) \
            or getattr(pl_module, "context", None) or pl_module
        blocks = 0
        for name, module in root.named_modules():
            if isinstance(module, MoEMlp):
                # two hooks per block: the pre-hook reads the policy off the gate
                # logits, the post-hook the realised overflow off the layer.
                self._handles.append(module.register_forward_pre_hook(self._hook(name, module)))
                self._handles.append(module.register_forward_hook(self._post_hook(name)))
                blocks += 1
        if blocks:
            print(f"[routing] monitoring {blocks} MoE block(s) per epoch: "
                  f"drop rate, share, routing entropy, gate entropy "
                  f"(train_aux cannot show these — see docs/HPARAMS.md)")

    def teardown(self, trainer, pl_module, stage=None):
        for h in self._handles:
            h.remove()
        self._handles = []

    # -- epoch boundaries ------------------------------------------------------

    def on_train_epoch_start(self, trainer, pl_module):
        self._acc = {}
        self._active = True

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        pass

    def on_validation_start(self, trainer, pl_module):
        self._active = False          # only training tokens count

    def on_validation_end(self, trainer, pl_module):
        self._active = not trainer.sanity_checking and trainer.training

    def on_train_epoch_end(self, trainer, pl_module):
        self._active = False
        if trainer.sanity_checking or not self._acc:
            return
        import math

        stats = {}
        for name, acc in self._acc.items():
            tokens = max(1, acc["tokens"])
            counts = acc["counts"].cpu()            # the ONE sync per epoch
            share = counts / counts.sum().clamp(min=1)
            mean_p = (acc["prob_sum"].cpu() / tokens)
            experts = int(counts.numel())
            imbalance = float((share - 1.0 / experts).clamp(min=0).sum())

            def _h(q):
                q = q[q > 0]
                return float(-(q * q.log()).sum()) if q.numel() else 0.0

            stats[name] = {
                "tokens": int(tokens),
                "num_experts": experts,
                "share": [round(float(v), 6) for v in share],
                "mean_gate_prob": [round(float(v), 6) for v in mean_p],
                "imbalance": round(imbalance, 6),
                "drop_rate": round(float(acc["dropped"].cpu()) / tokens, 6),
                "dropless": self.dropless,
                "route_entropy": round(_h(share), 6),
                "gate_entropy": round(float(acc["entropy_sum"].cpu()) / tokens, 6),
                "max_entropy": round(math.log(experts), 6),
                "aux_recomputed": round(float(experts * (share * mean_p).sum()), 8),
            }
            if acc.get("realised_seen"):
                stats[name]["drop_rate_realised"] = round(
                    float(acc["realised"].cpu()) / tokens, 6)
        self.last_stats = stats
        n = len(stats)
        for key in ("drop_rate", "imbalance", "route_entropy", "gate_entropy",
                    "drop_rate_realised"):
            present = [s[key] for s in stats.values() if key in s]
            if not present:
                continue
            pl_module.log(f"train_{'moe_imbalance' if key == 'imbalance' else key}",
                          torch.tensor(sum(present) / len(present)),
                          on_step=False, on_epoch=True)
        for name, s in stats.items():
            realised = (f" (realised {s['drop_rate_realised']:.1%})"
                        if "drop_rate_realised" in s else "")
            print(f"[routing] {name}: share {[f'{v:.3f}' for v in s['share']]} | "
                  f"drops {s['drop_rate']:.1%}{realised} | imbalance {s['imbalance']:.3f} | "
                  f"H(route) {s['route_entropy']:.2f}/{s['max_entropy']:.2f} | "
                  f"H(gate) {s['gate_entropy']:.2f}/{s['max_entropy']:.2f} | "
                  f"aux {s['aux_recomputed']:.4f}")


class PrintEpochMetrics(pl.Callback):
    """One human-readable line per epoch (the CSV/W&B logs stay canonical)."""

    def __init__(self):
        self._epoch_start = None

    def on_train_epoch_start(self, trainer, pl_module):
        self._epoch_start = time.time()

    # NOTE: this must be on_train_epoch_end, not on_validation_epoch_end —
    # during validation (which runs inside the train epoch) the current
    # epoch's train aggregates are not yet in callback_metrics, so the print
    # would pair epoch N's val metrics with epoch N-1's train metrics.
    def on_train_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        elapsed = time.time() - self._epoch_start if self._epoch_start else 0.0
        mins, secs = divmod(elapsed, 60)
        m = trainer.callback_metrics

        def _get(key):
            v = m.get(key)
            return v.item() if isinstance(v, torch.Tensor) else (v or 0.0)

        print(
            f"Epoch {trainer.current_epoch:3d} | "
            f"train acc(mixed): {_get('train_acc_mixed'):.1%} | "
            f"val acc: {_get('val_acc'):.1%} (top5 {_get('val_acc_top5'):.1%}) | "
            f"train loss: {_get('train_loss'):.4f} | val loss: {_get('val_loss'):.4f} | "
            f"aux: {_get('train_aux'):.4f} | {int(mins)}m {int(secs)}s"
        )

    def on_test_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics

        def _get(key):
            v = m.get(key)
            return v.item() if isinstance(v, torch.Tensor) else (v or 0.0)

        print(f"Test | acc: {_get('test_acc'):.1%} | loss: {_get('test_loss'):.4f}")


def _routing_monitor_wanted(cfg: dict) -> bool:
    """MoE actually placed somewhere, and the monitor not switched off."""
    abl = cfg["model"]["ablation"]
    return bool(cfg["model"]["moe"].get("routing_monitor", True)
                and abl["use_moe"] and any(abl["moe_placement"]))


def build_loggers(cfg: dict) -> list:
    """CSV always; TensorBoard and W&B by config flag."""
    from pytorch_lightning.loggers import CSVLogger

    run_name = cfg["run_name"]
    loggers = [CSVLogger(save_dir=cfg["log_root"], name=run_name)]

    if cfg.get("use_tensorboard"):
        from pytorch_lightning.loggers import TensorBoardLogger

        loggers.append(TensorBoardLogger(save_dir=cfg["log_root"], name=run_name))

    if cfg.get("use_wandb"):
        from pytorch_lightning.loggers import WandbLogger

        loggers.append(
            WandbLogger(
                project=cfg["wandb_project"],
                name=run_name,
                group=cfg.get("experiment_group"),
                # cfg is JSON-safe by construction — log it whole.
                config=cfg,
                log_model=False,
            )
        )
    return loggers


def build_trainer(cfg: dict, extra_callbacks: list | None = None) -> pl.Trainer:
    """Standard Trainer for this project.

    Checkpoint filenames are slash-free by construction (the v9 lineage
    monitored ``MulticlassAccuracy/val`` and the ``/`` in the filename template
    silently created nested directories).
    """
    # A resume must not silently change what the checkpoint cannot carry.
    # Here, not in the CLI: the notebooks call these factories directly.
    assert_resume_identity(cfg)
    ckpt_dir = os.path.join(cfg["checkpoint_root"], cfg["run_name"])
    # The schedule is always built for cfg["epochs"]; stop_at_epoch only ends
    # the run early, so a resumed run picks up the same cosine.
    max_epochs = cfg.get("stop_at_epoch") or cfg["epochs"]
    checkpoint_cb = ModelCheckpoint(
        dirpath=ckpt_dir,
        monitor="val_acc",
        mode="max",
        save_top_k=2,
        # last.ckpt is owned by RollingCheckpoint (see its docstring); with
        # save_last=True Lightning would write it only on top-k epochs.
        save_last=False,
        auto_insert_metric_name=False,
        filename="epoch{epoch:03d}-valacc{val_acc:.4f}",
    )
    callbacks = [
        checkpoint_cb,
        RollingCheckpoint(ckpt_dir),
        RopeFreqSnapshot(ckpt_dir),
        LearningRateMonitor(logging_interval="epoch"),
        PrintEpochMetrics(),
    ]
    if _routing_monitor_wanted(cfg):
        # Before ResultsWriter, so results.json picks up THIS epoch's routing.
        callbacks.append(RoutingMonitor(cfg))
    # results.json / results.md in the run directory, every epoch.
    callbacks.append(ResultsWriter(cfg, ckpt_dir))
    if cfg.get("milestones"):
        callbacks.append(MilestoneCheckpoint(cfg["milestones"], ckpt_dir))
    if extra_callbacks:
        callbacks.extend(extra_callbacks)

    return pl.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        devices=1,
        precision=cfg["precision"] if torch.cuda.is_available() else 32,
        gradient_clip_val=cfg["optim"]["grad_clip"],
        # micro-batch x this == cfg["effective_batch_size"], which is what the
        # recipe's LR is calibrated for. Clipping is applied to the accumulated
        # gradient by Lightning, i.e. once per optimizer step, as intended.
        accumulate_grad_batches=cfg.get("accumulate_grad_batches", 1),
        # "warn" (not True): True would make PL call
        # torch.use_deterministic_algorithms without warn_only, turning
        # nondeterministic-op warnings into mid-run crashes.
        deterministic=("warn" if cfg["deterministic"] else None),
        benchmark=not cfg["deterministic"],
        callbacks=callbacks,
        logger=build_loggers(cfg),
        log_every_n_steps=50,
        # None (the default) means Lightning's own default: every batch.
        **{k: cfg[k] for k in ("limit_train_batches", "limit_val_batches")
           if cfg.get(k) is not None},
    )


def build_ssl_trainer(cfg: dict, extra_callbacks: list | None = None) -> pl.Trainer:
    """Trainer for SSL pretraining (SimMIM / JEPA): monitors ``ssl_loss``, no
    val loop. Same checkpoint files as the supervised trainer (``last.ckpt``,
    milestones, RoPE frequency snapshots, results.json)."""
    # A resume must not silently change what the checkpoint cannot carry.
    # Here, not in the CLI: the notebooks call these factories directly.
    assert_resume_identity(cfg)
    ckpt_dir = os.path.join(cfg["checkpoint_root"], cfg["run_name"])
    checkpoint_cb = ModelCheckpoint(
        dirpath=ckpt_dir,
        monitor="ssl_loss",
        mode="min",
        save_top_k=1,
        save_last=False,          # last.ckpt: RollingCheckpoint (see its docstring)
        auto_insert_metric_name=False,
        filename="epoch{epoch:03d}-loss{ssl_loss:.4f}",
    )
    callbacks = [checkpoint_cb, RollingCheckpoint(ckpt_dir), RopeFreqSnapshot(ckpt_dir),
                 LearningRateMonitor(logging_interval="step")]
    if _routing_monitor_wanted(cfg):
        callbacks.append(RoutingMonitor(cfg))
    callbacks.append(ResultsWriter(cfg, ckpt_dir))
    if cfg.get("milestones"):
        callbacks.append(MilestoneCheckpoint(cfg["milestones"], ckpt_dir))
    if extra_callbacks:
        callbacks.extend(extra_callbacks)
    # The schedule is built for ssl.epochs; stop_at_epoch only ends the run
    # early, exactly as in the supervised trainer.
    max_epochs = cfg.get("stop_at_epoch") or cfg["ssl"]["epochs"]
    return pl.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        devices=1,
        precision=cfg["precision"] if torch.cuda.is_available() else 32,
        gradient_clip_val=cfg["ssl"]["grad_clip"],
        # micro-batch x this = the effective batch LitJEPA prints; the EMA
        # update is per optimizer step, so accumulation is safe here.
        accumulate_grad_batches=cfg.get("accumulate_grad_batches") or 1,
        # "warn" (not True): True would make PL call
        # torch.use_deterministic_algorithms without warn_only, turning
        # nondeterministic-op warnings into mid-run crashes.
        deterministic=("warn" if cfg["deterministic"] else None),
        benchmark=not cfg["deterministic"],
        callbacks=callbacks,
        logger=build_loggers(cfg),
        log_every_n_steps=50,
        **{k: cfg[k] for k in ("limit_train_batches",) if cfg.get(k) is not None},
    )


In [ ]:
# ── model + trainer ─────────────────────────────────────────────────────────
lit = LitClassifier(cfg)        # prints the optimizer groups and any warm start
trainer = build_trainer(cfg)    # checkpoints, RoutingMonitor, ResultsWriter, loggers


## Train

`trainer.fit` below starts (or resumes) the run. Mechanics:

- **Resume:** set `RESUME_FROM = "<ckpt_root>/<run_name>/last.ckpt"` in the
  CONFIG cell — full state (optimizer, scheduler, epoch, callbacks); identity
  is checked against the run's results.json first.
- **Long schedules in pieces:** set `STOP_AT_EPOCH` (the cosine still spans
  `EPOCHS`); milestones land in `milestone-epochNNN.ckpt`.
- **Watch:** `results.md` in the run directory rewrites every epoch; the
  `[routing]` console lines are the MoE health signal, not `train_aux`.
- A killed run loses nothing: `last.ckpt` is at most one epoch old.


In [ ]:
trainer.fit(lit, train_loader, val_loader,
            ckpt_path=cfg["ckpt_path"] if cfg.get("mode") == "resume" else None)


In [ ]:
# ── the run's results.md, as written this epoch ─────────────────────────────
import os
run_dir = os.path.join(cfg["checkpoint_root"], cfg["run_name"])
path = os.path.join(run_dir, "results.md")
print(open(path).read() if os.path.exists(path)
      else f"no results yet at {path} — train first")
# across runs: python tools/compare_runs.py <checkpoint_root>/*/results.json
